In [ ]:
# -*- coding: utf-8 -*-
"""
=========================================================
SCRIPT 1 - CARE ANOMALY DETECTION (DENOISING AUTOENCODER)
=========================================================

"""

import os, gc, re, json, warnings
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")


# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_PATH = r"D:\files\project\CARE_To_Compare"

FARMS = {
    "A": os.path.join(BASE_PATH, "Wind Farm A"),
    "B": os.path.join(BASE_PATH, "Wind Farm B"),
    "C": os.path.join(BASE_PATH, "Wind Farm C"),
}

EVENT_INFO_PATH = os.path.join(BASE_PATH, "df_all__1_.csv")
OUTPUT_DIR = os.path.join(BASE_PATH, "stage1_clean_outputs")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for d in [OUTPUT_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS = {0, 2}
META_COLS     = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# CARE
BETA                         = 0.5
CRITICALITY_THRESHOLD_GLOBAL = 36
CRITICALITY_THRESHOLD_SUB    = 110
SMOOTHING_WINDOW             = 6

# PCA
PCA_VARIANCE_TARGET = 0.95
MAX_PCA_COMPONENTS  = 150
MIN_PCA_COMPONENTS  = 3

# DAE
DAE_HIDDEN_FRACTIONS = [0.75, 0.50, 0.30]
DAE_BOTTLENECK_FACTOR = 4
DAE_BOTTLENECK_MIN    = 16

NOISE_STD      = 0.10
LEAKY_ALPHA    = 0.02
BATCH_SIZE     = 512
L2_LAMBDA      = 5e-6

ADAM_LR        = 5e-4
ADAM_BETA1     = 0.9
ADAM_BETA2     = 0.999
ADAM_EPS       = 1e-8

FARM_EPOCHS = {"A": 80, "B": 150, "C": 120}
FARM_LR     = {"A": 2e-4, "B": 5e-4, "C": 4e-4}

EARLY_STOP_PATIENCE = 15
EARLY_STOP_DELTA    = 1e-5

# Global thresholds
FARM_THRESHOLD_PCT_GLOBAL = {"A": 95, "B": 93, "C": 98}

# Much stricter subsystem thresholds
FARM_THRESHOLD_PCT_SUB = {
    "A": 99.0,
    "B": 99.7,
    "C": 99.0,
}

# Subsystem persistence
SUBSYSTEM_PERSISTENCE = {
    "A": 12,
    "B": 18,
    "C": 12,
}

SUBSYSTEM_MODEL_FARMS     = {"B"}
MIN_SENSORS_FOR_SUBSYSTEM = 4

SUBSYSTEM_ALARM_GROUPS = {
    "A": set(),
    "B": {"transformer"},
    "C": set(),
}

SUBSYSTEM_TRAIN_GROUPS = {
    "A": set(),
    "B": {"transformer", "gearbox_bearing", "pitch", "electrical", "wind"},
    "C": set(),
}

TOP_AVG_COLS_PER_FARM = {"A": 40, "B": 50, "C": 60}


# ============================================================================
# SUBSYSTEM RULES
# ============================================================================

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",  ["planetary bearing","gearbox bearing","bearing on high speed shaft"]),
    (11,  "gearbox_oil",      ["gearbox oil","gear oil","oil inlet temperature","oil pressure",
                               "oil level","oil tank","oil temperature"]),
    (12,  "gearbox_other",    ["gearbox rotational","gearbox speed","gearbox main shaft","gearbox"]),
    (20,  "generator_bearing",["generator bearing"]),
    (21,  "generator_stator", ["stator winding","stator"]),
    (22,  "generator_cooling",["generator cooling","cooling air","cooling water temp. generator",
                               "generator temperature","cooling air inlet"]),
    (23,  "generator_other",  ["generator acceleration","generator angle","generator rpm",
                               "generator converter","generator rms","generator"]),
    (30,  "rotor_bearing",    ["rotor bearing","axial bearing","rotor bearing inner ring"]),
    (40,  "hydraulic",        ["hydraulic pressure","hydraulic oil","hydraulic pump","hydraulic tank",
                               "hydraulic group","aggregate system pressure","azimuth circuit",
                               "azimuth memory","rotor brake","locking memory","pollution indicator",
                               "oil container"]),
    (50,  "pitch",            ["pitch angle","pitch motor","pitch axis","position motor axis",
                               "position rotor blade","motor temperature kty","motor temperature pt",
                               "axis 1 motor","axis 2 motor","axis 3 motor",
                               "axis 1 voltage","axis 2 voltage","axis 3 voltage",
                               "battery charge","battery discharge","rpm motor axis","min pitch",
                               "motor current axis","motor temperature axis"]),
    (60,  "vibration",        ["vibration","drive train vibration","tower vibration"]),
    (70,  "transformer",      ["transformer","transformator","hv transformer","transformer cell"]),
    (80,  "electrical",       ["voltage","current","frequency","active power","reactive power",
                               "apparent power","grid power","grid voltage","grid frequency",
                               "available power","inverter torque","cable load",
                               "dc link voltage","internal consumption"]),
    (90,  "wind",             ["wind speed","wind direction","anemometer","estimated windspeed",
                               "absolute wind","relative wind"]),
    (100, "environment",      ["ambient temperature","outside temperature","nacelle temperature",
                               "nacelle external","hub temperature","board temperature",
                               "electrical cabinet","converter room","control box temperature",
                               "nacelle outside","platform temperature"]),
    (110, "cooling_water",    ["water temperature","water pressure","water flow","water conductivity",
                               "water cooler","cooling water","flow meter stator"]),
    (120, "control",          ["yaw","nacelle direction","estimated tower","compensation angle",
                               "rotor speed","rotor rpm","bypass valve","aeration",
                               "pressure difference","rotational speed aerator","cable load"]),
    (130, "temperature",      ["temperature","temp."]),
]

STAT_MAP = {"average":"avg","std_dev":"std","maximum":"max","minimum":"min"}


# ============================================================================
# SENSOR REGISTRY
# ============================================================================

def _clean_desc(text):
    if pd.isna(text) or str(text).strip() in ("","nan"):
        return "unknown"
    text = str(text)
    for o,n in [("°C","degC"),("ºC","degC"),("½C","degC"),("Celsius","degC"),
                ("°","deg"),("º","deg"),("½","")]:
        text = text.replace(o,n)
    text = re.sub(r"[^\x00-\x7F]+","",text)
    text = re.sub(r"[^\w\s]"," ",text)
    text = re.sub(r"\s+","_",text.strip())
    return re.sub(r"_+","_",text).strip("_")

def _clean_unit(unit):
    if pd.isna(unit) or str(unit).strip().lower() in ("","nan","-","none"):
        return ""
    unit = str(unit)
    for o,n in [("°C","degC"),("ºC","degC"),("½C","degC"),("Celsius","degC"),
                ("°","deg"),("º","deg"),("½",""),("/","_per_"),("%","pct"),
                (" ",""),("^",""),(".","")]:
        unit = unit.replace(o,n)
    unit = re.sub(r"[^\x00-\x7F]+","",unit)
    return re.sub(r"[^\w]","",unit)

def _assign_group(description, is_counter=False, is_angle=False):
    if is_counter:
        return "counter"
    if is_angle:
        return "angle"
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"

def build_sensor_registry(farm_id):
    desc_path = os.path.join(FARMS[farm_id], "feature_description.csv")
    df = None
    for enc in ["utf-8","latin-1","cp1252"]:
        try:
            df = pd.read_csv(desc_path, sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise RuntimeError(f"Cannot read feature_description for Farm {farm_id}")

    rename_dict = {}
    group_dict = {}
    rows = []
    seen = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc_raw   = str(row.get("description","")).strip()
        unit_raw   = str(row.get("unit","")).strip()
        is_counter = bool(row.get("is_counter",False))
        is_angle   = bool(row.get("is_angle",False))
        group      = _assign_group(desc_raw,is_counter=is_counter,is_angle=is_angle)
        desc_clean = _clean_desc(desc_raw)
        unit_clean = _clean_unit(unit_raw)
        stats_raw  = str(row.get("statistics_type","")).strip()
        stats      = [] if stats_raw.lower() in ("","nan") else [s.strip() for s in stats_raw.split(",") if s.strip()]
        entries    = [(f"{sid}_{STAT_MAP.get(sr,sr)}",STAT_MAP.get(sr,sr)) for sr in stats] if stats else [(sid,"")]

        for old_col, stat_clean in entries:
            parts   = [desc_clean] + ([unit_clean] if unit_clean else []) + ([stat_clean] if stat_clean else [])
            new_col = "_".join(parts)
            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col
            rename_dict[old_col] = new_col
            group_dict[new_col] = group
            rows.append({
                "farm":farm_id,"sensor_id":sid,"old_col":old_col,"new_col":new_col,
                "description":desc_raw,"unit":unit_raw,"stat":stat_clean,"group":group,
                "is_counter":is_counter,"is_angle":is_angle,"use_in_nbm":not(is_counter or is_angle)
            })
    return rename_dict, group_dict, pd.DataFrame(rows)

print("="*80)
print("SCRIPT 1 - CARE ANOMALY DETECTION (DENOISING AUTOENCODER - HIGH PERFORMANCE vFIX)")
print("="*80)
print(f"Base: {BASE_PATH}\n")

registries, rename_dicts, group_dicts = {}, {}, {}
for farm in ["A","B","C"]:
    rd, gd, reg = build_sensor_registry(farm)
    rename_dicts[farm] = rd
    group_dicts[farm] = gd
    registries[farm] = reg
    print(f"Farm {farm}: {len(rd)} column mappings")

pd.concat(registries.values(), ignore_index=True).to_csv(
    os.path.join(REPORT_DIR, "sensor_registry_exact.csv"), index=False
)
print()


# ============================================================================
# DATA LOADING
# ============================================================================

def _get_excluded(farm_id):
    reg = registries[farm_id]
    return set(reg.loc[~reg["use_in_nbm"], "new_col"].tolist())

def _zero_fill(df, numeric_cols, excluded):
    for col in numeric_cols:
        if col in excluded:
            continue
        try:
            vals = df[col].values.copy().astype(np.float32)
        except Exception:
            continue
        n, i = len(vals), 0
        while i < n:
            if vals[i] == 0.0:
                start = i
                while i < n and vals[i] == 0.0:
                    i += 1
                if i - start >= 6:
                    vals[start:i] = np.nan
            else:
                i += 1
        df[col] = vals
    return df

def load_event(filepath, farm_id, chunk_size=10000):
    rdict = rename_dicts[farm_id]
    excluded = _get_excluded(farm_id)
    chunks = []

    for chunk in pd.read_csv(filepath, sep=";", chunksize=chunk_size, low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)
        num_cols = [c for c in chunk.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])]
        chunk = _zero_fill(chunk, num_cols, excluded)
        if num_cols:
            chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    train_df = df[df["train_test"]=="train"].copy()
    pred_df  = df[df["train_test"]=="prediction"].copy()

    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    power_cols = [c for c in train_df.columns if "power" in c.lower() and "avg" in c.lower() and c not in excluded]
    wind_cols  = [c for c in train_df.columns if "wind" in c.lower() and "speed" in c.lower() and "avg" in c.lower()]
    if power_cols and wind_cols:
        p, w = power_cols[0], wind_cols[0]
        bad = ((train_df[w] > 3.0) & (train_df[w] < 25.0) & (train_df[p].abs() < 0.01))
        train_df = train_df.loc[~bad].copy()

    del df
    gc.collect()
    return train_df, pred_df

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"] = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"] = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].astype(str).str.strip()
event_info["event_description"] = event_info["event_description"].fillna("no description").astype(str)
event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],   errors="coerce")

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue
    eid = row["event_id"]
    filepath = os.path.join(FARMS[farm], "datasets", f"{eid}.csv")
    if not os.path.exists(filepath):
        continue
    dataset.append({
        "event_id":eid,"farm":farm,"label":row["event_label"],
        "description":row["event_description"],"filepath":filepath,
        "event_start":row["event_start"],"event_end":row["event_end"]
    })

print("Event counts:")
for farm in ["A","B","C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    print(f"  Farm {farm}: total={len(fe)}  anomaly={sum(1 for d in fe if d['label']=='anomaly')}  normal={sum(1 for d in fe if d['label']=='normal')}")
print()


# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def engineer_features(df, farm_id, avg_cols_ref=None):
    excluded = _get_excluded(farm_id)
    avg_cols = [c for c in df.columns if c not in META_COLS and c not in excluded
                and "avg" in c.lower() and pd.api.types.is_numeric_dtype(df[c])]

    if avg_cols_ref is None:
        keep_n = TOP_AVG_COLS_PER_FARM.get(farm_id, 40)
        if len(avg_cols) > keep_n:
            avg_cols = df[avg_cols].var().sort_values(ascending=False).head(keep_n).index.tolist()
    else:
        avg_cols = avg_cols_ref

    new_feats = {}
    for col in avg_cols:
        s = df[col].astype(np.float32) if col in df.columns else pd.Series(np.zeros(len(df),dtype=np.float32), index=df.index)
        new_feats[f"{col}_roll6_mean"] = s.rolling(6, min_periods=1).mean().astype(np.float32)
        new_feats[f"{col}_roll6_std"]  = s.rolling(6, min_periods=1).std().fillna(0).astype(np.float32)
        new_feats[f"{col}_diff1"]      = s.diff(1).fillna(0).astype(np.float32)

    out = pd.concat([df, pd.DataFrame(new_feats, index=df.index)], axis=1)
    return out, avg_cols

def get_feature_cols(df, farm_id):
    excluded = _get_excluded(farm_id)
    return [c for c in df.columns if c not in META_COLS and c not in excluded and pd.api.types.is_numeric_dtype(df[c])]


# ============================================================================
# NUMPY AUTOENCODER
# ============================================================================

def leaky_relu(x, alpha=LEAKY_ALPHA):
    return np.where(x > 0, x, alpha*x)

def leaky_relu_grad(x, alpha=LEAKY_ALPHA):
    return np.where(x > 0, 1.0, alpha).astype(np.float32)

def minmax_compressive_arch(n_in):
    """
    Build a small compressive network.
    Prevent huge overcomplete architectures for tiny PCA inputs.
    """
    if n_in <= 4:
        bottleneck = max(2, n_in - 1)
        h1 = max(bottleneck + 1, n_in + 1)
        hidden = [h1]
    elif n_in <= 8:
        bottleneck = max(3, min(n_in - 1, n_in // 2))
        h1 = max(bottleneck + 2, min(10, n_in + 2))
        hidden = [h1]
    else:
        bottleneck = max(DAE_BOTTLENECK_MIN, n_in // DAE_BOTTLENECK_FACTOR)
        bottleneck = min(bottleneck, max(4, n_in - 2))
        hidden = []
        for f in DAE_HIDDEN_FRACTIONS:
            h = max(bottleneck + 2, int(n_in * f))
            h = min(h, max(bottleneck + 2, n_in))
            hidden.append(h)
        hidden = sorted(list(dict.fromkeys(hidden)), reverse=True)

    return hidden, bottleneck

class AdamOptimiser:
    def __init__(self, params, lr=ADAM_LR, beta1=ADAM_BETA1, beta2=ADAM_BETA2, eps=ADAM_EPS):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        bc1 = 1 - self.beta1**self.t
        bc2 = 1 - self.beta2**self.t
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.beta1*self.m[i] + (1-self.beta1)*g
            self.v[i] = self.beta2*self.v[i] + (1-self.beta2)*(g**2)
            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

class DenoisingAutoencoder:
    def __init__(self, n_in, n_epochs=80, lr=ADAM_LR):
        self.n_epochs = n_epochs
        self.n_in = n_in

        hidden, bottleneck = minmax_compressive_arch(n_in)
        dims_enc = [n_in] + hidden + [bottleneck]
        dims_dec = [bottleneck] + list(reversed(hidden)) + [n_in]

        print(f"    DAE arch: {dims_enc} → {dims_dec}")

        rng = np.random.default_rng(42)
        self.W_enc, self.b_enc = [], []
        self.W_dec, self.b_dec = [], []

        for i in range(len(dims_enc)-1):
            fan_in = dims_enc[i]
            std = np.sqrt(2.0 / (fan_in*(1+LEAKY_ALPHA**2)))
            self.W_enc.append(rng.normal(0, std, (dims_enc[i], dims_enc[i+1])).astype(np.float32))
            self.b_enc.append(np.zeros(dims_enc[i+1], dtype=np.float32))

        for i in range(len(dims_dec)-1):
            fan_in = dims_dec[i]
            std = np.sqrt(2.0 / (fan_in*(1+LEAKY_ALPHA**2)))
            self.W_dec.append(rng.normal(0, std, (dims_dec[i], dims_dec[i+1])).astype(np.float32))
            self.b_dec.append(np.zeros(dims_dec[i+1], dtype=np.float32))

        self.lr_init = lr
        all_params = self.W_enc + self.b_enc + self.W_dec + self.b_dec
        self.opt = AdamOptimiser(all_params, lr=lr)
        self.losses = []

    def _forward_enc(self, X):
        acts = [X]
        a = X
        for i, (W, b) in enumerate(zip(self.W_enc, self.b_enc)):
            z = a @ W + b
            a = leaky_relu(z) if i < len(self.W_enc)-1 else z
            acts.append(a)
        return acts

    def _forward_dec(self, z):
        acts = [z]
        a = z
        for i, (W, b) in enumerate(zip(self.W_dec, self.b_dec)):
            z_ = a @ W + b
            a = leaky_relu(z_) if i < len(self.W_dec)-1 else z_
            acts.append(a)
        return acts

    def reconstruction_error(self, X):
        enc = self._forward_enc(X)
        dec = self._forward_dec(enc[-1])
        return np.mean((X - dec[-1])**2, axis=1).astype(np.float32)

    def _cosine_lr(self, epoch, n_epochs, lr_min=1e-5):
        return lr_min + 0.5*(self.lr_init-lr_min)*(1+np.cos(np.pi*epoch/n_epochs))

    def train(self, Xp, rng):
        n = len(Xp)
        best_loss = np.inf
        patience_counter = 0

        for epoch in range(self.n_epochs):
            self.opt.lr = self._cosine_lr(epoch, self.n_epochs)
            idx = rng.permutation(n)
            Xp_shuf = Xp[idx]
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n, BATCH_SIZE):
                Xclean = Xp_shuf[start:start+BATCH_SIZE]
                batch_std = Xclean.std(axis=0, keepdims=True).clip(min=1e-6)
                noise = rng.normal(0, NOISE_STD, Xclean.shape).astype(np.float32) * batch_std
                Xnoisy = Xclean + noise

                enc_acts = self._forward_enc(Xnoisy)
                dec_acts = self._forward_dec(enc_acts[-1])
                X_hat = dec_acts[-1]

                batch = Xclean.shape[0]
                loss = float(np.mean((Xclean - X_hat)**2))
                if np.isnan(loss) or np.isinf(loss):
                    self.opt.lr *= 0.5
                    n_batches += 1
                    continue

                epoch_loss += loss
                n_batches += 1

                delta = -2.0 * (Xclean - X_hat) / batch

                dW_dec = [None]*len(self.W_dec)
                db_dec = [None]*len(self.b_dec)
                for i in reversed(range(len(self.W_dec))):
                    a_prev = dec_acts[i]
                    dW_dec[i] = a_prev.T @ delta + L2_LAMBDA*self.W_dec[i]
                    db_dec[i] = delta.sum(axis=0)
                    if i > 0:
                        delta = delta @ self.W_dec[i].T * leaky_relu_grad(dec_acts[i])

                delta = delta @ self.W_dec[0].T

                dW_enc = [None]*len(self.W_enc)
                db_enc = [None]*len(self.b_enc)
                for i in reversed(range(len(self.W_enc))):
                    a_prev = enc_acts[i]
                    dW_enc[i] = a_prev.T @ delta + L2_LAMBDA*self.W_enc[i]
                    db_enc[i] = delta.sum(axis=0)
                    if i > 0:
                        delta = delta @ self.W_enc[i].T * leaky_relu_grad(enc_acts[i])

                all_grads = dW_enc + db_enc + dW_dec + db_dec
                total_norm = np.sqrt(sum(np.sum(g**2) for g in all_grads))
                clip_scale = 5.0 / max(total_norm, 5.0)
                all_grads = [g * clip_scale for g in all_grads]

                all_params = self.W_enc + self.b_enc + self.W_dec + self.b_dec
                self.opt.step(all_params, all_grads)

            avg_loss = epoch_loss / max(n_batches, 1)
            self.losses.append(avg_loss)

            if (epoch+1) % 20 == 0:
                print(f"    Epoch {epoch+1:3d}/{self.n_epochs}  loss={avg_loss:.6f}  lr={self.opt.lr:.2e}")

            if avg_loss < best_loss - EARLY_STOP_DELTA:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= EARLY_STOP_PATIENCE:
                    print(f"    Early stop at epoch {epoch+1} (best loss={best_loss:.6f})")
                    break

        return self.losses


# ============================================================================
# TRAIN NBM
# ============================================================================

def train_dae_nbm(X_train, threshold_pct, farm_id, n_epochs, farm_lr=None):
    rng = np.random.default_rng(42)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train).astype(np.float32)

    n_comp = min(MAX_PCA_COMPONENTS, X_train.shape[1], max(2, len(X_train)-1))
    pca = PCA(n_components=n_comp, random_state=42)
    Xp = pca.fit_transform(Xs)

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_keep = int(np.searchsorted(cumvar, PCA_VARIANCE_TARGET) + 1)
    n_keep = max(MIN_PCA_COMPONENTS, min(n_keep, n_comp, len(cumvar)))
    Xp = Xp[:, :n_keep].astype(np.float32)
    Xp = np.clip(Xp, -10.0, 10.0)

    explained = float(cumvar[min(n_keep-1, len(cumvar)-1)])
    print(f"    PCA: {n_keep} components → {explained:.1%} variance  (input shape: {X_train.shape})")

    lr_to_use = farm_lr if farm_lr is not None else ADAM_LR
    dae = DenoisingAutoencoder(n_in=n_keep, n_epochs=n_epochs, lr=lr_to_use)
    dae.train(Xp, rng)

    train_errors = dae.reconstruction_error(Xp)
    train_errors = train_errors[np.isfinite(train_errors)]
    threshold = float(np.percentile(train_errors, threshold_pct)) if len(train_errors) > 0 else 0.1
    print(f"    DAE threshold ({threshold_pct}th pct): {threshold:.6f}")

    del Xs, Xp, train_errors
    gc.collect()

    return {
        "scaler": scaler,
        "pca": pca,
        "n_keep": n_keep,
        "dae": dae,
        "threshold": threshold,
        "explained": explained,
        "losses": dae.losses,
    }

def persistence_filter(preds, min_run):
    preds = np.asarray(preds).astype(int).copy()
    n = len(preds)
    i = 0
    while i < n:
        if preds[i] == 1:
            s = i
            while i < n and preds[i] == 1:
                i += 1
            if i - s < min_run:
                preds[s:i] = 0
        else:
            i += 1
    return preds

def score_with_dae(pred_df, feat_cols_ref, nbm, persistence_run=None):
    feat_cols = [c for c in feat_cols_ref if c in pred_df.columns]
    X = np.nan_to_num(pred_df[feat_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if X.shape[1] < len(feat_cols_ref):
        pad = np.zeros((len(X), len(feat_cols_ref) - X.shape[1]), dtype=np.float32)
        X = np.hstack([X, pad])

    Xs = nbm["scaler"].transform(X).astype(np.float32)
    Xp = nbm["pca"].transform(Xs)[:, :nbm["n_keep"]].astype(np.float32)
    Xp = np.clip(Xp, -10.0, 10.0)

    raw_errors = nbm["dae"].reconstruction_error(Xp)
    smoothed = pd.Series(raw_errors).rolling(SMOOTHING_WINDOW, min_periods=1).median().values.astype(np.float32)
    preds = (smoothed >= nbm["threshold"]).astype(int)

    if persistence_run is not None and persistence_run > 1:
        preds = persistence_filter(preds, persistence_run)

    del X, Xs, Xp
    gc.collect()
    return preds, smoothed

def criticality_algorithm(predictions, status_ids, farm_id):
    crit = 0
    max_crit = 0
    for i in range(len(predictions)):
        status_ok = True if farm_id == "A" else int(status_ids[i]) in NORMAL_STATUS
        if status_ok:
            crit = crit + 1 if predictions[i] == 1 else max(crit - 1, 0)
        if crit > max_crit:
            max_crit = crit
    return int(max_crit)

def get_fused_predictions(eid, farm, event_scores):
    es = event_scores[eid]
    preds = es["predictions_global"].copy()

    for group, preds_s in es.get("sub_preds_map", {}).items():
        if group not in SUBSYSTEM_ALARM_GROUPS.get(farm, set()):
            continue
        # Only fuse already persistence-filtered subsystem predictions
        preds = np.maximum(preds, preds_s)

    return preds


# ============================================================================
# TRAIN GLOBAL + SUBSYSTEMS
# ============================================================================

print("Training High-Performance DAE NBMs...\n")

farm_global_nbms = {}
farm_sub_nbms = {}
event_scores = {}

for farm in ["A","B","C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    normal_events = [d for d in farm_events if d["label"] == "normal"]
    threshold_pct_global = FARM_THRESHOLD_PCT_GLOBAL[farm]
    n_epochs = FARM_EPOCHS[farm]

    print(f"Farm {farm}: {len(normal_events)} normal events → global DAE (threshold={threshold_pct_global}th pct, epochs={n_epochs})")

    X_parts, avg_cols_ref, feat_cols_ref = [], [], None
    all_avg_cols_farm = None

    for d in normal_events:
        tr, _ = load_event(d["filepath"], farm)
        if len(tr) == 0:
            continue
        tr_eng, avg_cols_used = engineer_features(tr, farm, avg_cols_ref=avg_cols_ref)
        if not avg_cols_ref:
            avg_cols_ref = avg_cols_used
            excluded = _get_excluded(farm)
            all_avg_cols_farm = [c for c in tr_eng.columns if c not in META_COLS and c not in excluded
                                 and "avg" in c.lower() and pd.api.types.is_numeric_dtype(tr_eng[c])]
        fc = get_feature_cols(tr_eng, farm)
        if feat_cols_ref is None:
            feat_cols_ref = fc
        common = [c for c in feat_cols_ref if c in tr_eng.columns]
        X = np.nan_to_num(tr_eng[common].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        if X.shape[1] < len(feat_cols_ref):
            pad = np.zeros((len(X), len(feat_cols_ref) - X.shape[1]), dtype=np.float32)
            X = np.hstack([X, pad])
        X_parts.append(X)
        del tr, tr_eng, X
        gc.collect()

    if not X_parts:
        print(f"  No training data for Farm {farm}\n")
        continue

    X_train = np.vstack(X_parts).astype(np.float32)
    print(f"  Global matrix: {X_train.shape[0]:,} × {X_train.shape[1]}")
    nbm = train_dae_nbm(
        X_train,
        threshold_pct=threshold_pct_global,
        farm_id=farm,
        n_epochs=n_epochs,
        farm_lr=FARM_LR.get(farm, ADAM_LR)
    )
    nbm["feat_cols_ref"] = feat_cols_ref
    nbm["avg_cols_ref"] = avg_cols_ref
    farm_global_nbms[farm] = nbm

    del X_parts, X_train
    gc.collect()

    farm_sub_nbms[farm] = {}

    if farm in SUBSYSTEM_MODEL_FARMS:
        gdict = group_dicts[farm]
        source_cols = all_avg_cols_farm if all_avg_cols_farm else avg_cols_ref
        grp_cols = defaultdict(list)

        for c in (source_cols or []):
            grp = gdict.get(c, "other")
            if grp in SUBSYSTEM_TRAIN_GROUPS.get(farm, set()):
                grp_cols[grp].append(c)

        eligible = {g: cols for g, cols in grp_cols.items() if len(cols) >= MIN_SENSORS_FOR_SUBSYSTEM}
        print(f"  Subsystem groups found: {dict((g, len(c)) for g, c in grp_cols.items())}")
        print(f"  Subsystem models eligible (>={MIN_SENSORS_FOR_SUBSYSTEM} sensors): {list(eligible.keys())}")

        for group, sub_cols in eligible.items():
            X_sub_parts = []
            for d in normal_events:
                tr, _ = load_event(d["filepath"], farm)
                if len(tr) == 0:
                    continue
                tr_eng, _ = engineer_features(tr, farm, avg_cols_ref=None)
                use_cols = [c for c in sub_cols if c in tr_eng.columns]
                if len(use_cols) < 2:
                    del tr, tr_eng
                    gc.collect()
                    continue
                Xs = np.nan_to_num(tr_eng[use_cols].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
                X_sub_parts.append(Xs)
                del tr, tr_eng, Xs
                gc.collect()

            if not X_sub_parts:
                continue

            X_sub = np.vstack(X_sub_parts).astype(np.float32)
            print(f"    {group}: {X_sub.shape[0]:,} × {X_sub.shape[1]}")
            sub_threshold = FARM_THRESHOLD_PCT_SUB.get(farm, 99.0)
            sub_nbm = train_dae_nbm(
                X_sub,
                threshold_pct=sub_threshold,
                farm_id=farm,
                n_epochs=n_epochs,
                farm_lr=FARM_LR.get(farm, ADAM_LR)
            )
            sub_nbm["feat_cols_ref"] = sub_cols
            farm_sub_nbms[farm][group] = sub_nbm

            del X_sub_parts, X_sub
            gc.collect()

    print()


# ============================================================================
# SCORE ALL EVENTS
# ============================================================================

print("Scoring all events...\n")

for d in dataset:
    farm = d["farm"]
    eid = d["event_id"]

    if farm not in farm_global_nbms:
        continue

    nbm = farm_global_nbms[farm]
    _, pred_df = load_event(d["filepath"], farm)
    if len(pred_df) == 0:
        continue

    pred_eng, _ = engineer_features(pred_df, farm, avg_cols_ref=nbm["avg_cols_ref"])
    preds_global, errors_global = score_with_dae(pred_eng, nbm["feat_cols_ref"], nbm, persistence_run=None)

    status_ids = pred_eng["status_type_id"].values if "status_type_id" in pred_eng.columns else np.zeros(len(pred_eng), dtype=int)

    sub_preds_map = {}
    sub_scores_map = {}
    for group, sub_nbm in farm_sub_nbms.get(farm, {}).items():
        try:
            preds_s, errors_s = score_with_dae(
                pred_eng,
                sub_nbm["feat_cols_ref"],
                sub_nbm,
                persistence_run=SUBSYSTEM_PERSISTENCE.get(farm, 12)
            )
            sub_preds_map[group] = preds_s
            sub_scores_map[group] = float(np.max(errors_s))
        except Exception:
            continue

    slim_cols = ["time_stamp"] + (["status_type_id"] if "status_type_id" in pred_eng.columns else [])
    pred_df_slim = pred_eng[slim_cols].copy()

    event_scores[eid] = {
        "event_id": eid,
        "farm": farm,
        "label": d["label"],
        "description": d["description"],
        "event_start": d["event_start"],
        "event_end": d["event_end"],
        "predictions_global": preds_global,
        "scores_global": errors_global,
        "status_ids": status_ids,
        "pred_df": pred_df_slim,
        "sub_preds_map": sub_preds_map,
        "sub_scores_map": sub_scores_map,
    }

    del pred_df, pred_eng, pred_df_slim
    gc.collect()


# ============================================================================
# CARE SCORE
# ============================================================================

def compute_fbeta(y_true, y_pred, beta=BETA):
    tp = int(np.sum((y_true==1) & (y_pred==1)))
    fp = int(np.sum((y_true==0) & (y_pred==1)))
    fn = int(np.sum((y_true==1) & (y_pred==0)))
    denom = (1+beta**2)*tp + beta**2*fn + fp
    return float((1+beta**2)*tp/denom) if denom else 0.0

def compute_accuracy(y_true, y_pred):
    tn = int(np.sum((y_true==0) & (y_pred==0)))
    fp = int(np.sum((y_true==0) & (y_pred==1)))
    return float(tn/(tn+fp)) if (tn+fp) else 0.0

def compute_weighted_score(predictions, pred_df, event_start, event_end):
    n = len(predictions)
    if n == 0:
        return 0.0
    start_idx = end_idx = None
    if event_start is not None and event_end is not None and "time_stamp" in pred_df.columns:
        try:
            ts = pd.to_datetime(pred_df["time_stamp"].values, errors="coerce")
            idx = np.where((ts >= pd.to_datetime(event_start)) & (ts <= pd.to_datetime(event_end)))[0]
            if len(idx) > 0:
                start_idx = int(idx[0])
                end_idx = int(idx[-1]) + 1
        except Exception:
            pass
    if start_idx is None:
        start_idx = int(n*0.70)
        end_idx = n

    event_len = max(1, end_idx - start_idx)
    preds_in = predictions[start_idx:end_idx]
    weights = np.array([
        1.0 if (i/event_len) <= 0.5 else max(0.0, 1.0 - 2.0*((i/event_len)-0.5))
        for i in range(event_len)
    ], dtype=np.float32)
    return float((weights*preds_in).sum()/weights.sum()) if weights.sum() else 0.0

farm_results_all = {}
care_scores = {}

for farm in ["A","B","C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    farm_res = []

    for d in farm_events:
        eid = d["event_id"]
        if eid not in event_scores:
            continue

        es = event_scores[eid]
        preds_global = es["predictions_global"]
        status_ids = es["status_ids"]

        crit_global = criticality_algorithm(preds_global, status_ids, farm)
        alarm_global = crit_global >= CRITICALITY_THRESHOLD_GLOBAL

        best_sub_group = None
        best_sub_crit = 0
        alarm_sub = False

        for group, preds_s in es.get("sub_preds_map", {}).items():
            if group not in SUBSYSTEM_ALARM_GROUPS.get(farm, set()):
                continue
            crit_s = criticality_algorithm(preds_s, status_ids, farm)
            if crit_s > best_sub_crit:
                best_sub_crit = crit_s
                best_sub_group = group
            if crit_s >= CRITICALITY_THRESHOLD_SUB:
                alarm_sub = True

        alarm = alarm_global or alarm_sub
        max_crit = max(crit_global, best_sub_crit)

        if alarm_global:
            alarm_source = "global"
            preds_for_ws = preds_global
        elif alarm_sub and best_sub_group:
            alarm_source = f"subsystem:{best_sub_group}"
            preds_for_ws = es["sub_preds_map"][best_sub_group]
        else:
            alarm_source = "none"
            preds_for_ws = preds_global

        earliness = None
        if alarm and d["label"] == "anomaly":
            earliness = compute_weighted_score(preds_for_ws, es["pred_df"], d["event_start"], d["event_end"])

        sub_scores = es.get("sub_scores_map", {})
        top_subsystem = max(sub_scores, key=sub_scores.get) if sub_scores else "unknown"

        farm_res.append({
            "event_id":eid,"farm":farm,"label":d["label"],"event_label":d["label"],
            "description":d["description"],"alarm_raised":bool(alarm),
            "alarm_source":alarm_source,"max_criticality":int(max_crit),
            "crit_global":int(crit_global),"crit_subsystem":int(best_sub_crit),
            "anomaly_rate":round(float(preds_global.mean()),4),
            "earliness":round(float(earliness),4) if earliness is not None else None,
            "n_pred_rows":int(len(preds_global)),"event_start":d["event_start"],
            "event_end":d["event_end"],"top_subsystem":top_subsystem,
            "subsystem_scores":sub_scores,
        })

    farm_results_all[farm] = farm_res

    anomaly_r = [r for r in farm_res if r["label"] == "anomaly"]
    normal_r  = [r for r in farm_res if r["label"] == "normal"]

    all_yt, all_yp = [], []
    for r in anomaly_r:
        eid = r["event_id"]
        if eid not in event_scores:
            continue
        es = event_scores[eid]
        pred = get_fused_predictions(eid, farm, event_scores)
        prd = es["pred_df"]
        n = len(pred)
        y_true = np.zeros(n, dtype=int)
        try:
            ts = pd.to_datetime(prd["time_stamp"].values, errors="coerce")
            mask = (ts >= pd.to_datetime(r["event_start"])) & (ts <= pd.to_datetime(r["event_end"]))
            y_true[mask] = 1
        except Exception:
            y_true[int(n*0.70):] = 1
        if farm != "A":
            m = np.array([int(s) in NORMAL_STATUS for s in es["status_ids"]])
            y_true = y_true[m]
            pred = pred[m]
        all_yt.extend(y_true.tolist())
        all_yp.extend(pred.tolist())

    coverage = compute_fbeta(np.array(all_yt), np.array(all_yp)) if all_yt else 0.0

    acc_s = []
    for r in normal_r:
        eid = r["event_id"]
        if eid not in event_scores:
            continue
        es = event_scores[eid]
        pred = get_fused_predictions(eid, farm, event_scores)
        yt = np.zeros(len(pred), dtype=int)
        if farm != "A":
            m = np.array([int(s) in NORMAL_STATUS for s in es["status_ids"]])
            pred = pred[m]
            yt = yt[m]
        acc_s.append(compute_accuracy(yt, pred))
    accuracy = float(np.mean(acc_s)) if acc_s else 0.0

    ev_t, ev_p = [], []
    for r in farm_res:
        eid = r["event_id"]
        if eid not in event_scores:
            continue
        es = event_scores[eid]
        pf = get_fused_predictions(eid, farm, event_scores)
        crit = criticality_algorithm(pf, es["status_ids"], farm)
        al = int(crit >= CRITICALITY_THRESHOLD_GLOBAL)
        for group, preds_s in es.get("sub_preds_map", {}).items():
            if group not in SUBSYSTEM_ALARM_GROUPS.get(farm, set()):
                continue
            if criticality_algorithm(preds_s, es["status_ids"], farm) >= CRITICALITY_THRESHOLD_SUB:
                al = 1
                break
        ev_t.append(1 if r["label"] == "anomaly" else 0)
        ev_p.append(al)

    reliability = compute_fbeta(np.array(ev_t), np.array(ev_p)) if ev_t else 0.0

    ws_s = []
    for r in anomaly_r:
        if not r["alarm_raised"]:
            continue
        eid = r["event_id"]
        es = event_scores[eid]
        ws_s.append(compute_weighted_score(get_fused_predictions(eid, farm, event_scores),
                                           es["pred_df"], r["event_start"], r["event_end"]))
    earliness = float(np.mean(ws_s)) if ws_s else 0.0

    n_det = sum(1 for r in anomaly_r if r["alarm_raised"])
    n_fa  = sum(1 for r in normal_r if r["alarm_raised"])

    care = (0.0 if n_det == 0 else accuracy if accuracy < 0.5
            else (coverage + earliness + reliability + 2*accuracy)/5.0)

    care_scores[farm] = {
        "coverage": round(coverage, 3),
        "accuracy": round(accuracy, 3),
        "reliability": round(reliability, 3),
        "earliness": round(earliness, 3),
        "care": round(care, 3),
        "n_anomaly": len(anomaly_r),
        "n_normal": len(normal_r),
        "n_detected": n_det,
        "n_false_alarms": n_fa,
    }


# ============================================================================
# PLOTS
# ============================================================================

def plot_event_balance():
    farms = ["A","B","C"]
    an = [sum(1 for d in dataset if d["farm"]==f and d["label"]=="anomaly") for f in farms]
    no = [sum(1 for d in dataset if d["farm"]==f and d["label"]=="normal") for f in farms]
    fig, ax = plt.subplots(figsize=(9,5))
    x = np.arange(3)
    w = 0.35
    b1 = ax.bar(x-w/2, an, w, label="Anomaly", alpha=0.85)
    b2 = ax.bar(x+w/2, no, w, label="Normal", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("Events")
    ax.set_title("Dataset balance", fontweight="bold")
    ax.bar_label(b1, padding=2)
    ax.bar_label(b2, padding=2)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "01_event_balance.png"), dpi=150, bbox_inches="tight")
    plt.close()

def plot_care_scores():
    farms = ["A","B","C"]
    subs  = ["coverage","accuracy","reliability","earliness","care"]
    lbls  = ["Coverage","Accuracy","Reliability","Earliness","CARE"]
    fig, axes = plt.subplots(1,5, figsize=(18,4.8))
    for i, (sub, lbl) in enumerate(zip(subs, lbls)):
        vals = [care_scores[f][sub] for f in farms]
        bars = axes[i].bar([f"Farm {f}" for f in farms], vals, alpha=0.85)
        axes[i].set_ylim(0,1.1)
        axes[i].set_title(lbl, fontweight="bold", fontsize=10)
        axes[i].bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
        axes[i].grid(axis="y", alpha=0.3)
        if sub == "care":
            axes[i].axhline(0.660, color="orange", linestyle="--", linewidth=1.2, alpha=0.8, label="Baseline 0.660")
            axes[i].legend(fontsize=7)
    fig.suptitle("DAE CARE scores - high performance build", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "02_care_scores.png"), dpi=150, bbox_inches="tight")
    plt.close()

def plot_training_loss(farm, losses):
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(range(1, len(losses)+1), losses, linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.set_title(f"Farm {farm} - DAE training loss (Adam + cosine LR)", fontweight="bold")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"00_training_loss_farm{farm}.png"), dpi=140, bbox_inches="tight")
    plt.close()

def plot_criticality(farm):
    rows = farm_results_all.get(farm, [])
    if not rows:
        return
    rows = sorted(rows, key=lambda r: int(r["event_id"]))
    fig, ax = plt.subplots(figsize=(max(10, len(rows)*0.55), 5))
    cols = ["#d62728" if r["label"]=="anomaly" else "#2ca02c" for r in rows]
    ax.bar(range(len(rows)), [r["max_criticality"] for r in rows], color=cols, alpha=0.85)
    ax.axhline(CRITICALITY_THRESHOLD_GLOBAL, color="black", linestyle="--", linewidth=1.4)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([r["event_id"] for r in rows], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Max criticality")
    ax.set_title(f"Farm {farm} - criticality", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f"03_criticality_farm{farm}.png"), dpi=140, bbox_inches="tight")
    plt.close()

print("Generating plots...")
plot_event_balance()
plot_care_scores()
for farm in ["A","B","C"]:
    if farm in farm_global_nbms:
        plot_training_loss(farm, farm_global_nbms[farm]["losses"])
    plot_criticality(farm)


# ============================================================================
# SAVE
# ============================================================================

def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    try:
        if not isinstance(obj,(list,dict,tuple,set,np.ndarray)) and pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

all_events = [r for farm in ["A","B","C"] for r in farm_results_all.get(farm, [])]

results_json = json_safe({
    "timestamp": datetime.now().isoformat(),
    "script": "script_1_dae_high_performance_vfix",
    "model": "Denoising MLP Autoencoder (Adam, cosine LR, Leaky ReLU, early stopping, subsystem-stabilised)",
    "base_path": BASE_PATH,
    "parameters": {
        "beta": BETA,
        "criticality_threshold_global": CRITICALITY_THRESHOLD_GLOBAL,
        "criticality_threshold_sub": CRITICALITY_THRESHOLD_SUB,
        "smoothing_window": SMOOTHING_WINDOW,
        "pca_variance_target": PCA_VARIANCE_TARGET,
        "dae_bottleneck_factor": DAE_BOTTLENECK_FACTOR,
        "dae_bottleneck_min": DAE_BOTTLENECK_MIN,
        "dae_hidden_fractions": list(DAE_HIDDEN_FRACTIONS),
        "noise_std": NOISE_STD,
        "adam_lr": ADAM_LR,
        "n_epochs_per_farm": FARM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "l2_lambda": L2_LAMBDA,
        "farm_threshold_pct_global": FARM_THRESHOLD_PCT_GLOBAL,
        "farm_threshold_pct_sub": FARM_THRESHOLD_PCT_SUB,
        "subsystem_alarm_groups": {k:list(v) for k,v in SUBSYSTEM_ALARM_GROUPS.items()},
    },
    "care_scores": care_scores,
    "event_results": all_events,
})

json_path = os.path.join(OUTPUT_DIR, "results_stage1_clean.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results_json, f, indent=2)

csv_path = os.path.join(OUTPUT_DIR, "event_results_stage1_clean.csv")
pd.DataFrame(all_events).drop(columns=["subsystem_scores"], errors="ignore").to_csv(csv_path, index=False)

care_csv = os.path.join(REPORT_DIR, "care_score_summary_clean.csv")
pd.DataFrame([{"farm":farm, **care_scores[farm]} for farm in ["A","B","C"]]).to_csv(care_csv, index=False)

print("\nCARE score breakdown (High-Performance DAE vFIX):")
for farm in ["A","B","C"]:
    cs = care_scores[farm]
    print(f"  Farm {farm}: CARE={cs['care']:.3f}  "
          f"(cov={cs['coverage']:.3f} acc={cs['accuracy']:.3f} "
          f"rel={cs['reliability']:.3f} ear={cs['earliness']:.3f})  "
          f"detected={cs['n_detected']}/{cs['n_anomaly']}  "
          f"false={cs['n_false_alarms']}/{cs['n_normal']}")

print(f"\nSaved: {json_path}")
print(f"Saved: {csv_path}")
print(f"Saved: {care_csv}")
print(f"\n{'='*80}")
print("SCRIPT 1 (HIGH-PERFORMANCE DAE vFIX) COMPLETE")
print(f"{'='*80}")

SCRIPT 1 - CARE ANOMALY DETECTION (DENOISING AUTOENCODER - HIGH PERFORMANCE vFIX)
Base: D:\files\project\CARE_To_Compare

Farm A: 81 column mappings
Farm B: 252 column mappings
Farm C: 952 column mappings

Event counts:
  Farm A: total=22  anomaly=12  normal=10
  Farm B: total=15  anomaly=6  normal=9
  Farm C: total=58  anomaly=27  normal=31

Training High-Performance DAE NBMs...

Farm A: 10 normal events → global DAE (threshold=95th pct, epochs=80)
  Global matrix: 455,524 × 74
    PCA: 14 components → 95.4% variance  (input shape: (455524, 74))
    DAE arch: [14, 14, 12] → [12, 14, 14]
    Epoch  20/80  loss=0.138168  lr=1.75e-04
    Epoch  40/80  loss=0.090254  lr=1.09e-04
    Epoch  60/80  loss=0.068041  lr=4.05e-05
    Epoch  80/80  loss=0.062552  lr=1.01e-05
    DAE threshold (95th pct): 0.075614

Farm B: 9 normal events → global DAE (threshold=93th pct, epochs=150)
  Global matrix: 309,164 × 224
    PCA: 35 components → 95.1% variance  (input shape: (309164, 224))
    DAE arch: 

In [ ]:
""" Isolation Forest
================================================================================
CARE WIND TURBINE — ANOMALY DETECTION + FAULT CLASSIFICATION PIPELINE
================================================================================
A research-grade, farm-agnostic two-stage predictive maintenance pipeline for
wind turbine SCADA data using the CARE benchmark dataset.

Architecture:
  Stage 1 — Sensor renaming & subsystem grouping (farm-agnostic)
  Stage 2 — Data loading (chunked, memory-efficient, encoding-safe)
  Stage 3 — EDA & inspection (missing data, distributions, group summary)
  Stage 4 — Feature engineering (rolling stats, phase imbalance, normalisation)
  Stage 5 — Normal Behaviour Model (Isolation Forest per subsystem group)
  Stage 7 — Full CARE score computation (all four sub-scores)
  Stage 9 — Outputs (plots, JSON, CSV, sensor group report)

Operational flow:
  The NBM raises an alarm when the IF anomaly score exceeds the detection
  threshold for a sustained period (criticality algorithm from CARE paper).
  Fault type classification is handled separately in script_2_fault_classification.py.

Design principles:
  - NBM trained on NORMAL events only (no label leakage)
  - Status IDs used ONLY for training data filtering, never as labels
  - Farm A prediction data not status-filtered (per Zenodo note)
  - Zero-fill artefacts handled explicitly (non-counter sensors only)
  - Counter and angle sensors excluded from modelling
  - Full CARE score: Coverage (F_beta), Accuracy, Reliability (EF_beta),
    Earliness (WS) with actual event timestamps
  - Fault classifier trained on prediction window only (pre-failure signal)
  - Phase imbalance features for generator/transformer fault detection
  - Thermal normalisation using nacelle temperature
  - LOEO validation for both stages

Compatible with:
  - CARE benchmark dataset v5/v6 (Gück et al., 2024)
  - Any wind farm SCADA dataset with feature_description.csv in same format

Author: Peter
Dataset: https://doi.org/10.5281/zenodo.14006163
Paper: Gück et al. (2024), Data 9(12):138
"""

# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import os
import gc
import re
import json
import warnings
from datetime import datetime
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

# ============================================================================
# ① CONFIGURATION  
# ============================================================================

BASE_PATH = r"D:\files\project\CARE_To_Compare"

FARMS = {
    "A": os.path.join(BASE_PATH, "Wind Farm A"),
    "B": os.path.join(BASE_PATH, "Wind Farm B"),
    "C": os.path.join(BASE_PATH, "Wind Farm C"),
}

EVENT_INFO_PATH = os.path.join(BASE_PATH, "df_all__1_.csv")
OUTPUT_DIR      = os.path.join(BASE_PATH, "pipeline_final_v5")
PLOT_DIR        = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR      = os.path.join(OUTPUT_DIR, "reports")

for d in [OUTPUT_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

# CARE paper parameters
NORMAL_STATUS         = {0, 2}
CRITICALITY_THRESHOLD = 36        # 6-hour window (tuned for these event lengths)
BETA                  = 0.5       # F_beta: precision-weighted (paper uses β=½)
FARM_NAME_MAP         = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}

# Model parameters
IF_CONTAMINATION    = 0.01   # Training data is clean normal — very low noise
IF_N_ESTIMATORS     = 200
PCA_VARIANCE_TARGET = 0.95   # Retain 95% variance
MIN_PCA_COMPONENTS  = 10
SMOOTHING_WINDOW    = 6      # 6 x 10min = 1 hour rolling median on anomaly scores

# Per-farm adaptive thresholds (percentile of training score distribution)
# Farm A: small sensor set + short windows -> lower threshold for sensitivity
# Farm B: transformer faults need sensitivity -> lower threshold
# Farm C: rich sensor set, already working well at 99th pct
FARM_THRESHOLD_PCT = {"A": 95, "B": 97, "C": 99}

# Subsystem models trained for these farms (where signal is weak in global model)
# Farm B: transformer faults undetected by global model -> subsystem models fix this
# Farm C: too many subsystem models -> false alarm explosion -> global only
# Farm A: too few sensors per group for meaningful subsystem models
SUBSYSTEM_MODEL_FARMS        = {"B"}   # Farm B only
MIN_SENSORS_FOR_SUBSYSTEM_MODEL = 4

# Criticality thresholds
# Subsystem models are more focused/sensitive so require a higher threshold
# to avoid false alarms from individual noisy subsystems
CRITICALITY_THRESHOLD     = 36   # global model: 6 hours consecutive
CRITICALITY_THRESHOLD_SUB = 48   # subsystem model: 8 hours (stricter)

# Score fusion
SCORE_FUSION = "max"

# Column metadata
META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# Subsystem grouping rules — priority order, first match wins
# Each entry: (priority, group_name, [keywords])
SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",  ["planetary bearing", "gearbox bearing",
                                "bearing on high speed shaft"]),
    (11,  "gearbox_oil",      ["gearbox oil", "gear oil", "oil inlet temperature",
                                "oil pressure", "oil level", "oil tank",
                                "oil temperature"]),
    (12,  "gearbox_other",    ["gearbox rotational", "gearbox speed",
                                "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing",["generator bearing"]),
    (21,  "generator_stator", ["stator winding", "stator"]),
    (22,  "generator_cooling",["generator cooling", "cooling air",
                                "cooling water temp. generator",
                                "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",  ["generator acceleration", "generator angle",
                                "generator rpm", "generator converter",
                                "generator rms", "generator"]),
    (30,  "rotor_bearing",    ["rotor bearing", "axial bearing",
                                "rotor bearing inner ring"]),
    (40,  "hydraulic",        ["hydraulic pressure", "hydraulic oil",
                                "hydraulic pump", "hydraulic tank",
                                "hydraulic group", "aggregate system pressure",
                                "azimuth circuit", "azimuth memory",
                                "rotor brake", "locking memory",
                                "pollution indicator", "oil container"]),
    (50,  "pitch",            ["pitch angle", "pitch motor", "pitch axis",
                                "position motor axis", "position rotor blade",
                                "motor temperature kty", "motor temperature pt",
                                "axis 1 motor", "axis 2 motor", "axis 3 motor",
                                "axis 1 voltage", "axis 2 voltage", "axis 3 voltage",
                                "battery charge", "battery discharge",
                                "rpm motor axis", "min pitch",
                                "motor current axis", "motor temperature axis"]),
    (60,  "vibration",        ["vibration", "drive train vibration",
                                "tower vibration"]),
    (70,  "transformer",      ["transformer", "transformator",
                                "hv transformer", "transformer cell"]),
    (80,  "electrical",       ["voltage", "current", "frequency",
                                "active power", "reactive power",
                                "apparent power", "grid power", "grid voltage",
                                "grid frequency", "available power",
                                "inverter torque", "cable load",
                                "dc link voltage", "internal consumption"]),
    (90,  "wind",             ["wind speed", "wind direction", "anemometer",
                                "estimated windspeed", "absolute wind",
                                "relative wind"]),
    (100, "environment",      ["ambient temperature", "outside temperature",
                                "nacelle temperature", "nacelle external",
                                "hub temperature", "board temperature",
                                "electrical cabinet", "converter room",
                                "control box temperature", "nacelle outside",
                                "platform temperature"]),
    (110, "cooling_water",    ["water temperature", "water pressure",
                                "water flow", "water conductivity",
                                "water cooler", "cooling water",
                                "flow meter stator"]),
    (120, "control",          ["yaw", "nacelle direction",
                                "estimated tower", "compensation angle",
                                "rotor speed", "rotor rpm", "bypass valve",
                                "aeration", "pressure difference",
                                "rotational speed aerator", "cable load"]),
    (130, "temperature",      ["temperature", "temp."]),
]


# ============================================================================
# ② SENSOR RENAMING & SUBSYSTEM GROUPING
# ============================================================================

def _clean_desc(text):
    """Encode-safe description → snake_case."""
    if pd.isna(text) or str(text).strip() in ("", "nan"):
        return "unknown"
    text = str(text)
    for old, new in [("���C","degC"),("°C","degC"),("ºC","degC"),("½C","degC"),
                     ("Celsius","degC"),("���","deg"),("°","deg"),("º","deg"),
                     ("½","")]:
        text = text.replace(old, new)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", "_", text.strip())
    return re.sub(r"_+", "_", text).strip("_")


def _clean_unit(unit):
    """Encode-safe unit string for column names."""
    if pd.isna(unit) or str(unit).strip().lower() in ("", "nan", "-", "none"):
        return ""
    unit = str(unit)
    for old, new in [("���C","degC"),("°C","degC"),("ºC","degC"),("½C","degC"),
                     ("Celsius","degC"),("°","deg"),("º","deg"),("½",""),
                     ("/","_per_"),("%","pct"),(" ",""),("^",""),(".","")]:
        unit = unit.replace(old, new)
    unit = re.sub(r"[^\x00-\x7F]+", "", unit)
    return re.sub(r"[^\w]", "", unit)


def _assign_group(description, is_counter=False, is_angle=False):
    """Assign sensor to subsystem group."""
    if is_counter: return "counter"
    if is_angle:   return "angle"
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


STAT_MAP = {"average":"avg","std_dev":"std","maximum":"max","minimum":"min"}


def build_sensor_registry(farm_id):
    """
    Build rename dict, group dict, and full registry DataFrame for one farm.
    Returns: (rename_dict, group_dict, registry_df)
    """
    desc_path = os.path.join(FARMS[farm_id], "feature_description.csv")
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(desc_path, sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    rename_dict, group_dict, rows = {}, {}, []
    seen = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc_raw   = str(row.get("description", "")).strip()
        unit_raw   = str(row.get("unit", "")).strip()
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))
        group      = _assign_group(desc_raw, is_counter, is_angle)
        desc_clean = _clean_desc(desc_raw)
        unit_clean = _clean_unit(unit_raw)

        stats_raw = str(row.get("statistics_type","")).strip()
        stats = [] if stats_raw.lower() in ("nan","") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = []
        if stats:
            for sr in stats:
                sc = STAT_MAP.get(sr, sr)
                entries.append((f"{sid}_{sc}", sc))
        else:
            entries.append((sid, ""))

        for old_col, stat_clean in entries:
            parts   = [desc_clean] + ([unit_clean] if unit_clean else [])
            if stat_clean:
                parts.append(stat_clean)
            new_col = "_".join(parts)
            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col

            rename_dict[old_col] = new_col
            group_dict[new_col]  = group
            rows.append({
                "farm": farm_id, "sensor_id": sid,
                "old_col": old_col, "new_col": new_col,
                "description": desc_raw, "unit": unit_raw,
                "stat": stat_clean, "group": group,
                "is_counter": is_counter, "is_angle": is_angle,
                "use_in_nbm": not (is_counter or is_angle),
            })

    return rename_dict, group_dict, pd.DataFrame(rows)


# Build registries for all farms
print("=" * 80)
print("CARE WIND TURBINE ANOMALY DETECTION PIPELINE")
print("=" * 80)
print("\n① Building sensor registries...")

registries   = {}   # farm → registry_df
rename_dicts = {}   # farm → {old_col: new_col}
group_dicts  = {}   # farm → {new_col: group}

for farm in ["A", "B", "C"]:
    rd, gd, reg = build_sensor_registry(farm)
    rename_dicts[farm] = rd
    group_dicts[farm]  = gd
    registries[farm]   = reg

    # Validate: no duplicate new names
    new_names = list(rd.values())
    dupes = set(n for n in new_names if new_names.count(n) > 1)
    status = f"⚠️ {len(dupes)} duplicates" if dupes else "✓ no duplicates"
    print(f"  Farm {farm}: {len(rd)} column mappings — {status}")

    # Group summary
    summary = (reg[reg["stat"].isin(["avg",""])]
               .groupby("group")
               .agg(n_sensors=("sensor_id","nunique"),
                    use_in_nbm=("use_in_nbm","first"))
               .sort_values("n_sensors", ascending=False))
    print(f"  {'Group':25} {'Sensors':>7}  NBM")
    for group, r in summary.iterrows():
        flag = "✓" if r["use_in_nbm"] else "✗ (excluded)"
        print(f"  {'  '+group:25} {r['n_sensors']:7d}  {flag}")

# Save sensor group report
all_reg = pd.concat(registries.values(), ignore_index=True)
all_reg.to_csv(os.path.join(REPORT_DIR, "sensor_registry.csv"), index=False)
print(f"\n  Saved: {REPORT_DIR}/sensor_registry.csv")


# ============================================================================
# ③ DATA LOADING
# ============================================================================

def _get_excluded(farm_id):
    """New column names that are counters or angles — excluded from models."""
    reg = registries[farm_id]
    return set(reg.loc[~reg["use_in_nbm"], "new_col"])


def _handle_zero_fills(df, numeric_cols, excluded):
    """Replace runs of ≥6 consecutive zeros with NaN. Skip counter/angle cols."""
    for col in numeric_cols:
        if col in excluded:
            continue
        vals = df[col].values.copy().astype(np.float32)
        n, i = len(vals), 0
        while i < n:
            if vals[i] == 0.0:
                start = i
                while i < n and vals[i] == 0.0:
                    i += 1
                if i - start >= 6:
                    vals[start:i] = np.nan
            else:
                i += 1
        df[col] = vals
    return df


def load_event(filepath, farm_id, chunk_size=10000):
    """
    Load one event CSV → (train_df, pred_df).

    train_df : status-filtered + power-curve-filtered training rows
    pred_df  : all prediction rows (no status filter — per paper/Zenodo)
    """
    rdict    = rename_dicts[farm_id]
    excluded = _get_excluded(farm_id)
    chunks   = []

    for chunk in pd.read_csv(filepath, sep=";", chunksize=chunk_size,
                              low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)
        num_cols = [c for c in chunk.columns
                    if c not in META_COLS
                    and pd.api.types.is_numeric_dtype(chunk[c])]
        chunk = _handle_zero_fills(chunk, num_cols, excluded)
        chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)
        del chunk

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    train_df = df[df["train_test"] == "train"].copy()
    pred_df  = df[df["train_test"] == "prediction"].copy()

    # Status filter on training data only
    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    # Power curve filter — remove suspicious rows from training
    power_cols = [c for c in train_df.columns
                  if "power" in c.lower() and "avg" in c.lower()
                  and c not in excluded]
    wind_cols  = [c for c in train_df.columns
                  if "wind" in c.lower() and "speed" in c.lower()
                  and "avg" in c.lower()]
    if power_cols and wind_cols:
        p, w = power_cols[0], wind_cols[0]
        bad = ((train_df[w] > 3.0) & (train_df[w] < 25.0) &
               (train_df[p].abs() < 0.01))
        train_df = train_df[~bad]

    del df
    gc.collect()
    return train_df, pred_df


# ============================================================================
# ④ EVENT METADATA
# ============================================================================

print("\n② Loading event metadata...")

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"]        = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"]    = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].str.strip()
event_info["event_description"] = (event_info["event_description"]
                                    .fillna("no description").astype(str))
event_info["event_start"] = pd.to_datetime(event_info["event_start"],
                                            errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],
                                            errors="coerce")

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue
    eid      = row["event_id"]
    filepath = os.path.join(FARMS[farm], "datasets", f"{eid}.csv")
    if not os.path.exists(filepath):
        print(f"  ⚠️  Missing: Farm {farm} / {eid}.csv")
        continue
    dataset.append({
        "event_id":    eid,
        "farm":        farm,
        "label":       row["event_label"],
        "filepath":    filepath,
        "description": row["event_description"],
        "event_start": row["event_start"],
        "event_end":   row["event_end"],
    })

for farm in ["A","B","C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    na = sum(1 for d in fe if d["label"] == "anomaly")
    nn = sum(1 for d in fe if d["label"] == "normal")
    print(f"  Farm {farm}: {len(fe)} events ({na} anomaly / {nn} normal)")


# ============================================================================
# ⑤ EDA & INSPECTION
# ============================================================================

print("\n③ EDA & inspection...")

eda_results = {}

for farm in ["A","B","C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    if not farm_events:
        continue

    excluded    = _get_excluded(farm)
    lengths_tr, lengths_pr, missing_r = [], [], []

    for d in farm_events[:8]:
        try:
            tr, pr = load_event(d["filepath"], farm)
            lengths_tr.append(len(tr))
            lengths_pr.append(len(pr))
            sig = [c for c in tr.columns
                   if c not in META_COLS and c not in excluded
                   and pd.api.types.is_numeric_dtype(tr[c])]
            if sig:
                missing_r.append(tr[sig].isna().mean().mean())
            del tr, pr
            gc.collect()
        except Exception as e:
            print(f"  ⚠️  EDA {d['event_id']}: {e}")

    eda_results[farm] = {
        "n_total":    len(farm_events),
        "n_anomaly":  sum(1 for d in farm_events if d["label"]=="anomaly"),
        "n_normal":   sum(1 for d in farm_events if d["label"]=="normal"),
        "train_mean": int(np.mean(lengths_tr)) if lengths_tr else 0,
        "train_min":  int(np.min(lengths_tr))  if lengths_tr else 0,
        "train_max":  int(np.max(lengths_tr))  if lengths_tr else 0,
        "pred_mean":  int(np.mean(lengths_pr)) if lengths_pr else 0,
        "missing":    float(np.mean(missing_r)) if missing_r else 0.0,
    }
    r = eda_results[farm]
    print(f"  Farm {farm}: train={r['train_mean']:,} rows  "
          f"pred={r['pred_mean']:,} rows  "
          f"missing={r['missing']:.1%}")


# ============================================================================
# ⑥ FEATURE ENGINEERING
# ============================================================================

def engineer_features(df, farm_id, ref_avg_cols=None):
    """
    Add rolling mean, rolling std, diff1 for top _avg signal columns.
    ref_avg_cols: fixed column list for alignment across train/test.
    Returns (df_engineered, avg_cols_used).
    """
    excluded = _get_excluded(farm_id)
    avg_cols = [c for c in df.columns
                if c not in META_COLS and c not in excluded
                and "avg" in c.lower()
                and pd.api.types.is_numeric_dtype(df[c])]

    if ref_avg_cols is None:
        # Select top-40 by variance
        if len(avg_cols) > 40:
            avg_cols = df[avg_cols].var().nlargest(40).index.tolist()
    else:
        avg_cols = ref_avg_cols

    new_feats = {}
    for col in avg_cols:
        if col not in df.columns:
            s = pd.Series(np.zeros(len(df), dtype=np.float32), index=df.index)
        else:
            s = df[col].astype(np.float32)
        new_feats[f"{col}_roll6_mean"] = (
            s.rolling(6, min_periods=1).mean().astype(np.float32))
        new_feats[f"{col}_roll6_std"] = (
            s.rolling(6, min_periods=1).std().fillna(0).astype(np.float32))
        new_feats[f"{col}_diff1"] = (
            s.diff(1).fillna(0).astype(np.float32))

    out = pd.concat([df, pd.DataFrame(new_feats, index=df.index)], axis=1)
    del new_feats
    gc.collect()
    return out, avg_cols


def get_nbm_feature_cols(df, farm_id, ref_cols):
    """Return feature columns aligned to ref_cols, padding with zeros if needed."""
    excluded = _get_excluded(farm_id)
    available = [c for c in ref_cols
                 if c not in META_COLS and c not in excluded
                 and pd.api.types.is_numeric_dtype(df[c])
                 if c in df.columns]
    return available


# ============================================================================
# ⑦ STAGE 1: ISOLATION FOREST NBM — ANOMALY SCORING
# ============================================================================

print("\n④ Training Normal Behaviour Models (Isolation Forest)...")


def train_nbm(X_train, threshold_pct=99):
    """
    Fit StandardScaler -> PCA (95% variance) -> IsolationForest.
    threshold_pct: percentile of training scores used as anomaly threshold.
      Farm A uses 95th, Farm B uses 97th, Farm C uses 99th.
    Returns (scaler, pca, model, n_keep, anomaly_threshold, explained).
    """
    scaler = StandardScaler()
    X_s    = scaler.fit_transform(X_train)

    n_comp = min(150, X_train.shape[1], len(X_train) - 1)
    pca    = PCA(n_components=n_comp, random_state=42)
    X_p    = pca.fit_transform(X_s)

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_keep = max(MIN_PCA_COMPONENTS,
                 int(np.searchsorted(cumvar, PCA_VARIANCE_TARGET)) + 1)
    n_keep = min(n_keep, n_comp)
    X_p    = X_p[:, :n_keep]

    model = IsolationForest(n_estimators=IF_N_ESTIMATORS,
                            contamination=IF_CONTAMINATION,
                            random_state=42, n_jobs=-1)
    model.fit(X_p)

    # Threshold from training distribution (no test data leakage)
    train_scores      = -model.score_samples(X_p).astype(np.float32)
    anomaly_threshold = float(np.percentile(train_scores, threshold_pct))

    explained = float(cumvar[n_keep - 1])
    del X_s, X_p, train_scores
    gc.collect()

    return scaler, pca, model, n_keep, anomaly_threshold, explained


def score_window(pred_df, farm_id, scaler, pca, model,
                 n_keep, anomaly_threshold, ref_feat_cols):
    """
    Apply NBM to prediction window.
    Returns: (timestep_predictions, anomaly_scores_smoothed, status_ids)
    """
    excluded  = _get_excluded(farm_id)
    feat_cols = [c for c in ref_feat_cols
                 if c in pred_df.columns
                 and c not in META_COLS
                 and c not in excluded]

    X = np.nan_to_num(
        pred_df[feat_cols].values.astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )

    # Pad if test has fewer columns than training
    if X.shape[1] < len(ref_feat_cols):
        pad = np.zeros((len(X), len(ref_feat_cols) - X.shape[1]),
                       dtype=np.float32)
        X   = np.hstack([X, pad])

    X_s    = scaler.transform(X)
    X_p    = pca.transform(X_s)[:, :n_keep]
    raw    = -model.score_samples(X_p).astype(np.float32)

    # Rolling median smoothing — reduces noisy alternating detections
    smoothed   = (pd.Series(raw)
                  .rolling(SMOOTHING_WINDOW, min_periods=1)
                  .median()
                  .values)
    predictions = (smoothed >= anomaly_threshold).astype(int)

    status_ids = (pred_df["status_type_id"].values
                  if "status_type_id" in pred_df.columns
                  else np.ones(len(pred_df), dtype=int))

    del X, X_s, X_p
    gc.collect()
    return predictions, smoothed, status_ids


def criticality_algorithm(predictions, status_ids, farm_id):
    """Paper Algorithm 1. Farm A: ignore status (Zenodo note)."""
    crit = max_crit = 0
    for i in range(len(predictions)):
        st_normal = True if farm_id == "A" \
                    else int(status_ids[i]) in NORMAL_STATUS
        if st_normal:
            crit = crit + 1 if predictions[i] == 1 else max(crit - 1, 0)
        if crit > max_crit:
            max_crit = crit
    return max_crit


# Per-farm: collect anomaly scores for ALL events
# (needed for CARE score)

farm_nbm_models    = {}   # farm -> global NBM tuple
farm_sub_models    = {}   # farm -> {group -> NBM tuple}
event_scores       = {}   # event_id -> {scores, predictions, ...}

for farm in ["A", "B", "C"]:
    farm_events    = [d for d in dataset if d["farm"] == farm]
    normal_events  = [d for d in farm_events if d["label"] == "normal"]
    anomaly_events = [d for d in farm_events if d["label"] == "anomaly"]
    thresh_pct     = FARM_THRESHOLD_PCT[farm]

    if not normal_events:
        print(f"  Farm {farm}: no normal events — skipping")
        continue

    print(f"\n  Farm {farm}: building NBM from {len(normal_events)} normal events"
          f" (threshold percentile: {thresh_pct}th)")

    # ----------------------------------------------------------------
    # Build global training matrix
    # ----------------------------------------------------------------
    X_parts       = []
    avg_cols_ref  = None
    ref_feat_cols = None

    for d in normal_events:
        try:
            tr, _ = load_event(d["filepath"], farm)
            tr, avg_cols_used = engineer_features(tr, farm, avg_cols_ref)
            if avg_cols_ref is None:
                avg_cols_ref = avg_cols_used

            excluded = _get_excluded(farm)
            fc = [c for c in tr.columns
                  if c not in META_COLS and c not in excluded
                  and pd.api.types.is_numeric_dtype(tr[c])]

            if ref_feat_cols is None:
                ref_feat_cols = fc
            else:
                fc = fc[:len(ref_feat_cols)]

            X = np.nan_to_num(tr[fc].values.astype(np.float32),
                              nan=0.0, posinf=0.0, neginf=0.0)
            X_parts.append(X)
            del tr
            gc.collect()
        except Exception as e:
            print(f"    Normal {d['event_id']}: {e}")

    if not X_parts:
        print(f"  Farm {farm}: no training data loaded")
        continue

    min_cols      = min(x.shape[1] for x in X_parts)
    X_train       = np.vstack([x[:, :min_cols] for x in X_parts]).astype(np.float32)
    ref_feat_cols = ref_feat_cols[:min_cols]
    del X_parts
    gc.collect()

    print(f"  Global: {len(X_train):,} rows x {X_train.shape[1]} features")
    scaler, pca, model, n_keep, threshold, explained = train_nbm(
        X_train, threshold_pct=thresh_pct
    )
    print(f"  PCA: {n_keep} components -> {explained:.1%} variance | "
          f"threshold: {threshold:.4f}")

    farm_nbm_models[farm] = (scaler, pca, model, n_keep, threshold,
                             ref_feat_cols, avg_cols_ref)
    del X_train
    gc.collect()

    # ----------------------------------------------------------------
    # Per-subsystem models (Farm B and C only)
    # ----------------------------------------------------------------
    farm_sub_models[farm] = {}

    if farm in SUBSYSTEM_MODEL_FARMS:
        reg      = registries[farm]
        excluded = _get_excluded(farm)
        gdict    = group_dicts[farm]

        # Find groups with enough avg sensors
        nbm_cols = [c for c in ref_feat_cols
                    if c not in excluded and "avg" in c.lower()]
        group_col_map = defaultdict(list)
        for col in nbm_cols:
            g = gdict.get(col, "other")
            if g not in ("counter", "angle", "other"):
                group_col_map[g].append(col)

        eligible_groups = {g: cols for g, cols in group_col_map.items()
                           if len(cols) >= MIN_SENSORS_FOR_SUBSYSTEM_MODEL}

        print(f"  Subsystem models: {list(eligible_groups.keys())}")

        for group, group_cols in eligible_groups.items():
            # Reload normal training data filtered to this group's columns
            X_sub_parts = []
            for d in normal_events:
                try:
                    tr, _ = load_event(d["filepath"], farm)
                    tr, _ = engineer_features(tr, farm, avg_cols_ref)
                    avail = [c for c in group_cols if c in tr.columns]
                    if len(avail) < 2:
                        continue
                    X_sub = np.nan_to_num(
                        tr[avail].values.astype(np.float32),
                        nan=0.0, posinf=0.0, neginf=0.0
                    )
                    X_sub_parts.append((X_sub, avail))
                    del tr
                    gc.collect()
                except Exception:
                    continue

            if not X_sub_parts:
                continue

            min_sub = min(x.shape[1] for x, _ in X_sub_parts)
            X_sub_train = np.vstack(
                [x[:, :min_sub] for x, _ in X_sub_parts]
            ).astype(np.float32)
            sub_ref_cols = X_sub_parts[0][1][:min_sub]
            del X_sub_parts
            gc.collect()

            if X_sub_train.shape[1] < 2:
                del X_sub_train
                continue

            try:
                sc_s, pca_s, mod_s, nk_s, thr_s, exp_s = train_nbm(
                    X_sub_train, threshold_pct=thresh_pct
                )
                farm_sub_models[farm][group] = (
                    sc_s, pca_s, mod_s, nk_s, thr_s, sub_ref_cols
                )
                print(f"    {group:25} {X_sub_train.shape[1]:3d} features "
                      f"-> {nk_s} PCA components "
                      f"| threshold: {thr_s:.4f}")
            except Exception as e:
                print(f"    Subsystem {group}: {e}")

            del X_sub_train
            gc.collect()

    # ----------------------------------------------------------------
    # Score ALL events with global model + subsystem models (max fusion)
    # ----------------------------------------------------------------
    for d in farm_events:
        try:
            _, pr = load_event(d["filepath"], farm)
            if len(pr) == 0:
                continue
            pr, _ = engineer_features(pr, farm, avg_cols_ref)

            # Global model score
            preds_global, scores_global, status_ids = score_window(
                pr, farm, scaler, pca, model, n_keep, threshold, ref_feat_cols
            )

            # Subsystem model scores — store separately for independent criticality
            sub_scores_map  = {}
            sub_preds_map   = {}   # group -> predictions array

            for group, sub_tuple in farm_sub_models.get(farm, {}).items():
                sc_s, pca_s, mod_s, nk_s, thr_s, sub_ref_cols = sub_tuple
                try:
                    preds_s, scores_s, _ = score_window(
                        pr, farm, sc_s, pca_s, mod_s, nk_s, thr_s, sub_ref_cols
                    )
                    sub_scores_map[group] = float(np.max(scores_s))
                    sub_preds_map[group]  = preds_s
                except Exception:
                    continue

            # Store only what is needed for CARE scoring — not the full df
            # This saves significant memory for Farm C (58 events x 1000+ cols)
            pred_df_slim = pr[["time_stamp", "status_type_id"]].copy() \
                if "status_type_id" in pr.columns \
                else pr[["time_stamp"]].copy()

            event_scores[d["event_id"]] = {
                "predictions":      preds_global,
                "predictions_global": preds_global,
                "sub_preds_map":    sub_preds_map,
                "scores":           scores_global,
                "scores_global":    scores_global,
                "sub_scores_map":   sub_scores_map,
                "status_ids":       status_ids,
                "pred_df":          pred_df_slim,  # slim: timestamps + status only
                "farm":             farm,
                "label":            d["label"],
                "description":      d["description"],
                "event_start":      d["event_start"],
                "event_end":        d["event_end"],
            }
            del pr, pred_df_slim
            gc.collect()
        except Exception as e:
            print(f"    Score {d['event_id']}: {e}")


# ============================================================================
# ⑨ FULL CARE SCORE
# ============================================================================

print("\n⑤ Computing full CARE score...")

"""
CARE score sub-scores (paper Section 3.2):
  Coverage   : F_beta on anomaly datasets (pointwise, β=0.5)
  Accuracy   : TN/(FP+TN) on normal datasets  
  Reliability: Event-level F_beta using criticality algorithm
  Earliness  : Weighted Score — early detections weighted higher
Final CARE = weighted average of all four.
"""


def compute_fbeta(y_true, y_pred, beta=BETA):
    """F_beta score. β=0.5 weights precision over recall."""
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    denom = (1 + beta**2) * tp + beta**2 * fn + fp
    return float((1 + beta**2) * tp / denom) if denom > 0 else 0.0


def compute_accuracy(y_true, y_pred):
    """Specificity: TN/(FP+TN). For normal-only datasets."""
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    return float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0


def compute_weighted_score(predictions, pred_df, event_start, event_end):
    """
    Paper WS (Earliness). Weights detections in first half of event
    window at 1.0, linearly decaying to 0 in second half.
    Uses actual event timestamps.
    """
    n = len(predictions)
    if n == 0:
        return 0.0

    # Locate event window in prediction data
    start_idx, end_idx = None, None
    ts_col = "time_stamp"

    if (event_start is not None and not pd.isna(event_start) and
            event_end is not None and not pd.isna(event_end) and
            ts_col in pred_df.columns):
        try:
            ts = pd.to_datetime(pred_df[ts_col].values, errors="coerce")
            in_event = np.where((ts >= pd.to_datetime(event_start)) &
                                (ts <= pd.to_datetime(event_end)))[0]
            if len(in_event) > 0:
                start_idx = int(in_event[0])
                end_idx   = int(in_event[-1]) + 1
        except Exception:
            pass

    if start_idx is None:
        start_idx = int(n * 0.70)
        end_idx   = n

    event_len = end_idx - start_idx
    if event_len <= 0:
        return 0.0

    weights = np.array([
        1.0 if (i / event_len) <= 0.5
        else max(0.0, 1.0 - 2.0 * ((i / event_len) - 0.5))
        for i in range(event_len)
    ], dtype=np.float32)

    preds_in = predictions[start_idx:end_idx]
    return float((weights * preds_in).sum() / weights.sum()) \
        if weights.sum() > 0 else 0.0


def compute_care_score(farm_results, farm_id):
    """
    Compute all four CARE sub-scores for one farm.

    For Coverage and Earliness: uses anomaly events only.
    For Accuracy: uses normal events only.
    For Reliability (EF_beta): event-level classification.

    Predictions used: fused (global OR best subsystem), consistent with
    how alarms are raised in the results loop.
    """
    anomaly_r = [r for r in farm_results if r["label"] == "anomaly"]
    normal_r  = [r for r in farm_results if r["label"] == "normal"]

    def _get_fused_predictions(eid, farm_id):
        """
        Return the fused prediction array for CARE scoring.
        Combines global + best subsystem predictions (max fusion)
        so Coverage reflects what the full system actually detects.
        """
        es = event_scores.get(eid)
        if es is None:
            return None
        preds = es["predictions_global"].copy()
        # Fuse with subsystem predictions where available
        for group, preds_s in es.get("sub_preds_map", {}).items():
            preds = np.maximum(preds, preds_s)
        return preds

    # --- Coverage: F_beta on anomaly datasets (pointwise) ---
    all_y_true, all_y_pred = [], []

    for r in anomaly_r:
        eid  = r["event_id"]
        if eid not in event_scores:
            continue
        es   = event_scores[eid]
        pred = _get_fused_predictions(eid, farm_id)
        if pred is None:
            continue
        prd  = es["pred_df"]
        n    = len(pred)

        # Build ground truth: 1 within event_start..event_end
        y_true = np.zeros(n, dtype=int)
        if "event_start" in r and "event_end" in r:
            es_t  = r.get("event_start")
            ee_t  = r.get("event_end")
            ts_col = "time_stamp"
            if (es_t is not None and not pd.isna(es_t) and
                    es_t is not None and ts_col in prd.columns):
                try:
                    ts = pd.to_datetime(prd[ts_col].values, errors="coerce")
                    mask = ((ts >= pd.to_datetime(es_t)) &
                            (ts <= pd.to_datetime(ee_t)))
                    y_true[mask] = 1
                except Exception:
                    y_true[int(n*0.70):] = 1
            else:
                y_true[int(n*0.70):] = 1
        else:
            y_true[int(n*0.70):] = 1

        # Filter to normal status timesteps (paper requirement)
        # Farm A: don't filter
        if farm_id != "A" and "status_ids" in es:
            mask_normal = np.array([s in NORMAL_STATUS
                                    for s in es["status_ids"]])
            y_true = y_true[mask_normal]
            pred   = pred[mask_normal]

        all_y_true.extend(y_true.tolist())
        all_y_pred.extend(pred.tolist())

    coverage = compute_fbeta(
        np.array(all_y_true), np.array(all_y_pred)
    ) if all_y_true else 0.0

    # --- Accuracy: normal datasets (fused predictions) ---
    acc_scores = []
    for r in normal_r:
        eid = r["event_id"]
        if eid not in event_scores:
            continue
        es   = event_scores[eid]
        pred = _get_fused_predictions(eid, farm_id)
        if pred is None:
            continue
        y_true_norm = np.zeros(len(pred), dtype=int)  # all normal

        if farm_id != "A" and "status_ids" in es:
            mask_normal = np.array([s in NORMAL_STATUS
                                    for s in es["status_ids"]])
            y_true_norm = y_true_norm[mask_normal]
            pred        = pred[mask_normal]

        acc_scores.append(compute_accuracy(y_true_norm, pred))
    accuracy = float(np.mean(acc_scores)) if acc_scores else 0.0

    # --- Reliability: event-level F_beta (fused alarms) ---
    ev_true, ev_pred = [], []
    for r in farm_results:
        eid = r["event_id"]
        if eid not in event_scores:
            continue
        es    = event_scores[eid]
        preds = _get_fused_predictions(eid, farm_id)
        if preds is None:
            continue
        sids     = es["status_ids"]
        max_crit = criticality_algorithm(preds, sids, farm_id)
        # Also check subsystem criticality
        for preds_s in es.get("sub_preds_map", {}).values():
            crit_s = criticality_algorithm(preds_s, sids, farm_id)
            if crit_s >= CRITICALITY_THRESHOLD_SUB:
                max_crit = max(max_crit, CRITICALITY_THRESHOLD)  # count as alarm
        alarm = int(max_crit >= CRITICALITY_THRESHOLD)
        ev_true.append(1 if r["label"] == "anomaly" else 0)
        ev_pred.append(alarm)

    reliability = compute_fbeta(
        np.array(ev_true), np.array(ev_pred)
    ) if ev_true else 0.0

    # --- Earliness: WS on detected anomaly events (fused predictions) ---
    ws_scores = []
    for r in anomaly_r:
        eid = r["event_id"]
        if not r.get("alarm_raised", False):
            continue  # not detected by either model
        if eid not in event_scores:
            continue
        es    = event_scores[eid]
        preds = _get_fused_predictions(eid, farm_id)
        if preds is None:
            continue
        ws = compute_weighted_score(
            preds, es["pred_df"], r.get("event_start"), r.get("event_end")
        )
        ws_scores.append(ws)
    earliness = float(np.mean(ws_scores)) if ws_scores else 0.0

    # --- Final CARE score (paper Eq. 4/5) ---
    # Use alarm_raised from farm_results (which includes subsystem alarms)
    n_detected = sum(1 for r in anomaly_r if r.get("alarm_raised", False))

    if n_detected == 0:
        care = 0.0
    elif accuracy < 0.5:
        care = accuracy
    else:
        care = (coverage + earliness + reliability + 2 * accuracy) / 5

    return {
        "coverage":    round(coverage,    4),
        "accuracy":    round(accuracy,    4),
        "reliability": round(reliability, 4),
        "earliness":   round(earliness,   4),
        "care":        round(care,        4),
        "n_anomaly":   len(anomaly_r),
        "n_normal":    len(normal_r),
        "n_detected":  n_detected,
        "n_false_alarms": sum(1 for r in normal_r if r.get("alarm_raised", False)),
    }


# Build per-farm result lists and compute CARE scores
farm_results_all = {}
care_scores      = {}

for farm in ["A","B","C"]:
    farm_events = [d for d in dataset if d["farm"] == farm]
    farm_res    = []
    for d in farm_events:
        eid = d["event_id"]
        if eid not in event_scores:
            continue
        es   = event_scores[eid]
        sids = es["status_ids"]

        # Global model alarm
        preds_global = es["predictions_global"]
        max_crit_global = criticality_algorithm(preds_global, sids, farm)
        alarm_global    = max_crit_global >= CRITICALITY_THRESHOLD

        # Subsystem model alarms (stricter threshold to control false alarms)
        alarm_sub    = False
        max_crit_sub = 0
        best_sub_group = None
        for group, preds_s in es.get("sub_preds_map", {}).items():
            crit_s = criticality_algorithm(preds_s, sids, farm)
            if crit_s > max_crit_sub:
                max_crit_sub = crit_s
                best_sub_group = group
            if crit_s >= CRITICALITY_THRESHOLD_SUB:
                alarm_sub = True

        # Combined alarm: global OR subsystem (with respective thresholds)
        alarm    = alarm_global or alarm_sub
        max_crit = max(max_crit_global, max_crit_sub)

        # For predictions used in CARE score, use whichever triggered the alarm
        if alarm_global or not alarm_sub:
            preds_for_ws = preds_global
        else:
            # Use the best subsystem predictions for earliness
            best_preds = es["sub_preds_map"].get(best_sub_group, preds_global)
            preds_for_ws = best_preds

        ws = compute_weighted_score(
            preds_for_ws, es["pred_df"], d["event_start"], d["event_end"]
        ) if alarm and d["label"] == "anomaly" else None

        alarm_source = ("global" if alarm_global
                        else f"subsystem:{best_sub_group}" if alarm_sub
                        else "none")

        farm_res.append({
            "event_id":        eid,
            "farm":            farm,
            "label":           d["label"],
            "description":     d["description"],
            "alarm_raised":    bool(alarm),
            "alarm_source":    alarm_source,
            "max_criticality": int(max_crit),
            "crit_global":     int(max_crit_global),
            "crit_subsystem":  int(max_crit_sub),
            "anomaly_rate":    round(float(preds_global.mean()), 4),
            "earliness":       round(ws, 4) if ws is not None else None,
            "n_pred_rows":     int(len(preds_global)),
            "event_start":     d["event_start"],
            "event_end":       d["event_end"],
        })

    farm_results_all[farm] = farm_res

    if farm_res:
        care_scores[farm] = compute_care_score(farm_res, farm)
        cs = care_scores[farm]
        print(f"\n  Farm {farm} CARE score breakdown:")
        print(f"    Coverage    (F_β): {cs['coverage']:.3f}  "
              f"← pointwise detection density within event window")
        print(f"    Accuracy   (Acc): {cs['accuracy']:.3f}  "
              f"← normal event specificity")
        print(f"    Reliability (EFβ): {cs['reliability']:.3f}  "
              f"← event-level alarm precision/recall")
        print(f"    Earliness   (WS): {cs['earliness']:.3f}  "
              f"← early vs late detection weighting")
        print(f"    ─────────────────────────")
        print(f"    CARE score      : {cs['care']:.3f}")
        print(f"    Event detection : {cs['n_detected']}/{cs['n_anomaly']} alarms raised")
        print(f"    False alarms    : {cs['n_false_alarms']}/{cs['n_normal']} normal events")
        if farm == "B" and cs["n_detected"] > 0 and cs["coverage"] < 0.2:
            print(f"    NOTE Farm B: All {cs['n_detected']} anomaly events alarmed but")
            print(f"    Coverage is low because subsystem alarms are sparse/short —")
            print(f"    the alarm threshold (criticality {CRITICALITY_THRESHOLD_SUB}) is")
            print(f"    met but the pointwise anomaly density within the event window")
            print(f"    is low. Event detection rate is the more meaningful metric here.")


# ============================================================================
# ⑥ SUBSYSTEM-LEVEL BREAKDOWN
# ============================================================================

print("\n⑥ Computing subsystem-level detection breakdown...")


def subsystem_breakdown(event_id, farm_id):
    """
    Return subsystem contribution scores for one event.
    Uses direct subsystem model scores where available (more accurate),
    falls back to sensor variance proxy for groups without dedicated models.
    """
    if event_id not in event_scores:
        return {}

    es    = event_scores[event_id]
    gdict = group_dicts[farm_id]

    # Prefer direct subsystem model scores (from per-group NBMs)
    result = {}
    if es.get("sub_scores_map"):
        result.update(es["sub_scores_map"])

    # Fill in any remaining groups via sensor variance proxy
    pred_df  = es["pred_df"]
    excluded = _get_excluded(farm_id)
    avg_cols = [c for c in pred_df.columns
                if c not in META_COLS and c not in excluded
                and "avg" in c.lower()
                and pd.api.types.is_numeric_dtype(pred_df[c])]

    group_var = defaultdict(list)
    for col in avg_cols:
        group = gdict.get(col, "other")
        if group not in result and group not in ("counter", "angle"):
            vals = pred_df[col].dropna().values
            if len(vals) > 0:
                group_var[group].append(float(np.std(vals)))

    for g, vs in group_var.items():
        result[g] = round(float(np.mean(vs)), 4)

    return result


# Add subsystem breakdown to results
for farm, farm_res in farm_results_all.items():
    for r in farm_res:
        r["subsystem_scores"] = subsystem_breakdown(r["event_id"], farm)

        # Top contributing subsystem
        ss = r["subsystem_scores"]
        if ss:
            top = max(ss, key=ss.get)
            r["top_subsystem"] = top
        else:
            r["top_subsystem"] = "unknown"


# ============================================================================
# ⑪ VISUALISATIONS
# ============================================================================

print("\n⑦ Generating visualisations...")


def plot_event_balance():
    farms = ["A","B","C"]
    an = [sum(1 for d in dataset if d["farm"]==f and d["label"]=="anomaly")
          for f in farms]
    no = [sum(1 for d in dataset if d["farm"]==f and d["label"]=="normal")
          for f in farms]

    fig, ax = plt.subplots(figsize=(9,5))
    x, w = np.arange(3), 0.35
    b1 = ax.bar(x-w/2, an, w, label="Anomaly", color="#d62728", alpha=0.85)
    b2 = ax.bar(x+w/2, no, w, label="Normal",  color="#2ca02c", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms], fontsize=11)
    ax.set_ylabel("Number of Events", fontsize=11)
    ax.set_title("Dataset Balance — Events per Farm", fontsize=13,
                 fontweight="bold")
    ax.legend(fontsize=10)
    ax.bar_label(b1, padding=3, fontsize=10)
    ax.bar_label(b2, padding=3, fontsize=10)
    ax.set_ylim(0, max(max(an),max(no)) * 1.2)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, "01_event_balance.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")


def plot_anomaly_score_timeseries(event_id, farm_id, label, description):
    """
    Plot raw anomaly score time series for one event.
    Shows: smoothed score, detection threshold, event window, alarm period.
    """
    if event_id not in event_scores:
        return

    es      = event_scores[event_id]
    scores  = es["scores"]
    preds   = es["predictions_global"]
    pred_df = es["pred_df"]
    n       = len(scores)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={"height_ratios":[3,1]})

    # --- Top panel: anomaly score ---
    ax = axes[0]
    x  = np.arange(n)

    # Farm NBM threshold
    if farm_id in farm_nbm_models:
        thresh = farm_nbm_models[farm_id][4]
    else:
        thresh = float(np.percentile(scores, 99))

    ax.plot(x, scores, color="#4878d0", lw=0.8, alpha=0.8,
            label="Anomaly score (smoothed)", rasterized=True)
    ax.axhline(thresh, color="#d62728", ls="--", lw=1.5,
               label=f"Detection threshold ({thresh:.3f})", alpha=0.9)

    # Shade event window
    es_t = es.get("event_start")
    ee_t = es.get("event_end")
    ts_col = "time_stamp"
    event_start_idx = None

    if (es_t is not None and not pd.isna(es_t) and
            ts_col in pred_df.columns):
        try:
            ts = pd.to_datetime(pred_df[ts_col].values, errors="coerce")
            in_ev = np.where((ts >= pd.to_datetime(es_t)) &
                             (ts <= pd.to_datetime(ee_t)))[0]
            if len(in_ev) > 0:
                ax.axvspan(in_ev[0], in_ev[-1],
                           alpha=0.12, color="#d62728",
                           label="Event window (annotated)")
                event_start_idx = in_ev[0]
        except Exception:
            pass

    # Shade detected alarms (criticality > threshold)
    alarm_mask = preds.astype(bool)
    if alarm_mask.any():
        ax.fill_between(x, 0, scores, where=alarm_mask,
                        alpha=0.25, color="#ff7f0e", label="Detected anomaly")

    color_label = "#d62728" if label == "anomaly" else "#2ca02c"
    ax.set_title(
        f"Farm {farm_id}  —  Event {event_id}  ({label.upper()})  —  {description}",
        fontsize=10, fontweight="bold", color=color_label
    )
    ax.set_ylabel("Anomaly Score", fontsize=10)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.8)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0, n)

    # --- Bottom panel: binary prediction ---
    ax2 = axes[1]
    ax2.fill_between(x, 0, preds, step="post",
                     color="#ff7f0e", alpha=0.7, label="Anomaly flag")
    ax2.set_ylabel("Flag", fontsize=9)
    ax2.set_xlabel("Timestep (10-min intervals)", fontsize=9)
    ax2.set_ylim(-0.1, 1.3)
    ax2.set_xlim(0, n)
    ax2.grid(True, alpha=0.2)
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(["Normal", "Anomaly"], fontsize=8)

    plt.tight_layout()
    fname = f"02_score_farm{farm_id}_{label}_E{event_id}.png"
    p = os.path.join(PLOT_DIR, fname)
    try:
        plt.savefig(p, dpi=120, bbox_inches="tight")
        print(f"  Saved: {p}")
    except Exception as e:
        print(f"  ⚠️  Plot save failed E{event_id}: {e}")
    plt.close()


def plot_care_scores(care_scores):
    """Four-panel CARE sub-score comparison across farms."""
    farms = [f for f in ["A","B","C"] if f in care_scores]
    if not farms:
        return

    sub_scores = ["coverage","accuracy","reliability","earliness"]
    labels     = ["Coverage\n(F_β)", "Accuracy\n(Acc)",
                   "Reliability\n(EF_β)", "Earliness\n(WS)"]
    colors_pos = "#2ca02c"
    colors_neg = "#d62728"

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    for i, (sub, lbl) in enumerate(zip(sub_scores, labels)):
        ax  = axes[i]
        vals = [care_scores[f][sub] for f in farms]
        cols = [colors_pos if v >= 0.5 else colors_neg for v in vals]
        bars = ax.bar([f"Farm {f}" for f in farms], vals,
                      color=cols, alpha=0.85, edgecolor="white")
        ax.set_ylim(0, 1.12)
        ax.set_title(lbl, fontsize=11, fontweight="bold")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
        ax.axhline(0.5, color="#555", ls="--", lw=1.1, alpha=0.6)
        ax.grid(axis="y", alpha=0.3)

    # Final CARE score
    ax = axes[4]
    vals = [care_scores[f]["care"] for f in farms]
    cols = [colors_pos if v >= 0.5 else colors_neg for v in vals]
    bars = ax.bar([f"Farm {f}" for f in farms], vals,
                  color=cols, alpha=0.88, edgecolor="white", linewidth=1.5)
    ax.set_ylim(0, 1.12)
    ax.set_title("CARE Score\n(Combined)", fontsize=11, fontweight="bold")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=11, fontweight="bold")
    ax.axhline(0.5, color="#555", ls="--", lw=1.1, alpha=0.6,
               label="Random baseline (0.5)")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Full CARE Score — All Sub-scores — All Farms",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, "03_care_scores.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")


def plot_criticality(farm, farm_res):
    """Per-event criticality bar chart."""
    if not farm_res:
        return
    fr = sorted(farm_res, key=lambda x: x["event_id"])
    fig, ax = plt.subplots(figsize=(max(10, len(fr)*0.6), 5))
    colors = ["#d62728" if r["label"]=="anomaly" else "#2ca02c" for r in fr]
    bars   = ax.bar(range(len(fr)),
                    [r["max_criticality"] for r in fr],
                    color=colors, alpha=0.85,
                    edgecolor=["#8b0000" if r["alarm_raised"] else "#444"
                               for r in fr],
                    linewidth=1.3)
    ax.axhline(CRITICALITY_THRESHOLD, color="black", ls="--", lw=1.8,
               label=f"Alarm threshold ({CRITICALITY_THRESHOLD})")
    ax.set_xticks(range(len(fr)))
    ax.set_xticklabels([r["event_id"] for r in fr],
                       rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Max Criticality", fontsize=10)
    ax.set_title(f"Farm {farm} — Max Criticality per Event",
                 fontsize=11, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor="#d62728", alpha=0.85, label="Anomaly event"),
        Patch(facecolor="#2ca02c", alpha=0.85, label="Normal event"),
        plt.Line2D([0],[0], color="black", ls="--",
                   label=f"Threshold ({CRITICALITY_THRESHOLD})"),
    ], fontsize=9, loc="upper right")
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, f"04_criticality_farm{farm}.png")
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")




def plot_missing_heatmap(farm):
    """Missing data heatmap for training data."""
    farm_events = [d for d in dataset if d["farm"] == farm]
    excluded    = _get_excluded(farm)
    missing_data = {}

    for d in farm_events[:10]:
        try:
            tr, _ = load_event(d["filepath"], farm)
            sig = [c for c in tr.columns
                   if c not in META_COLS and c not in excluded
                   and "avg" in c.lower()
                   and pd.api.types.is_numeric_dtype(tr[c])][:30]
            if sig:
                missing_data[f"E{d['event_id']}_{d['label'][:3]}"] = \
                    tr[sig].isna().mean()
            del tr
            gc.collect()
        except Exception:
            continue

    if not missing_data:
        return

    miss_df = pd.DataFrame(missing_data).T
    fig, ax = plt.subplots(figsize=(max(12, len(miss_df.columns)*0.4+3),
                                    max(3, len(miss_df)*0.55+3)))
    sns.heatmap(miss_df, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.4, linecolor="#ddd")
    ax.set_title(f"Farm {farm} — Missing Data Rate (Training, signal cols)",
                 fontsize=11, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90,
                       fontsize=5.5, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
    plt.tight_layout()
    p = os.path.join(PLOT_DIR, f"06_missing_farm{farm}.png")
    plt.savefig(p, dpi=130, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")


# Generate all plots
# FIX: time series plots now called AFTER event_scores is fully populated
# so pred_df references are valid

plot_event_balance()

for farm in ["A", "B", "C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    if not fe:
        continue

    # Anomaly score time series: 2 anomaly + 2 normal per farm
    # Prioritise detected anomaly events (most informative for dissertation)
    if farm in farm_results_all:
        detected_ids = {r["event_id"] for r in farm_results_all[farm]
                        if r["alarm_raised"] and r["label"] == "anomaly"}
    else:
        detected_ids = set()

    shown = 0
    # First: show detected anomaly events
    for d in fe:
        if shown >= 2:
            break
        if d["event_id"] in detected_ids:
            plot_anomaly_score_timeseries(
                d["event_id"], farm, d["label"], d["description"]
            )
            shown += 1
    # Then: show one missed anomaly + one normal
    for d in fe:
        if shown >= 4:
            break
        if d["label"] == "anomaly" and d["event_id"] not in detected_ids:
            plot_anomaly_score_timeseries(
                d["event_id"], farm, d["label"], d["description"]
            )
            shown += 1
    for d in fe:
        if shown >= 4:
            break
        if d["label"] == "normal":
            plot_anomaly_score_timeseries(
                d["event_id"], farm, d["label"], d["description"]
            )
            shown += 1

    # Per-farm criticality and feature importance
    if farm in farm_results_all:
        plot_criticality(farm, farm_results_all[farm])
    plot_missing_heatmap(farm)

plot_care_scores(care_scores)



print("\n⑧ Saving all outputs...")

def json_safe(obj):
    if isinstance(obj, dict):   return {k: json_safe(v) for k,v in obj.items()}
    if isinstance(obj, list):   return [json_safe(i) for i in obj]
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.bool_):       return bool(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    if isinstance(obj, pd.Timestamp):   return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list,dict,np.ndarray)) else False:
        return None
    return obj


# Full results JSON
all_event_results = []
for farm, farm_res in farm_results_all.items():
    all_event_results.extend(farm_res)

output = json_safe({
    "timestamp":   datetime.now().isoformat(),
    "pipeline":    "CARE Wind Turbine Anomaly Detection + Fault Classification Pipeline",
    "version":     "final",
    "parameters": {
        "criticality_threshold":     CRITICALITY_THRESHOLD,
        "criticality_threshold_sub": CRITICALITY_THRESHOLD_SUB,
        "beta":                      BETA,
        "if_contamination":          IF_CONTAMINATION,
        "if_n_estimators":           IF_N_ESTIMATORS,
        "pca_variance_target":       PCA_VARIANCE_TARGET,
        "smoothing_window":          SMOOTHING_WINDOW,
        "farm_threshold_pct":        FARM_THRESHOLD_PCT,
    },
    "eda":                  eda_results,
    "care_scores":          care_scores,
    "event_results":        all_event_results,
})

rj = os.path.join(OUTPUT_DIR, "results_final.json")
with open(rj, "w") as f:
    json.dump(output, f, indent=2)
print(f"  Saved: {rj}")

# Event results CSV — includes both anomaly detection and fault classification
results_clean = []
for r in all_event_results:
    results_clean.append({
        "event_id":          r["event_id"],
        "farm":              r["farm"],
        "label":             r["label"],
        "description":       r["description"],
        # Stage 1: anomaly detection
        "alarm_raised":      r["alarm_raised"],
        "alarm_source":      r.get("alarm_source", ""),
        "max_criticality":   r["max_criticality"],
        "crit_global":       r.get("crit_global", ""),
        "crit_subsystem":    r.get("crit_subsystem", ""),
        "anomaly_rate":      r["anomaly_rate"],
        "earliness":         r["earliness"],
        "top_subsystem":     r.get("top_subsystem", ""),
        "n_pred_rows":       r["n_pred_rows"],
    })

rc = os.path.join(OUTPUT_DIR, "event_results_final.csv")
pd.DataFrame(results_clean).to_csv(rc, index=False)
print(f"  Saved: {rc}")

# CARE score summary
care_summary_rows = []
for farm, cs in care_scores.items():
    care_summary_rows.append({"farm": farm, **cs})
cs_path = os.path.join(REPORT_DIR, "care_score_summary.csv")
pd.DataFrame(care_summary_rows).to_csv(cs_path, index=False)
print(f"  Saved: {cs_path}")





# Sensor registry already saved above

print("\n" + "=" * 80)
print("✅ PIPELINE COMPLETE")
print("=" * 80)
print(f"\nOutputs → {OUTPUT_DIR}")

print(f"\n{'─'*50}")
print("STAGE 1 — ANOMALY DETECTION (CARE Scores):")
for farm, cs in care_scores.items():
    print(f"  Farm {farm}: {cs['care']:.3f}  "
          f"(cov={cs['coverage']:.3f} acc={cs['accuracy']:.3f} "
          f"rel={cs['reliability']:.3f} ear={cs['earliness']:.3f})  "
          f"detected={cs['n_detected']}/{cs['n_anomaly']}")

CARE WIND TURBINE ANOMALY DETECTION PIPELINE

① Building sensor registries...
  Farm A: 81 column mappings — ✓ no duplicates
  Group                     Sensors  NBM
    electrical                   14  ✓
    temperature                  11  ✓
    generator_other               7  ✓
    angle                         4  ✗ (excluded)
    transformer                   3  ✓
    generator_stator              3  ✓
    environment                   2  ✓
    other                         2  ✓
    generator_bearing             2  ✓
    control                       1  ✓
    cooling_water                 1  ✓
    gearbox_other                 1  ✓
    gearbox_bearing               1  ✓
    hydraulic                     1  ✓
    wind                          1  ✓
  Farm B: 252 column mappings — ✓ no duplicates
  Group                     Sensors  NBM
    electrical                   16  ✓
    transformer                   9  ✓
    pitch                         6  ✓
    counter                     

In [ ]:
"""
================================================================================
SCRIPT 2 — WIND TURBINE FAULT CLASSIFICATION
================================================================================

"""

import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter, defaultdict
import xgboost as xgb

warnings.filterwarnings("ignore")

# ============================================================================
# ① PATHS & CONFIGURATION
# ============================================================================

BASE_DIR        = r"D:\files\project\CARE_To_Compare"   # FIXED
EVENT_INFO_PATH = os.path.join(BASE_DIR, "df_all__1_.csv")

FARMS = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "datasets"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "datasets"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "datasets"),
}

FEATURE_DESC = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "feature_description.csv"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "feature_description.csv"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "feature_description.csv"),
}

OUTPUT_DIR = os.path.join(BASE_DIR, "fault_classification_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_SEP       = ";"
FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS = {0, 2}

CONSECUTIVE_WINDOW     = 5
TRAIN_ON_PRED_WINDOW   = True
CONFIDENCE_THRESHOLD   = 0.45
MIN_EXAMPLES_FOR_CLASS = 3
OVERSAMPLE_NOISE_STD   = 0.02

XGB_PARAMS_SMALL = dict(
    max_depth=4,
    n_estimators=300,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    reg_alpha=0.1,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

FAULT_SENSOR_GROUPS = {
    "transformer":         ["transformer", "electrical"],
    "gearbox":             ["gearbox_bearing", "gearbox_oil", "gearbox_other", "vibration"],
    "gearbox_oil":         ["gearbox_oil", "gearbox_other"],
    "generator_bearing":   ["generator_bearing", "generator_stator",
                            "generator_cooling", "generator_other", "electrical"],
    "hydraulic":           ["hydraulic"],
    "rotor_bearing":       ["rotor_bearing", "vibration"],
    "pitch_electrical":    ["pitch", "electrical"],
    "pitch_mechanical":    ["pitch"],
    "converter":           ["electrical", "vibration"],
    "communication":       ["electrical"],
    "rotor_brake":         ["hydraulic", "rotor_bearing"],
    "yaw":                 ["rotor", "electrical"],
    "cooling_water":       ["cooling_water", "generator_cooling"],
    "electrical_internal": ["electrical"],
    "normal":              [],
}

OPERATIONAL_GROUPS = {"environment", "wind", "rotor", "electrical"}

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",   ["planetary bearing", "gearbox bearing",
                                 "bearing on high speed shaft"]),
    (11,  "gearbox_oil",       ["gearbox oil", "gear oil", "oil inlet temperature",
                                 "oil pressure", "oil level", "oil tank", "oil temperature"]),
    (12,  "gearbox_other",     ["gearbox rotational", "gearbox speed",
                                 "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing", ["generator bearing"]),
    (21,  "generator_stator",  ["stator winding", "stator"]),
    (22,  "generator_cooling", ["generator cooling", "cooling air",
                                 "cooling water temp. generator",
                                 "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",   ["generator acceleration", "generator angle",
                                 "generator rpm", "generator converter",
                                 "generator rms", "generator"]),
    (30,  "rotor_bearing",     ["rotor bearing", "axial bearing",
                                 "rotor bearing inner ring"]),
    (40,  "hydraulic",         ["hydraulic pressure", "hydraulic oil",
                                 "hydraulic pump", "hydraulic tank",
                                 "hydraulic group", "aggregate system pressure",
                                 "azimuth circuit", "azimuth memory",
                                 "rotor brake", "locking memory"]),
    (50,  "pitch",             ["pitch angle", "pitch motor", "pitch axis",
                                 "position motor axis", "position rotor blade",
                                 "motor temperature kty", "motor temperature pt",
                                 "battery charge", "battery discharge",
                                 "rpm motor axis", "min pitch",
                                 "motor current axis", "motor temperature axis"]),
    (60,  "vibration",         ["vibration", "drive train vibration", "tower vibration"]),
    (70,  "transformer",       ["transformer", "transformator",
                                 "hv transformer", "transformer cell"]),
    (80,  "electrical",        ["voltage", "current", "frequency",
                                 "active power", "reactive power", "apparent power",
                                 "grid power", "grid voltage", "grid frequency",
                                 "available power", "inverter torque", "cable load",
                                 "dc link voltage", "internal consumption"]),
    (90,  "wind",              ["wind speed", "wind direction", "anemometer",
                                 "estimated windspeed", "absolute wind", "relative wind"]),
    (100, "environment",       ["ambient temperature", "outside temperature",
                                 "nacelle temperature", "nacelle external",
                                 "hub temperature", "board temperature", "nacelle outside"]),
    (110, "cooling_water",     ["water temperature", "water pressure",
                                 "water flow", "water conductivity",
                                 "water cooler", "cooling water", "flow meter stator"]),
    (120, "rotor",             ["rotor speed", "rotor rpm", "nacelle direction", "yaw"]),
]


def assign_group(description):
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


# ============================================================================
# ② CORRECTED FAULT LABELS
# ============================================================================

FAULT_LABELS = {
    "A": {
        68: "transformer",
        22: "hydraulic",  72: "gearbox",
        73: "hydraulic",   0: "generator_bearing",
        26: "hydraulic",  40: "generator_bearing",
        42: "hydraulic",  10: "gearbox",
        45: "hydraulic",  84: "hydraulic",
        51: "gearbox",
    },
    "B": {
        34: "transformer",  7: "transformer",
        53: "rotor_bearing", 27: "rotor_bearing",
        19: "transformer",  77: "rotor_bearing",
    },
    "C": {
        55: "pitch_electrical",  81: "converter",
        47: "hydraulic",         12: "gearbox_oil",
         4: "pitch_electrical",  18: "rotor_brake",
        28: "pitch_electrical",  39: "gearbox_oil",
        66: "pitch_electrical",  15: "pitch_electrical",
        78: "rotor_brake",       79: "communication",
        30: "pitch_electrical",  33: "pitch_mechanical",
        11: "pitch_electrical",  44: "cooling_water",
        49: "gearbox_oil",       31: "communication",
        67: "transformer",        9: "yaw",
        91: "pitch_electrical",   5: "electrical_internal",
        90: "communication",     70: "pitch_electrical",
        35: "converter",         16: "pitch_electrical",
        76: "pitch_electrical",
    },
}


# ============================================================================
# ③ SENSOR RENAMING
# ============================================================================

STAT_MAP = {"average": "avg", "std_dev": "std", "maximum": "max", "minimum": "min"}

import re


def _clean(text, is_unit=False):
    if pd.isna(text) or str(text).strip() in ("", "nan", "-", "none"):
        return ""
    t = str(text).strip()
    for old, new in [
        ("���C","degC"),("°C","degC"),("ºC","degC"),("½C","degC"),
        ("Celsius","degC"),("���","deg"),("°","deg"),("º","deg"),("½",""),
        ("/","_per_"),("%","pct"),(" ",""),("^",""),(".",""),
    ]:
        t = t.replace(old, new)
    t = re.sub(r"[^\x00-\x7F]+", "", t)
    if not is_unit:
        t = re.sub(r"[^\w\s]", " ", t)
        t = re.sub(r"\s+", "_", t.strip())
        t = re.sub(r"_+", "_", t).strip("_")
    else:
        t = re.sub(r"[^\w]", "", t)
    return t


def build_rename_dict(farm_id):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(FEATURE_DESC[farm_id], sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    rename, counter_new, angle_new, seen = {}, set(), set(), {}
    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc       = _clean(str(row.get("description", "")))
        unit       = _clean(str(row.get("unit", "")), is_unit=True)
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))
        stats_raw  = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]
        entries = [(f"{sid}_{STAT_MAP.get(s,s)}", STAT_MAP.get(s,s))
                   for s in stats] if stats else [(sid, "")]
        for old_col, sc in entries:
            parts   = [desc] + ([unit] if unit else []) + ([sc] if sc else [])
            new_col = "_".join(parts)
            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col
            rename[old_col] = new_col
            if is_counter: counter_new.add(new_col)
            if is_angle:   angle_new.add(new_col)
    return rename, counter_new, angle_new


print("Building sensor rename dictionaries...")
rename_dicts, counter_sets, angle_sets = {}, {}, {}
for farm in ["A", "B", "C"]:
    rd, co, an = build_rename_dict(farm)
    rename_dicts[farm] = rd
    counter_sets[farm] = co
    angle_sets[farm]   = an
    print(f"  Farm {farm}: {len(rd)} mappings")

META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}


def build_group_dict(farm_id):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(FEATURE_DESC[farm_id], sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    rdict = rename_dicts[farm_id]
    gdict = {}
    stat_map = {"average":"avg","std_dev":"std","maximum":"max","minimum":"min"}
    for _, row in df.iterrows():
        sid  = str(row["sensor_name"]).strip()
        desc = str(row.get("description", "")).strip()
        grp  = assign_group(desc)
        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]
        entries = [f"{sid}_{stat_map.get(s,s)}" for s in stats] if stats else [sid]
        for old_col in entries:
            gdict[rdict.get(old_col, old_col)] = grp
    return gdict


print("Building subsystem group dictionaries...")
group_dicts = {}
for farm in ["A", "B", "C"]:
    group_dicts[farm] = build_group_dict(farm)
    counts = {}
    for g in group_dicts[farm].values():
        counts[g] = counts.get(g, 0) + 1
    top = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  Farm {farm}: {len(group_dicts[farm])} mappings | top groups: {top}")


# ============================================================================
# ④ SENSOR SELECTION
# ============================================================================

def get_selected_columns(fault_types, farm_id):
    gdict    = group_dicts[farm_id]
    excluded = counter_sets[farm_id] | angle_sets[farm_id]
    target_groups = set(OPERATIONAL_GROUPS)
    for ft in fault_types:
        for grp in FAULT_SENSOR_GROUPS.get(ft, []):
            target_groups.add(grp)
    selected = {col for col, grp in gdict.items()
                if grp in target_groups and col not in excluded}
    return selected, target_groups


# ============================================================================
# ⑤ DATA LOADING
# ============================================================================

def load_event_selected(filepath, farm_id, selected_cols=None):
    rdict, chunks = rename_dicts[farm_id], []
    for chunk in pd.read_csv(filepath, sep=CSV_SEP, chunksize=10000,
                             low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)
        if selected_cols is not None:
            keep  = [c for c in chunk.columns
                     if c in META_COLS or (c in selected_cols and "avg" in c.lower())]
            chunk = chunk[[c for c in keep if c in chunk.columns]]
        num_cols = [c for c in chunk.columns
                    if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])]
        for col in num_cols:
            if col in counter_sets[farm_id]:
                continue
            vals = chunk[col].values.copy().astype(np.float32)
            n, i = len(vals), 0
            while i < n:
                if vals[i] == 0.0:
                    s = i
                    while i < n and vals[i] == 0.0: i += 1
                    if i - s >= 6: vals[s:i] = np.nan
                else:
                    i += 1
            chunk[col] = vals
        chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)
        del chunk
    df = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    pred_df      = df[df["train_test"] == "prediction"].copy()
    fault_rows_df = pd.DataFrame()
    if "status_type_id" in df.columns and "train_test" in df.columns:
        fault_rows_df = df[(df["train_test"] == "train") &
                           (df["status_type_id"] == 4)].copy()
    del df; gc.collect()
    return pred_df, fault_rows_df


# ============================================================================
# ⑤ FEATURE ENGINEERING
# ============================================================================

def engineer_features(df, farm_id):
    excluded = counter_sets[farm_id] | angle_sets[farm_id]
    df = df.copy()
    avg_cols = [c for c in df.columns
                if c not in META_COLS and c not in excluded
                and "avg" in c.lower() and pd.api.types.is_numeric_dtype(df[c])]
    new_feats = {}

    for col in avg_cols:
        s = df[col].astype(np.float32)
        for w, wname in [(6,"1hr"),(18,"3hr"),(36,"6hr")]:
            new_feats[f"{col}_roll{wname}_mean"] = s.rolling(w, min_periods=1).mean().astype(np.float32)
            new_feats[f"{col}_roll{wname}_std"]  = s.rolling(w, min_periods=1).std().fillna(0).astype(np.float32)

    temp_cols = [c for c in avg_cols if any(k in c.lower() for k in ["temp","degc"])]
    for col in temp_cols:
        s = df[col].astype(np.float32)
        new_feats[f"{col}_diff1"]  = s.diff(1).fillna(0).astype(np.float32)
        new_feats[f"{col}_diff6"]  = s.diff(6).fillna(0).astype(np.float32)
        new_feats[f"{col}_diff18"] = s.diff(18).fillna(0).astype(np.float32)
        new_feats[f"{col}_accel6"] = s.diff(6).diff(6).fillna(0).astype(np.float32)

    curr_cols = sorted([c for c in df.columns if "current" in c.lower()
                        and "phase" in c.lower() and "avg" in c.lower()])
    if len(curr_cols) >= 3:
        mat = df[curr_cols[:3]].values.astype(np.float32)
        mean_c = np.mean(mat, axis=1)
        new_feats["generator_current_imbalance"] = np.std(mat, axis=1)
        new_feats["generator_current_range"]     = np.ptp(mat, axis=1)
        new_feats["generator_current_cv"]        = np.std(mat, axis=1) / (mean_c + 1e-6)

    stator_cols = sorted([c for c in df.columns if "stator" in c.lower()
                          and "avg" in c.lower() and pd.api.types.is_numeric_dtype(df[c])])
    if len(stator_cols) >= 3:
        mat = df[stator_cols[:3]].values.astype(np.float32)
        new_feats["generator_stator_temp_imbalance"] = np.std(mat, axis=1)
        new_feats["generator_stator_temp_range"]     = np.ptp(mat, axis=1)

    trans_cols = sorted([c for c in df.columns if "transformer" in c.lower()
                         and ("temp" in c.lower() or "degc" in c.lower()) and "avg" in c.lower()])
    if len(trans_cols) >= 3:
        mat = df[trans_cols[:3]].values.astype(np.float32)
        new_feats["transformer_temp_imbalance"] = np.std(mat, axis=1)
        new_feats["transformer_temp_range"]     = np.ptp(mat, axis=1)

    gb_bear_cols = sorted([c for c in df.columns if "planetary" in c.lower()
                           and "temp" in c.lower() and "avg" in c.lower()])
    if len(gb_bear_cols) >= 4:
        mat = df[gb_bear_cols].values.astype(np.float32)
        new_feats["planetary_bearing_temp_mean"]  = np.mean(mat, axis=1)
        new_feats["planetary_bearing_temp_std"]   = np.std(mat,  axis=1)
        new_feats["planetary_bearing_temp_range"] = np.ptp(mat,  axis=1)

    nacelle_col = next((c for c in df.columns if "nacelle" in c.lower()
                        and "temp" in c.lower() and "avg" in c.lower()), None)
    ambient_col = next((c for c in df.columns if "ambient" in c.lower()
                        and "avg" in c.lower()), None)
    ref_col = nacelle_col or ambient_col

    if ref_col:
        ref = df[ref_col].astype(np.float32)
        for col in avg_cols:
            if any(k in col.lower() for k in ["gearbox_oil","generator_bearing",
                                               "hydraulic_oil","transformer","hydraulic_oil_tank"]):
                new_feats[f"{col}_normalised"] = df[col].astype(np.float32) - ref

    power_col = next((c for c in df.columns if "active_power" in c.lower()
                      and "avg" in c.lower() and c not in counter_sets[farm_id]), None)
    wind_col  = next((c for c in df.columns if "wind" in c.lower()
                      and "speed" in c.lower() and "avg" in c.lower()), None)
    rotor_col = next((c for c in df.columns if ("rotor" in c.lower() or "generator" in c.lower())
                      and ("rpm" in c.lower() or "speed" in c.lower()) and "avg" in c.lower()), None)
    gbox_oil  = next((c for c in df.columns if "gearbox_oil" in c.lower()
                      and "temp" in c.lower() and "avg" in c.lower()), None)
    hydr_col  = next((c for c in df.columns if "hydraulic" in c.lower()
                      and "temp" in c.lower() and "avg" in c.lower()), None)
    pitch_col = next((c for c in df.columns if "pitch" in c.lower()
                      and "avg" in c.lower() and c not in angle_sets[farm_id]), None)

    if power_col and wind_col:
        wind = df[wind_col].astype(np.float32)
        pwr  = df[power_col].astype(np.float32)
        new_feats["power_curve_deviation"] = (pwr - 0.5 * wind**3).astype(np.float32)
        new_feats["power_fluctuation_std"] = pwr.rolling(18, min_periods=1).std().fillna(0).astype(np.float32)
    if rotor_col and power_col:
        new_feats["generator_efficiency"] = (df[power_col].astype(np.float32) /
                                             (df[rotor_col].astype(np.float32) + 1))
    if gbox_oil and rotor_col:
        new_feats["gearbox_temp_per_rpm"]  = df[gbox_oil].astype(np.float32) / (df[rotor_col].astype(np.float32) + 1)
    if gbox_oil and ref_col:
        new_feats["gearbox_oil_temp_rise"] = df[gbox_oil].astype(np.float32) - df[ref_col].astype(np.float32)
    if hydr_col and pitch_col:
        new_feats["hydraulic_stress"] = (df[hydr_col].astype(np.float32) *
                                         df[pitch_col].astype(np.float32).abs())

    new_df = pd.concat([df, pd.DataFrame(new_feats, index=df.index)], axis=1)
    del new_feats; gc.collect()
    feat_cols = [c for c in new_df.columns
                 if c not in META_COLS
                 and c not in counter_sets[farm_id]
                 and c not in angle_sets[farm_id]
                 and pd.api.types.is_numeric_dtype(new_df[c])]
    new_df[feat_cols] = new_df[feat_cols].fillna(0).astype(np.float32)
    return new_df, feat_cols


# ============================================================================
# ⑥ DETECTION RULES
# ============================================================================

def detect_failure(predictions, true_label, timestamps):
    n, w = len(predictions), CONSECUTIVE_WINDOW
    for i in range(n - w + 1):
        if np.all(predictions[i:i+w] == true_label):
            if timestamps is not None and len(timestamps) > 0:
                det_ts = pd.Timestamp(timestamps[i])
                end_ts = pd.Timestamp(timestamps[-1])
                return True, (end_ts - det_ts).total_seconds() / 86400
            return True, None
    return False, None


def detect_failure_window(predictions, true_label, window_frac=0.10):
    n = len(predictions)
    return bool(np.any(predictions[int(n*(1-window_frac)):] == true_label))


# ============================================================================
# ⑦ OVERSAMPLING
# ============================================================================

def oversample_minority_classes(X, y, min_examples=MIN_EXAMPLES_FOR_CLASS,
                                 noise_std=OVERSAMPLE_NOISE_STD):
    unique_cls, counts = np.unique(y, return_counts=True)
    X_aug, y_aug = [X], [y]
    feat_std = np.std(X, axis=0) + 1e-8
    rng = np.random.default_rng(42)
    for cls, cnt in zip(unique_cls, counts):
        if cnt < min_examples:
            needed  = min_examples - cnt
            src_idx = np.where(y == cls)[0]
            rep_idx = src_idx[np.arange(needed) % len(src_idx)]
            noise   = rng.normal(0, noise_std, size=(needed, X.shape[1])).astype(np.float32)
            X_aug.append(X[rep_idx] + noise * feat_std)
            y_aug.append(np.full(needed, cls, dtype=np.int32))
    return np.vstack(X_aug), np.concatenate(y_aug)


# ============================================================================
# ⑧ ONE LOEO FOLD
# ============================================================================

def run_one_fold(train_events, test_event, farm_id, verbose=True):
    fault_train = [d for d in train_events if d["fault_type"] != "normal"]
    if len(fault_train) < 2:
        return None

    fault_types_in_train = list(set(d["fault_type"] for d in fault_train))
    selected_cols, target_groups = get_selected_columns(fault_types_in_train, farm_id)

    if verbose:
        print(f"      Groups: {sorted(target_groups)} | selected cols: {len(selected_cols)}")

    X_parts, y_parts, feat_cols_ref = [], [], None

    for d in fault_train:
        try:
            pr, fault_rows = load_event_selected(d["filepath"], farm_id, selected_cols)
            pred_window_size = len(pr)
            if pred_window_size > 0:
                pr_eng, fc = engineer_features(pr, farm_id)
                if feat_cols_ref is None: feat_cols_ref = fc
                fc_use = [c for c in feat_cols_ref if c in pr_eng.columns]
                X = np.nan_to_num(pr_eng[fc_use].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
                X_parts.append(X); y_parts.append(np.full(len(X), d["label_int"], dtype=np.int32))
                del pr, pr_eng
            if len(fault_rows) > 0 and feat_cols_ref is not None:
                max_fault_rows = max(pred_window_size, 500) * 2
                if len(fault_rows) > max_fault_rows:
                    fault_rows = fault_rows.sample(max_fault_rows, random_state=42)
                fr_eng, _ = engineer_features(fault_rows, farm_id)
                fc_use = [c for c in feat_cols_ref if c in fr_eng.columns]
                X_f = np.nan_to_num(fr_eng[fc_use].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
                X_parts.append(X_f); y_parts.append(np.full(len(X_f), d["label_int"], dtype=np.int32))
                del fault_rows, fr_eng
            gc.collect()
        except Exception as e:
            if verbose: print(f"      Skip train {d['event_id']}: {e}")
            continue

    if not X_parts: return None

    X_train = np.vstack(X_parts).astype(np.float32)
    y_train = np.concatenate(y_parts).astype(np.int32)
    del X_parts, y_parts; gc.collect()

    X_train, y_train = oversample_minority_classes(X_train, y_train)

    unique_cls    = sorted(np.unique(y_train).tolist())
    n_cls         = len(unique_cls)
    remap         = {old: new for new, old in enumerate(unique_cls)}
    reverse_remap = {new: old for old, new in remap.items()}
    y_remapped    = np.array([remap[y] for y in y_train])

    ulabels, counts = np.unique(y_remapped, return_counts=True)
    weights = {c: len(y_remapped) / (n_cls * cnt) for c, cnt in zip(ulabels, counts)}
    sw = np.array([weights[y] for y in y_remapped])

    params = XGB_PARAMS_SMALL.copy()
    params["objective"]   = "multi:softprob" if n_cls > 2 else "binary:logistic"
    params["eval_metric"] = "mlogloss"       if n_cls > 2 else "logloss"
    if n_cls > 2: params["num_class"] = n_cls

    clf = xgb.XGBClassifier(**params)
    try:
        clf.fit(X_train, y_remapped, sample_weight=sw)
    except Exception as e:
        if verbose: print(f"      Fit failed: {e}")
        del X_train, y_train; gc.collect()
        return None
    del X_train, y_train; gc.collect()

    try:
        pr, _ = load_event_selected(test_event["filepath"], farm_id, selected_cols)
        if len(pr) == 0: del clf; gc.collect(); return None
        pr, _ = engineer_features(pr, farm_id)
        fc_use = [c for c in feat_cols_ref if c in pr.columns]
        X_test = np.nan_to_num(pr[fc_use].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        if X_test.shape[1] < len(feat_cols_ref):
            X_test = np.hstack([X_test, np.zeros((len(X_test), len(feat_cols_ref)-X_test.shape[1]), dtype=np.float32)])
        timestamps = pr["time_stamp"].values if "time_stamp" in pr.columns else None
        probas         = clf.predict_proba(X_test)
        preds_remapped = np.argmax(probas, axis=1)
        max_probs      = np.max(probas, axis=1)
        preds_conf     = np.where(max_probs >= CONFIDENCE_THRESHOLD, preds_remapped, -1)
        preds_orig     = np.array([reverse_remap[p] if p >= 0 else -1 for p in preds_conf])
        del pr, X_test; gc.collect()
    except Exception as e:
        if verbose: print(f"      Test failed: {e}")
        del clf; gc.collect()
        return None

    true_label = test_event["label_int"]
    is_fault   = test_event["fault_type"] != "normal"
    detected, horizon, detected_window = False, None, False
    if is_fault and true_label in unique_cls:
        detected, horizon = detect_failure(preds_orig, true_label, timestamps)
        detected_window   = detect_failure_window(preds_orig, true_label)

    valid = preds_orig[preds_orig >= 0]
    if len(valid) > 0:
        pred_int  = Counter(valid.tolist()).most_common(1)[0][0]
        pred_type = {d["label_int"]: d["fault_type"] for d in fault_train}.get(pred_int, "unknown")
    else:
        pred_type = "uncertain"

    mean_conf   = float(np.mean(max_probs))
    uncert_frac = float(np.mean(preds_orig == -1))

    importance = {}
    if hasattr(clf, "feature_importances_"):
        n_base = min(len(feat_cols_ref), len(clf.feature_importances_))
        importance = dict(sorted(
            {col: float(imp) for col, imp in zip(feat_cols_ref[:n_base], clf.feature_importances_[:n_base])}.items(),
            key=lambda x: x[1], reverse=True)[:10])

    del clf, probas, preds_remapped, preds_conf, preds_orig; gc.collect()

    status = "✓" if detected else ("~" if detected_window else ("✗" if is_fault else "—"))
    if verbose:
        hor_str  = f"  ({horizon:.1f}d)" if horizon else ""
        unc_str  = f"  unc={uncert_frac:.0%}" if uncert_frac > 0.1 else ""
        win_str  = "  [window✓]" if detected_window and not detected else ""
        print(f"    E{test_event['event_id']:>4} [{test_event['fault_type']:22}] "
              f"pred=[{pred_type:22}] {status}{hor_str}  conf={mean_conf:.2f}{unc_str}{win_str}")

    return {
        "event_id":          test_event["event_id"],
        "farm":              farm_id,
        "fault_type":        test_event["fault_type"],
        "pred_type":         pred_type,
        "label_int":         true_label,
        "is_fault":          is_fault,
        "detected":          bool(detected),
        "detected_window":   bool(detected_window),
        "horizon_days":      round(horizon, 2) if horizon else None,
        "mean_confidence":   round(mean_conf, 4),
        "uncertain_frac":    round(uncert_frac, 4),
        "n_train_events":    len(fault_train),
        "n_classes_train":   n_cls,
        "n_selected_cols":   len(selected_cols),
        "sensor_groups":     sorted(target_groups),
        "feature_importance": importance,
    }


# ============================================================================
# ⑨ FARM LOEO
# ============================================================================

def run_farm_loeo(farm_id, dataset):
    farm_events = [d for d in dataset if d["farm"] == farm_id]
    if not farm_events:
        print(f"  No events for Farm {farm_id}"); return []

    fault_types = sorted(set(d["fault_type"] for d in farm_events))
    label_map   = {ft: i for i, ft in enumerate(fault_types)}
    for d in farm_events:
        d["label_int"] = label_map[d["fault_type"]]

    anomaly_events = [d for d in farm_events if d["label"] == "anomaly"]
    normal_events  = [d for d in farm_events if d["label"] == "normal"]
    print(f"\n  Farm {farm_id}: {len(anomaly_events)} anomaly events | {len(normal_events)} normal events")
    print(f"  Fault types: {fault_types}")

    results = []
    for i, test_event in enumerate(farm_events, 1):
        train_events = [d for d in farm_events if d["event_id"] != test_event["event_id"]]
        print(f"\n  [{i}/{len(farm_events)}] E{test_event['event_id']} [{test_event['fault_type']}]")
        result = run_one_fold(train_events, test_event, farm_id)
        if result: results.append(result)
    return results


# ============================================================================
# ⑩ METRICS
# ============================================================================

def compute_metrics(results, farm_id):
    fault_results = [r for r in results if r["is_fault"] and r["fault_type"] != "normal"]
    if not fault_results: return {}

    total        = len(fault_results)
    detected     = sum(1 for r in fault_results if r["detected"])
    detected_win = sum(1 for r in fault_results if r.get("detected_window"))
    rate         = detected / total
    rate_win     = detected_win / total

    classifiable = [r for r in fault_results if r.get("pred_type") not in ("uncertain","unknown",None)]
    n_correct    = sum(1 for r in classifiable if r.get("pred_type") == r["fault_type"])
    class_acc    = n_correct / len(classifiable) if classifiable else 0.0
    uncertain_n  = sum(1 for r in fault_results if r.get("pred_type") in ("uncertain",None))
    mean_conf    = float(np.mean([r.get("mean_confidence",0) for r in fault_results]))

    by_type = {}
    for r in fault_results:
        ft = r["fault_type"]
        if ft not in by_type:
            by_type[ft] = {"total":0,"detected":0,"detected_window":0,"horizons":[],"correct":0,"classifiable":0}
        by_type[ft]["total"] += 1
        if r["detected"]:              by_type[ft]["detected"] += 1
        if r.get("detected_window"):   by_type[ft]["detected_window"] += 1
        if r["horizon_days"] is not None: by_type[ft]["horizons"].append(r["horizon_days"])
        if r.get("pred_type") not in ("uncertain","unknown",None):
            by_type[ft]["classifiable"] += 1
            if r.get("pred_type") == ft: by_type[ft]["correct"] += 1

    all_horizons = [r["horizon_days"] for r in fault_results if r["horizon_days"] is not None]
    avg_horizon  = float(np.mean(all_horizons)) if all_horizons else 0.0

    print(f"\n  {'─'*70}")
    print(f"  Farm {farm_id} — FAULT CLASSIFICATION RESULTS")
    print(f"  {'─'*70}")
    print(f"  Detection rate (5-consecutive): {detected}/{total} = {rate:.1%}")
    print(f"  Detection rate (last-10% win):  {detected_win}/{total} = {rate_win:.1%}")
    print(f"  Classification acc:   {n_correct}/{len(classifiable)} = {class_acc:.1%}  (excl. {uncertain_n} uncertain)")
    print(f"  Mean confidence:      {mean_conf:.2f}")
    print(f"  Avg detection horizon:{avg_horizon:.1f} days before failure")
    print(f"\n  {'Fault Type':25} {'Det':>4}{'Win':>4}{'Tot':>4}  {'Det%':>6}  {'Win%':>6}  {'Cls%':>6}  {'Horizon':>10}")
    print(f"  {'─'*75}")
    for ft in sorted(by_type.keys()):
        v  = by_type[ft]
        dr = v["detected"] / v["total"] if v["total"] > 0 else 0
        wr = v["detected_window"] / v["total"] if v["total"] > 0 else 0
        ca = v["correct"] / v["classifiable"] if v["classifiable"] > 0 else 0
        h_ = np.mean(v["horizons"]) if v["horizons"] else 0
        print(f"  {ft:25} {v['detected']:4d}{v['detected_window']:4d}{v['total']:4d}  {dr:6.1%}  {wr:6.1%}  {ca:6.1%}  {h_:8.1f}d")

    return {
        "farm": farm_id, "total_faults": total, "detected": detected,
        "detected_window": detected_win, "detection_rate": round(rate,4),
        "detection_rate_window": round(rate_win,4), "classification_acc": round(class_acc,4),
        "n_uncertain": uncertain_n, "mean_confidence": round(mean_conf,4),
        "avg_horizon_days": round(avg_horizon,2), "by_type": by_type,
    }


# ============================================================================
# ⑪ PLOTS  — filenames match Script 5 expectations
# ============================================================================

def plot_results(all_metrics, all_results, output_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    farms = [f for f in ["A","B","C"] if f in all_metrics]
    if not farms: return

    # ---- 05_stage2_performance_by_farm.png
    # Paired bars: detection rate (green) + classification accuracy (blue)
    det_rates = [all_metrics[f]["detection_rate"]    for f in farms]
    cls_accs  = [all_metrics[f]["classification_acc"] for f in farms]
    x = np.arange(len(farms)); width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    b1 = ax.bar(x - width/2, det_rates, width, label="Detection rate (5-consecutive)",
                color="#2ca02c", alpha=0.88, edgecolor="white")
    b2 = ax.bar(x + width/2, cls_accs,  width, label="Classification accuracy",
                color="#4878d0", alpha=0.88, edgecolor="white")
    ax.bar_label(b1, labels=[f"{v:.1%}" for v in det_rates], padding=4, fontsize=11, fontweight="bold")
    ax.bar_label(b2, labels=[f"{v:.1%}" for v in cls_accs],  padding=4, fontsize=11, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels([f"Farm {f}" for f in farms], fontsize=12)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Rate", fontsize=12)
    ax.set_title("Detection Rate and Classification Accuracy by Farm\n"
                 "(LOEO, XGBoost multiclass, 5-consecutive prediction rule)",
                 fontsize=12, fontweight="bold")
    ax.axhline(0.5, color="#888", linestyle="--", linewidth=1.0, alpha=0.5)
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(output_dir, "05_stage2_performance_by_farm.png")
    plt.savefig(p, dpi=200, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}")

    # ---- 06_stage2_horizon_by_farm.png
    horizons = [all_metrics[f]["avg_horizon_days"] for f in farms]
    colours  = ["#2ca02c" if h >= 30 else "#ff7f0e" if h >= 8 else "#e05252" for h in horizons]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar([f"Farm {f}" for f in farms], horizons, color=colours, alpha=0.88, edgecolor="white")
    ax.axhline(30, color="#2ca02c", linestyle="--", linewidth=1.3, alpha=0.7, label="30-day planned maintenance threshold")
    ax.axhline(8,  color="#ff7f0e", linestyle=":",  linewidth=1.2, alpha=0.7, label="8-day expedited response threshold")
    ax.bar_label(bars, fmt="%.1f d", padding=3, fontsize=11, fontweight="bold")
    ax.set_ylabel("Average Warning Horizon (days)", fontsize=12)
    ax.set_title("Average Early Warning Time by Farm", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = os.path.join(output_dir, "06_stage2_horizon_by_farm.png")
    plt.savefig(p, dpi=200, bbox_inches="tight"); plt.close()
    print(f"  Saved: {p}")

    # ---- 07_stage2_farmC_by_type.png
    farm_c_results = [r for r in all_results if r["farm"] == "C" and r["is_fault"]]
    if farm_c_results:
        by_type = {}
        for r in farm_c_results:
            ft = r["fault_type"]
            if ft not in by_type: by_type[ft] = {"total":0,"detected":0}
            by_type[ft]["total"] += 1
            if r["detected"]: by_type[ft]["detected"] += 1
        types  = sorted(by_type.keys())
        rates_ = [by_type[t]["detected"] / by_type[t]["total"] for t in types]
        totals = [by_type[t]["total"] for t in types]
        cols_  = ["#2ca02c" if r >= 0.5 else "#d62728" for r in rates_]

        fig, ax = plt.subplots(figsize=(13, 5))
        bars = ax.bar(types, rates_, color=cols_, alpha=0.88, edgecolor="white")
        for bar, rate, total in zip(bars, rates_, totals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f"{rate:.0%}\n(n={total})", ha="center", va="bottom", fontsize=9)
        ax.set_ylim(0, 1.2); ax.set_ylabel("Detection Rate", fontsize=11)
        ax.set_title("Farm C — Detection Rate by Fault Type", fontsize=12, fontweight="bold")
        ax.axhline(0.5, color="#555", ls="--", lw=1.1, alpha=0.6)
        ax.tick_params(axis="x", rotation=30); ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        p = os.path.join(output_dir, "07_stage2_farmC_by_type.png")
        plt.savefig(p, dpi=200, bbox_inches="tight"); plt.close()
        print(f"  Saved: {p}")

    # ---- 14_stage2_horizon_by_fault_type.png
    detected_results = [r for r in all_results
                        if r["is_fault"] and r["detected"] and r["horizon_days"] is not None]
    if detected_results:
        by_farm_ft = {}
        for r in detected_results:
            key = (r["farm"], r["fault_type"])
            if key not in by_farm_ft: by_farm_ft[key] = []
            by_farm_ft[key].append(r["horizon_days"])

        rows = sorted([(f"{f} — {ft.replace('_',' ')}", np.mean(horizons), f)
                        for (f, ft), horizons in by_farm_ft.items()],
                       key=lambda x: x[1], reverse=True)
        labels_  = [r[0] for r in rows]
        values_  = [r[1] for r in rows]
        farm_ids = [r[2] for r in rows]
        colours_ = {"A":"#4878d0","B":"#2ca02c","C":"#e05252"}
        bar_cols = [colours_.get(f,"#aaaaaa") for f in farm_ids]

        fig, ax = plt.subplots(figsize=(14, 5))
        bars = ax.bar(labels_, values_, color=bar_cols, alpha=0.88, edgecolor="white")
        ax.axhline(30, color="#2ca02c", linestyle="--", linewidth=1.3, alpha=0.7, label="30-day threshold")
        ax.axhline(8,  color="#ff7f0e", linestyle=":",  linewidth=1.2, alpha=0.7, label="8-day threshold")
        ax.bar_label(bars, fmt="%.1f d", padding=2, fontsize=9)
        ax.set_ylabel("Average Warning Horizon (days)", fontsize=11)
        ax.set_title("Warning Horizon by Farm and Fault Type", fontsize=12, fontweight="bold")
        ax.legend(fontsize=9); ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        p = os.path.join(output_dir, "14_stage2_horizon_by_fault_type.png")
        plt.savefig(p, dpi=200, bbox_inches="tight"); plt.close()
        print(f"  Saved: {p}")

    # ---- Feature importance (supplementary)
    all_importance = {}
    for r in all_results:
        for feat, imp in r.get("feature_importance", {}).items():
            all_importance[feat] = all_importance.get(feat, 0) + imp
    if all_importance:
        top    = sorted(all_importance.items(), key=lambda x: x[1], reverse=True)[:15]
        fig, ax = plt.subplots(figsize=(11, 5))
        bars = ax.barh([t[0] for t in top], [t[1] for t in top], color="#4878d0", alpha=0.85)
        ax.set_xlabel("Cumulative Feature Importance", fontsize=10)
        ax.set_title("Top 15 Features — All Farms Combined", fontsize=11, fontweight="bold")
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        p = os.path.join(output_dir, "fault_class_feature_importance.png")
        plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  Saved: {p}")


# ============================================================================
# ⑫ MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("WIND TURBINE FAULT CLASSIFICATION — ALL FARMS (LOEO)")
    print("=" * 80)
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}\n")
    start = datetime.now()

    event_info = pd.read_csv(EVENT_INFO_PATH)
    event_info["farm"]        = event_info["Farm"].map(FARM_NAME_MAP)
    event_info["event_id"]    = event_info["event_id"].astype(str)
    event_info["event_label"] = event_info["event_label"].str.strip()
    event_info["event_description"] = event_info["event_description"].fillna("normal").astype(str)

    dataset = []
    for _, row in event_info.iterrows():
        farm = row["farm"]
        if pd.isna(farm): continue
        eid      = row["event_id"]
        eid_int  = int(eid) if eid.isdigit() else -1
        label    = row["event_label"]
        filepath = os.path.join(FARMS[farm], f"{eid}.csv")
        if not os.path.exists(filepath): continue
        fault_type = FAULT_LABELS.get(farm, {}).get(eid_int, "normal")
        if label == "normal": fault_type = "normal"
        dataset.append({
            "event_id": eid, "farm": farm, "label": label,
            "fault_type": fault_type, "filepath": filepath,
            "description": row["event_description"],
        })

    for farm in ["A","B","C"]:
        fe = [d for d in dataset if d["farm"] == farm]
        ft_counts = Counter(d["fault_type"] for d in fe if d["label"] == "anomaly")
        print(f"Farm {farm}: {len(fe)} events | anomaly fault types: {dict(ft_counts)}")

    all_results, all_metrics = [], {}

    for farm in ["A","B","C"]:
        print(f"\n{'='*80}")
        print(f"FARM {farm} — FAULT CLASSIFICATION (LOEO)")
        print("=" * 80)
        farm_results = run_farm_loeo(farm, dataset)
        all_results.extend(farm_results)
        if farm_results:
            metrics = compute_metrics(farm_results, farm)
            if metrics: all_metrics[farm] = metrics

    print(f"\n{'='*80}")
    print("OVERALL SUMMARY")
    print("=" * 80)
    for farm in ["A","B","C"]:
        if farm in all_metrics:
            m = all_metrics[farm]
            print(f"  Farm {farm}: {m['detected']}/{m['total_faults']} = {m['detection_rate']:.1%} "
                  f"(5-consec) | {m['detected_window']}/{m['total_faults']} = "
                  f"{m['detection_rate_window']:.1%} (last-10%)  |  avg warning: {m['avg_horizon_days']:.1f} days")

    print("\nGenerating plots...")
    plot_results(all_metrics, all_results, OUTPUT_DIR)

    # ---- Save JSON ----
    def json_safe(obj):
        if isinstance(obj, dict):             return {k: json_safe(v) for k,v in obj.items()}
        if isinstance(obj, list):             return [json_safe(i) for i in obj]
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.bool_):         return bool(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        return obj

    out = {
        "timestamp": datetime.now().isoformat(),
        "script":    "script_2_fault_classification_fixed",
        "consecutive_window": CONSECUTIVE_WINDOW,
        "metrics":   all_metrics,
        "results":   all_results,
    }
    rj = os.path.join(OUTPUT_DIR, "fault_classification_results.json")
    with open(rj, "w") as f:
        json.dump(json_safe(out), f, indent=2)
    print(f"\nSaved: {rj}")

    # ---- Save CSV — booleans cast to int so pandas writes 0/1 not True/False ----
    rc = os.path.join(OUTPUT_DIR, "fault_classification_results.csv")
    df_out = pd.DataFrame(all_results).drop(columns=["feature_importance"], errors="ignore")
    for col in ["detected", "detected_window", "is_fault"]:
        if col in df_out.columns:
            df_out[col] = df_out[col].astype(int)   # FIXED: bool → int
    df_out.to_csv(rc, index=False)
    print(f"Saved: {rc}")

    print(f"\nRuntime: {datetime.now() - start}")
    print("✅ DONE")


if __name__ == "__main__":
    main()

Building sensor rename dictionaries...
  Farm A: 81 mappings
  Farm B: 252 mappings
  Farm C: 952 mappings
Building subsystem group dictionaries...
  Farm A: 81 mappings | top groups: [('electrical', 29), ('other', 18), ('generator_other', 10), ('rotor', 5), ('pitch', 4)]
  Farm B: 252 mappings | top groups: [('electrical', 80), ('transformer', 36), ('pitch', 28), ('wind', 20), ('gearbox_bearing', 16)]
  Farm C: 952 mappings | top groups: [('electrical', 248), ('other', 152), ('pitch', 88), ('gearbox_oil', 64), ('gearbox_bearing', 64)]
WIND TURBINE FAULT CLASSIFICATION — ALL FARMS (LOEO)
Started: 22:08:42

Farm A: 22 events | anomaly fault types: {'transformer': 1, 'hydraulic': 6, 'gearbox': 3, 'generator_bearing': 2}
Farm B: 15 events | anomaly fault types: {'transformer': 3, 'rotor_bearing': 3}
Farm C: 58 events | anomaly fault types: {'pitch_electrical': 11, 'converter': 2, 'hydraulic': 1, 'gearbox_oil': 3, 'rotor_brake': 2, 'communication': 3, 'pitch_mechanical': 1, 'cooling_water'

In [2]:
"""
================================================================================
SCRIPT 3 — CROSS-FARM FAULT CLASSIFICATION VALIDATION
================================================================================
Tests whether fault signatures learned on one wind farm generalise to another.

Experimental design:
  Direction 1: Train on Farm A → Predict Farm C
  Direction 2: Train on Farm C → Predict Farm A
  Direction 3: Train on Farm B → Predict Farm A and Farm C (supplementary)

Feature representation:
  Raw sensor data cannot be used directly — Farm A has 86 features,
  Farm C has 957. Instead, we use SUBSYSTEM-LEVEL AGGREGATE features:
  for each physical subsystem group (gearbox_bearing, hydraulic, transformer,
  etc.) we compute mean, std, min, max, and trend of all sensors in that group.
  This produces a fixed-length representation (~50 features) that is
  physically meaningful and farm-agnostic.

  This approach is motivated by domain knowledge: a gearbox bearing failure
  manifests as elevated, trending temperatures in the gearbox bearing group
  regardless of whether the farm has 1 or 16 planetary bearing sensors.

Subsystem groups used (all present in every farm):
  gearbox_bearing, gearbox_oil, generator_bearing, generator_stator,
  hydraulic, transformer, environment, wind, electrical, rotor,
  pitch, vibration (where available)

Evaluation:
  For each test event in the target farm:
    - Model trained on ALL source farm events
    - Prediction: 5 consecutive correct classifications = detected
    - Metrics: detection rate per fault type + detection horizon

Fault type compatibility across farms:
  Comparable:   transformer, gearbox, hydraulic, generator_bearing
  Source-only:  (types that only appear in one farm — reported separately)

Author: Peter
Dataset: CARE benchmark v6 (Gück et al., 2024)
"""

import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter, defaultdict
import xgboost as xgb

warnings.filterwarnings("ignore")

# ============================================================================
# ① PATHS & CONFIGURATION
# ============================================================================

BASE_DIR        = r"D:\files\project\CARE_To_Compare"
EVENT_INFO_PATH = os.path.join(BASE_DIR, "df_all__1_.csv")

FARMS = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "datasets"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "datasets"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "datasets"),
}

FEATURE_DESC = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "feature_description.csv"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "feature_description.csv"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "feature_description.csv"),
}

OUTPUT_DIR = os.path.join(BASE_DIR, "cross_farm_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_SEP           = ";"
FARM_NAME_MAP     = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS     = {0, 2}
CONSECUTIVE_WINDOW = 5   # consecutive correct predictions = detected

# ============================================================================
# ② FAULT LABELS (corrected from raw event descriptions)
# ============================================================================

FAULT_LABELS = {
    "A": {
        68: "transformer",
        22: "hydraulic",      72: "gearbox",
        73: "hydraulic",       0: "generator_bearing",
        26: "hydraulic",      40: "generator_bearing",
        42: "hydraulic",      10: "gearbox",
        45: "hydraulic",      84: "hydraulic",
        51: "gearbox",         # gearbox bearings damaged → gearbox component level
    },
    "B": {
        34: "transformer",     7: "transformer",
        53: "rotor_bearing",  27: "rotor_bearing",
        19: "transformer",    77: "rotor_bearing",
    },
    "C": {
        55: "pitch_electrical",   81: "converter",
        47: "hydraulic",          12: "gearbox_oil",
         4: "pitch_electrical",   18: "rotor_brake",
        28: "pitch_electrical",   39: "gearbox_oil",
        66: "pitch_electrical",   15: "pitch_electrical",
        78: "rotor_brake",        79: "communication",
        30: "pitch_electrical",   33: "pitch_mechanical",
        11: "pitch_electrical",   44: "cooling_water",
        49: "gearbox_oil",        31: "communication",
        67: "transformer",         9: "yaw",
        91: "pitch_electrical",    5: "electrical_internal",
        90: "communication",      70: "pitch_electrical",
        35: "converter",          16: "pitch_electrical",
        76: "pitch_electrical",
    },
}

# Fault types that are comparable across farms
# (same physical component, detectable via same subsystem group)
COMPARABLE_FAULT_TYPES = {
    "transformer",
    "gearbox",       # includes gearbox_bearing (merged at component level)
    "gearbox_oil",
    "generator_bearing",
    "hydraulic",
    "rotor_bearing",
}

# ============================================================================
# ③ SENSOR RENAMING
# ============================================================================

STAT_MAP = {"average":"avg","std_dev":"std","maximum":"max","minimum":"min"}


def _clean(text, is_unit=False):
    if pd.isna(text) or str(text).strip() in ("", "nan", "-", "none"):
        return ""
    t = str(text).strip()
    for old, new in [
        ("���C","degC"),("°C","degC"),("ºC","degC"),("½C","degC"),
        ("Celsius","degC"),("���","deg"),("°","deg"),("º","deg"),("½",""),
        ("/","_per_"),("%","pct"),(" ",""),("^",""),(".",""),
    ]:
        t = t.replace(old, new)
    t = re.sub(r"[^\x00-\x7F]+", "", t)
    if not is_unit:
        t = re.sub(r"[^\w\s]", " ", t)
        t = re.sub(r"\s+", "_", t.strip())
        t = re.sub(r"_+", "_", t).strip("_")
    else:
        t = re.sub(r"[^\w]", "", t)
    return t


def build_rename_dict(farm_id):
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(FEATURE_DESC[farm_id], sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    rename, counter_new, angle_new = {}, set(), set()
    seen = {}

    for _, row in df.iterrows():
        sid        = str(row["sensor_name"]).strip()
        desc       = _clean(str(row.get("description", "")))
        unit       = _clean(str(row.get("unit", "")), is_unit=True)
        is_counter = bool(row.get("is_counter", False))
        is_angle   = bool(row.get("is_angle", False))

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = [(f"{sid}_{STAT_MAP.get(s,s)}", STAT_MAP.get(s,s))
                   for s in stats] if stats else [(sid, "")]

        for old_col, sc in entries:
            parts   = [desc] + ([unit] if unit else []) + ([sc] if sc else [])
            new_col = "_".join(parts)
            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"
            seen[new_col] = old_col
            rename[old_col] = new_col
            if is_counter: counter_new.add(new_col)
            if is_angle:   angle_new.add(new_col)

    return rename, counter_new, angle_new


print("Building sensor rename dictionaries...")
rename_dicts = {}
counter_sets = {}
angle_sets   = {}
for farm in ["A", "B", "C"]:
    rd, co, an = build_rename_dict(farm)
    rename_dicts[farm] = rd
    counter_sets[farm] = co
    angle_sets[farm]   = an
    print(f"  Farm {farm}: {len(rd)} mappings")

META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}

# ============================================================================
# ④ SUBSYSTEM GROUP RULES
# (same as Script 1 — determines which columns belong to which group)
# ============================================================================

SUBSYSTEM_RULES = [
    (10,  "gearbox_bearing",  ["planetary bearing", "gearbox bearing",
                                "bearing on high speed shaft"]),
    (11,  "gearbox_oil",      ["gearbox oil", "gear oil", "oil inlet temperature",
                                "oil pressure", "oil level", "oil tank",
                                "oil temperature"]),
    (12,  "gearbox_other",    ["gearbox rotational", "gearbox speed",
                                "gearbox main shaft", "gearbox"]),
    (20,  "generator_bearing",["generator bearing"]),
    (21,  "generator_stator", ["stator winding", "stator"]),
    (22,  "generator_cooling",["generator cooling", "cooling air",
                                "cooling water temp. generator",
                                "generator temperature", "cooling air inlet"]),
    (23,  "generator_other",  ["generator acceleration", "generator angle",
                                "generator rpm", "generator converter",
                                "generator rms", "generator"]),
    (30,  "rotor_bearing",    ["rotor bearing", "axial bearing",
                                "rotor bearing inner ring"]),
    (40,  "hydraulic",        ["hydraulic pressure", "hydraulic oil",
                                "hydraulic pump", "hydraulic tank",
                                "hydraulic group", "aggregate system pressure",
                                "azimuth circuit", "azimuth memory",
                                "rotor brake", "locking memory",
                                "pollution indicator", "oil container"]),
    (50,  "pitch",            ["pitch angle", "pitch motor", "pitch axis",
                                "position motor axis", "position rotor blade",
                                "motor temperature kty", "motor temperature pt",
                                "battery charge", "battery discharge",
                                "rpm motor axis", "min pitch",
                                "motor current axis", "motor temperature axis"]),
    (60,  "vibration",        ["vibration", "drive train vibration",
                                "tower vibration"]),
    (70,  "transformer",      ["transformer", "transformator",
                                "hv transformer", "transformer cell"]),
    (80,  "electrical",       ["voltage", "current", "frequency",
                                "active power", "reactive power",
                                "apparent power", "grid power", "grid voltage",
                                "grid frequency", "available power",
                                "inverter torque", "cable load",
                                "dc link voltage", "internal consumption"]),
    (90,  "wind",             ["wind speed", "wind direction", "anemometer",
                                "estimated windspeed", "absolute wind",
                                "relative wind"]),
    (100, "environment",      ["ambient temperature", "outside temperature",
                                "nacelle temperature", "nacelle external",
                                "hub temperature", "board temperature",
                                "nacelle outside"]),
    (110, "cooling_water",    ["water temperature", "water pressure",
                                "water flow", "water conductivity",
                                "water cooler", "cooling water",
                                "flow meter stator"]),
    (120, "rotor",            ["rotor speed", "rotor rpm",
                                "nacelle direction", "yaw"]),
]


def assign_group(description):
    desc_lower = description.lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


def build_group_dict(farm_id):
    """Map new column names to subsystem groups for one farm."""
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(FEATURE_DESC[farm_id], sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    rdict  = rename_dicts[farm_id]
    gdict  = {}
    seen   = {}

    for _, row in df.iterrows():
        sid  = str(row["sensor_name"]).strip()
        desc = str(row.get("description", "")).strip()
        grp  = assign_group(desc)

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]

        entries = [f"{sid}_{STAT_MAP.get(s,s)}" for s in stats] \
                  if stats else [sid]

        for old_col in entries:
            new_col = rdict.get(old_col, old_col)
            gdict[new_col] = grp

    return gdict


print("Building subsystem group dictionaries...")
group_dicts = {}
for farm in ["A", "B", "C"]:
    group_dicts[farm] = build_group_dict(farm)
    print(f"  Farm {farm}: {len(group_dicts[farm])} column→group mappings")

# ============================================================================
# ⑤ DATA LOADING
# ============================================================================

def load_event_raw(filepath, farm_id):
    """Load event CSV with renaming. Returns (train_df, pred_df)."""
    rdict  = rename_dicts[farm_id]
    chunks = []

    for chunk in pd.read_csv(filepath, sep=CSV_SEP, chunksize=10000,
                             low_memory=False, encoding="latin-1"):
        chunk = chunk.rename(columns=rdict)
        num_cols = [c for c in chunk.columns
                    if c not in META_COLS
                    and pd.api.types.is_numeric_dtype(chunk[c])]
        # Zero-run handling (skip counters)
        counter_n = counter_sets[farm_id]
        for col in num_cols:
            if col in counter_n:
                continue
            vals = chunk[col].values.copy().astype(np.float32)
            n, i = len(vals), 0
            while i < n:
                if vals[i] == 0.0:
                    s = i
                    while i < n and vals[i] == 0.0:
                        i += 1
                    if i - s >= 6:
                        vals[s:i] = np.nan
                else:
                    i += 1
            chunk[col] = vals
        chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)
        del chunk

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    train_df = df[df["train_test"] == "train"].copy()
    pred_df  = df[df["train_test"] == "prediction"].copy()

    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    del df
    gc.collect()
    return train_df, pred_df

# ============================================================================
# ⑥ SUBSYSTEM AGGREGATE FEATURES
# The key innovation: farm-agnostic feature representation
# ============================================================================

# Subsystem groups included in cross-farm features
# Only groups that exist meaningfully in all three farms
CROSS_FARM_GROUPS = [
    "gearbox_bearing", "gearbox_oil", "gearbox_other",
    "generator_bearing", "generator_stator", "generator_other",
    "generator_cooling",
    "hydraulic",
    "rotor_bearing",
    "transformer",
    "electrical",
    "wind",
    "environment",
    "rotor",
    "pitch",
    "vibration",
]

# Statistics computed per subsystem group
GROUP_STATS = ["mean", "std", "min", "max", "trend"]


def compute_subsystem_features(df, farm_id, row_level=True):
    """
    Compute subsystem-level aggregate features.

    For each subsystem group, aggregate all _avg sensors in that group
    into 5 statistics: mean, std, min, max, trend (last-first difference).

    This produces a fixed-length feature vector regardless of how many
    sensors each farm has in each group.

    Parameters:
        df       : DataFrame (training or prediction window)
        farm_id  : farm identifier for group dict lookup
        row_level: if True, compute per-timestep (for prediction)
                   if False, compute over entire window (for training summary)

    Returns:
        If row_level=True:  DataFrame of shape (n_rows, n_features)
        If row_level=False: 1D array of shape (n_features,)
    """
    gdict    = group_dicts[farm_id]
    excluded = counter_sets[farm_id] | angle_sets[farm_id]

    # Build group → list of avg column names
    group_cols = defaultdict(list)
    for col in df.columns:
        if (col not in META_COLS
                and col not in excluded
                and "avg" in col.lower()
                and pd.api.types.is_numeric_dtype(df[col])):
            grp = gdict.get(col, "other")
            if grp in CROSS_FARM_GROUPS:
                group_cols[grp].append(col)

    if row_level:
        # Per-timestep features: rolling aggregates over 6-step window
        feat_dict = {}
        for grp in CROSS_FARM_GROUPS:
            cols = group_cols.get(grp, [])
            if cols:
                mat = df[cols].values.astype(np.float32)
                mat = np.nan_to_num(mat, nan=0.0)
                group_mean = np.nanmean(mat, axis=1)
                group_std  = np.nanstd(mat,  axis=1)
                group_min  = np.nanmin(mat,  axis=1)
                group_max  = np.nanmax(mat,  axis=1)
                # Trend: rolling 6-step difference on group mean
                s = pd.Series(group_mean)
                group_trend = s.diff(6).fillna(0).values
            else:
                # Group not present in this farm → zeros
                n = len(df)
                group_mean  = np.zeros(n, dtype=np.float32)
                group_std   = np.zeros(n, dtype=np.float32)
                group_min   = np.zeros(n, dtype=np.float32)
                group_max   = np.zeros(n, dtype=np.float32)
                group_trend = np.zeros(n, dtype=np.float32)

            feat_dict[f"{grp}_mean"]  = group_mean.astype(np.float32)
            feat_dict[f"{grp}_std"]   = group_std.astype(np.float32)
            feat_dict[f"{grp}_min"]   = group_min.astype(np.float32)
            feat_dict[f"{grp}_max"]   = group_max.astype(np.float32)
            feat_dict[f"{grp}_trend"] = group_trend.astype(np.float32)

            # Rolling stats (1hr and 3hr windows)
            for w, wn in [(6, "1hr"), (18, "3hr")]:
                s_mean = pd.Series(group_mean)
                feat_dict[f"{grp}_roll{wn}_mean"] = (
                    s_mean.rolling(w, min_periods=1).mean()
                    .values.astype(np.float32))
                feat_dict[f"{grp}_roll{wn}_std"] = (
                    s_mean.rolling(w, min_periods=1).std()
                    .fillna(0).values.astype(np.float32))

        return pd.DataFrame(feat_dict, index=df.index)

    else:
        # Window-level summary (single vector per event)
        feats = []
        feat_names = []
        for grp in CROSS_FARM_GROUPS:
            cols = group_cols.get(grp, [])
            if cols:
                mat = df[cols].values.astype(np.float32)
                mat = np.nan_to_num(mat, nan=0.0)
                grp_vals = np.nanmean(mat, axis=1)  # average across sensors
            else:
                grp_vals = np.zeros(len(df), dtype=np.float32)

            feats.extend([
                float(np.mean(grp_vals)),
                float(np.std(grp_vals)),
                float(np.min(grp_vals)),
                float(np.max(grp_vals)),
                float(grp_vals[-1] - grp_vals[0]) if len(grp_vals) > 1 else 0.0,
                # Rate of change in second half vs first half
                float(np.mean(grp_vals[len(grp_vals)//2:]) -
                      np.mean(grp_vals[:len(grp_vals)//2])),
            ])
            for stat in ["mean","std","min","max","trend","half_trend"]:
                feat_names.append(f"{grp}_{stat}")

        return np.array(feats, dtype=np.float32), feat_names


def get_feature_names():
    """Return the fixed feature name list for cross-farm features."""
    names = []
    for grp in CROSS_FARM_GROUPS:
        for stat in ["mean", "std", "min", "max", "trend",
                     "roll1hr_mean", "roll1hr_std",
                     "roll3hr_mean", "roll3hr_std"]:
            names.append(f"{grp}_{stat}")
    return names

# ============================================================================
# ⑥b FARM NORMAL BASELINE — for z-score normalisation
# ============================================================================

def compute_farm_baseline(farm_id, dataset):
    """
    Compute per-feature mean and std from ALL normal events in a farm.
    Z-score normalisation makes subsystem aggregate features farm-agnostic
    by expressing each feature as deviation from that farm's own normal.
    This is the key insight: a gearbox at 65C is normal in Farm A but
    abnormal in Farm C — only the deviation from normal is comparable.
    """
    normal_events = [d for d in dataset
                     if d["farm"] == farm_id and d["fault_type"] == "normal"]

    if not normal_events:
        n_feat = len(get_feature_names())
        return np.zeros(n_feat, dtype=np.float32), np.ones(n_feat, dtype=np.float32)

    print(f"  Computing normal baseline from {len(normal_events)} "
          f"normal events in Farm {farm_id}...")

    all_rows = []
    for d in normal_events:
        try:
            tr, _ = load_event_raw(d["filepath"], farm_id)
            if len(tr) == 0:
                continue
            feat_df = compute_subsystem_features(tr, farm_id, row_level=True)
            all_rows.append(feat_df.values.astype(np.float32))
            del tr, feat_df
            gc.collect()
        except Exception as e:
            continue

    if not all_rows:
        n_feat = len(get_feature_names())
        return np.zeros(n_feat, dtype=np.float32), np.ones(n_feat, dtype=np.float32)

    all_data = np.vstack(all_rows).astype(np.float32)
    del all_rows
    gc.collect()

    mu  = np.nanmean(all_data, axis=0).astype(np.float32)
    std = np.nanstd(all_data,  axis=0).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std)

    print(f"    Baseline: {all_data.shape[0]:,} rows, {all_data.shape[1]} features")
    del all_data
    gc.collect()
    return mu, std


def zscore_normalise(X, mu, std):
    """Z-score normalise using precomputed farm baseline."""
    return ((X - mu) / std).astype(np.float32)


# ============================================================================
# ⑦ EVENT METADATA
# ============================================================================

print("\nLoading event metadata...")

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"]     = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"] = event_info["event_id"].astype(str)
event_info["event_label"] = event_info["event_label"].str.strip()
event_info["event_description"] = (
    event_info["event_description"].fillna("normal").astype(str))
event_info["event_start"] = pd.to_datetime(
    event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(
    event_info["event_end"],   errors="coerce")

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue
    eid      = row["event_id"]
    eid_int  = int(eid) if eid.isdigit() else -1
    label    = row["event_label"]
    filepath = os.path.join(FARMS[farm], f"{eid}.csv")
    if not os.path.exists(filepath):
        continue

    fault_type = FAULT_LABELS.get(farm, {}).get(eid_int, "normal")
    if label == "normal":
        fault_type = "normal"

    dataset.append({
        "event_id":    eid,
        "farm":        farm,
        "label":       label,
        "fault_type":  fault_type,
        "filepath":    filepath,
        "event_start": row["event_start"],
        "event_end":   row["event_end"],
    })

for farm in ["A", "B", "C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    anom = [d for d in fe if d["label"] == "anomaly"]
    print(f"  Farm {farm}: {len(fe)} events | "
          f"fault types: {dict(Counter(d['fault_type'] for d in anom))}")

# ============================================================================
# ⑧ BUILD TRAINING MATRIX FROM SOURCE FARM
# ============================================================================

def build_source_matrix(source_farm, dataset):
    """
    Build training matrix from ALL events in source farm.
    Uses row-level subsystem features from training windows.
    Returns (X, y, label_map, reverse_map, feat_names).
    """
    farm_events = [d for d in dataset if d["farm"] == source_farm]
    all_fault_types = sorted(set(d["fault_type"] for d in farm_events))
    label_map   = {ft: i for i, ft in enumerate(all_fault_types)}
    reverse_map = {i: ft for ft, i in label_map.items()}
    feat_names  = get_feature_names()

    X_parts, y_parts = [], []

    print(f"\n  Building training matrix from Farm {source_farm}...")
    for d in farm_events:
        try:
            tr, pred = load_event_raw(d["filepath"], source_farm)

            # KEY FIX: use prediction window for fault events (pre-failure signal),
            # training window for normal events (normal operation baseline).
            # Previously used training window for all — but training window is
            # status-filtered to normal, so fault events looked identical to normal.
            if d["fault_type"] != "normal" and len(pred) > 0:
                source_df = pred   # pre-failure degradation signal
            elif len(tr) > 0:
                source_df = tr     # normal operation baseline
            else:
                continue

            feat_df = compute_subsystem_features(source_df, source_farm, row_level=True)

            X = np.nan_to_num(
                feat_df.values.astype(np.float32),
                nan=0.0, posinf=0.0, neginf=0.0
            )
            y = np.full(len(X), label_map[d["fault_type"]], dtype=np.int32)
            X_parts.append(X)
            y_parts.append(y)
            del tr, pred, source_df, feat_df
            gc.collect()
        except Exception as e:
            print(f"    ⚠️  Skip {d['event_id']}: {e}")

    if not X_parts:
        return None, None, None, None, None

    X = np.vstack(X_parts).astype(np.float32)
    y = np.concatenate(y_parts).astype(np.int32)
    del X_parts, y_parts
    gc.collect()

    print(f"  Training matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
    print(f"  Classes: {all_fault_types}")

    # Compute source farm normal baseline and z-score normalise
    # This makes absolute sensor values farm-agnostic
    src_mu, src_std = compute_farm_baseline(source_farm, dataset)
    X = zscore_normalise(X, src_mu, src_std)
    print(f"  Z-score normalised using Farm {source_farm} normal baseline")

    return X, y, label_map, reverse_map, feat_names, src_mu, src_std

# ============================================================================
# ⑨ TRAIN CROSS-FARM CLASSIFIER
# ============================================================================

def train_cross_farm_classifier(X_train, y_train):
    """Train XGBoost on source farm data."""
    unique_cls = sorted(np.unique(y_train).tolist())
    n_cls      = len(unique_cls)
    remap      = {old: new for new, old in enumerate(unique_cls)}
    reverse_r  = {new: old for old, new in remap.items()}
    y_remapped = np.array([remap[y] for y in y_train])



    # Capped class weights — prevents dominant class from suppressing minority
    ulabels, counts = np.unique(y_remapped, return_counts=True)
    raw_w = {c: len(y_remapped) / (n_cls * cnt)
             for c, cnt in zip(ulabels, counts)}
    min_w = min(raw_w.values())
    weights = {c: min(w / min_w, 8.0) for c, w in raw_w.items()}
    sw = np.array([weights[y] for y in y_remapped])

    clf = xgb.XGBClassifier(
        objective="multi:softmax" if n_cls > 2 else "binary:logistic",
        num_class=n_cls if n_cls > 2 else None,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,   # slightly more regularisation for cross-farm
        eval_metric="mlogloss" if n_cls > 2 else "logloss",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    clf.fit(X_train, y_remapped, sample_weight=sw)
    return clf, remap, reverse_r, unique_cls

# ============================================================================
# ⑩ EVALUATE ON TARGET FARM
# ============================================================================

def detect_failure(predictions, true_label, timestamps):
    """5 consecutive correct predictions = detected."""
    n = len(predictions)
    w = CONSECUTIVE_WINDOW
    for i in range(n - w + 1):
        if np.all(predictions[i:i + w] == true_label):
            if timestamps is not None and len(timestamps) > 0:
                det_ts = pd.Timestamp(timestamps[i])
                end_ts = pd.Timestamp(timestamps[-1])
                return True, (end_ts - det_ts).total_seconds() / 86400
            return True, None
    return False, None


def evaluate_on_target(target_farm, clf, remap, reverse_r,
                       source_label_map, source_reverse_map,
                       dataset, feat_names, tgt_mu=None, tgt_std=None):
    """
    Apply cross-farm classifier to every event in target farm.
    Returns list of result dicts.
    """
    farm_events = [d for d in dataset if d["farm"] == target_farm]
    results = []

    print(f"\n  Evaluating on Farm {target_farm} ({len(farm_events)} events)...")

    for d in farm_events:
        is_fault  = d["label"] == "anomaly"
        true_type = d["fault_type"]

        try:
            _, pr = load_event_raw(d["filepath"], target_farm)
            if len(pr) == 0:
                continue

            feat_df = compute_subsystem_features(
                pr, target_farm, row_level=True
            )
            X_test = np.nan_to_num(
                feat_df.values.astype(np.float32),
                nan=0.0, posinf=0.0, neginf=0.0
            )

            # Z-score normalise using target farm baseline
            if tgt_mu is not None and tgt_std is not None:
                X_test = zscore_normalise(X_test, tgt_mu, tgt_std)

            # Map true label to source farm label space
            true_label_int = source_label_map.get(true_type, -1)

            timestamps = (pr["time_stamp"].values
                          if "time_stamp" in pr.columns else None)

            # Predict in source remapped space
            preds_remapped  = clf.predict(X_test)
            preds_source    = np.array([reverse_r[p] for p in preds_remapped])
            preds_type      = np.array([source_reverse_map[p]
                                        for p in preds_source])

            # Detection: 5 consecutive predictions of correct type
            detected, horizon = False, None
            if is_fault and true_label_int >= 0:
                detected, horizon = detect_failure(
                    preds_source, true_label_int, timestamps
                )

            # Most common predicted type
            pred_type = Counter(preds_type.tolist()).most_common(1)[0][0]

            # Is fault type comparable (exists in source farm)?
            is_comparable = true_type in COMPARABLE_FAULT_TYPES

            status = ("✓" if detected else
                      ("✗" if is_fault else "—"))
            comparable_str = "[comparable]" if is_comparable else "[source-only]"
            if is_fault:
                print(f"    E{d['event_id']:>4} [{true_type:22}] "
                      f"→ [{pred_type:22}] {status} {comparable_str}")

            del pr, feat_df, X_test, preds_remapped, preds_source, preds_type
            gc.collect()

            results.append({
                "event_id":      d["event_id"],
                "target_farm":   target_farm,
                "fault_type":    true_type,
                "pred_type":     pred_type,
                "is_fault":      is_fault,
                "is_comparable": is_comparable,
                "detected":      bool(detected),
                "horizon_days":  round(horizon, 2) if horizon else None,
            })

        except Exception as e:
            print(f"    ⚠️  E{d['event_id']}: {e}")

    return results

# ============================================================================
# ⑪ METRICS
# ============================================================================

def compute_cross_farm_metrics(results, source_farm, target_farm):
    """
    Compute cross-farm detection metrics.
    Reports separately for comparable and source-only fault types.
    """
    fault_res      = [r for r in results if r["is_fault"]]
    comparable_res = [r for r in fault_res if r["is_comparable"]]
    sourceonly_res = [r for r in fault_res if not r["is_comparable"]]

    def _metrics(res_list, label):
        if not res_list:
            return None
        total    = len(res_list)
        detected = sum(1 for r in res_list if r["detected"])
        rate     = detected / total
        horizons = [r["horizon_days"] for r in res_list
                    if r["horizon_days"] is not None]
        avg_h    = float(np.mean(horizons)) if horizons else 0.0

        # Per-type breakdown
        by_type = defaultdict(lambda: {"total":0,"detected":0,"horizons":[]})
        for r in res_list:
            ft = r["fault_type"]
            by_type[ft]["total"] += 1
            if r["detected"]:
                by_type[ft]["detected"] += 1
            if r["horizon_days"] is not None:
                by_type[ft]["horizons"].append(r["horizon_days"])

        print(f"\n  {label}:")
        print(f"  Overall: {detected}/{total} = {rate:.1%} | "
              f"avg horizon: {avg_h:.1f} days")
        print(f"  {'Fault Type':25} {'Det':>4}{'Tot':>4}  {'Rate':>6}  "
              f"{'Avg Horizon':>12}")
        print(f"  {'─'*58}")
        for ft in sorted(by_type.keys()):
            v  = by_type[ft]
            r_ = v["detected"] / v["total"] if v["total"] > 0 else 0
            h_ = np.mean(v["horizons"]) if v["horizons"] else 0
            print(f"  {ft:25} {v['detected']:4d}{v['total']:4d}  {r_:6.1%}  "
                  f"{h_:10.1f}d")

        return {
            "total": total, "detected": detected,
            "detection_rate": round(rate, 4),
            "avg_horizon_days": round(avg_h, 2),
            "by_type": {ft: dict(v) for ft, v in by_type.items()},
        }

    print(f"\n  {'='*60}")
    print(f"  Farm {source_farm} → Farm {target_farm} Cross-Farm Results")
    print(f"  {'='*60}")

    comp_m = _metrics(comparable_res,
                      "Comparable fault types (exist in both farms)")
    solo_m = _metrics(sourceonly_res,
                      "Source-only fault types (not in source training)")

    return {
        "source_farm":   source_farm,
        "target_farm":   target_farm,
        "comparable":    comp_m,
        "source_only":   solo_m,
        "all_faults":    _metrics(fault_res, "All fault types"),
    }

# ============================================================================
# ⑫ FEATURE IMPORTANCE
# ============================================================================

def get_feature_importance(clf, reverse_r, source_reverse_map, feat_names):
    """Extract and return top feature importances."""
    if not hasattr(clf, "feature_importances_"):
        return {}
    importance = {}
    for name, imp in zip(feat_names, clf.feature_importances_):
        importance[name] = float(imp)
    return dict(sorted(importance.items(),
                        key=lambda x: x[1], reverse=True)[:20])

# ============================================================================
# ⑬ PLOTS
# ============================================================================

def plot_cross_farm_results(all_cross_results, output_dir):
    """Summary plots for cross-farm validation."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("  matplotlib not available — skipping plots")
        return

    # ---- Detection rate comparison: within-farm vs cross-farm ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    directions = []
    comp_rates = []
    all_rates  = []

    for key, metrics in all_cross_results.items():
        src, tgt = metrics["source_farm"], metrics["target_farm"]
        directions.append(f"Farm {src}→{tgt}")
        comp = metrics.get("comparable", {}) or {}
        all_ = metrics.get("all_faults",  {}) or {}
        comp_rates.append(comp.get("detection_rate", 0))
        all_rates.append(all_.get("detection_rate", 0))

    colors = ["#4878d0", "#e05252", "#2ca02c", "#ff7f0e",
              "#9467bd", "#8c564b"]

    ax = axes[0]
    x  = np.arange(len(directions))
    w  = 0.35
    b1 = ax.bar(x - w/2, comp_rates, w, label="Comparable types",
                color="#4878d0", alpha=0.85)
    b2 = ax.bar(x + w/2, all_rates,  w, label="All fault types",
                color="#e05252", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(directions, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Detection Rate", fontsize=10)
    ax.set_title("Cross-Farm Detection Rate\n"
                 "(Train on source → Test on target)",
                 fontsize=11, fontweight="bold")
    ax.bar_label(b1, fmt="%.0%", padding=2, fontsize=8)
    ax.bar_label(b2, fmt="%.0%", padding=2, fontsize=8)
    ax.axhline(0.5, color="#555", ls="--", lw=1, alpha=0.6,
               label="50% reference")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # ---- Horizon comparison ----
    ax = axes[1]
    horizons = []
    for key, metrics in all_cross_results.items():
        src, tgt = metrics["source_farm"], metrics["target_farm"]
        comp = metrics.get("comparable", {}) or {}
        horizons.append(comp.get("avg_horizon_days", 0))

    bars = ax.bar(directions, horizons, color="#4878d0", alpha=0.85)
    ax.set_ylabel("Average Detection Horizon (days)", fontsize=10)
    ax.set_title("Cross-Farm Average Early Warning Time\n"
                 "(Comparable fault types only)",
                 fontsize=11, fontweight="bold")
    ax.bar_label(bars, fmt="%.1f d", padding=3, fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right", fontsize=9)

    plt.tight_layout()
    p = os.path.join(output_dir, "cross_farm_detection_rates.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")

    # ---- Per fault type cross-farm heatmap (A→C and C→A) ----
    key_ac = "A_to_C"
    key_ca = "C_to_A"

    for key, label in [(key_ac, "Farm A → Farm C"),
                       (key_ca, "Farm C → Farm A")]:
        if key not in all_cross_results:
            continue
        metrics = all_cross_results[key]
        by_type = (metrics.get("all_faults") or {}).get("by_type", {})
        if not by_type:
            continue

        types = sorted(by_type.keys())
        rates = [by_type[t]["detected"] / by_type[t]["total"]
                 if by_type[t]["total"] > 0 else 0
                 for t in types]
        colors_ = ["#2ca02c" if r >= 0.5 else "#d62728" for r in rates]

        fig, ax = plt.subplots(figsize=(max(10, len(types) * 1.2), 5))
        bars = ax.bar(types, rates, color=colors_, alpha=0.85)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("Detection Rate", fontsize=10)
        ax.set_title(f"Cross-Farm: {label} — Detection Rate by Fault Type",
                     fontsize=11, fontweight="bold")
        ax.bar_label(bars, fmt="%.0%", padding=3, fontsize=9)
        ax.axhline(0.5, color="#555", ls="--", lw=1, alpha=0.6)
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right", fontsize=9)
        plt.tight_layout()
        fname = f"cross_farm_{key.lower()}_by_type.png"
        p = os.path.join(output_dir, fname)
        plt.savefig(p, dpi=130, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {p}")


# ============================================================================
# ⑭ MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("CROSS-FARM FAULT CLASSIFICATION VALIDATION")
    print("=" * 80)
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}\n")
    start = datetime.now()

    print("Strategy: Subsystem-aggregate features (farm-agnostic representation)")
    print(f"Groups: {CROSS_FARM_GROUPS}")
    print(f"Features per group: mean, std, min, max, trend, "
          f"roll1hr_mean, roll1hr_std, roll3hr_mean, roll3hr_std")
    print(f"Total features: {len(CROSS_FARM_GROUPS) * 9}")

    all_cross_results = {}
    all_importances   = {}

    # ========================================================================
    # Direction 1: Farm A → Farm C
    # ========================================================================
    print(f"\n{'='*80}")
    print("DIRECTION 1: Train Farm A → Predict Farm C")
    print("=" * 80)

    # Pre-compute normal baselines for all three farms
    print("\nPre-computing normal baselines for z-score normalisation...")
    mu_a, std_a = compute_farm_baseline("A", dataset)
    mu_b, std_b = compute_farm_baseline("B", dataset)
    mu_c, std_c = compute_farm_baseline("C", dataset)
    print("Baselines computed.\n")

    X_a, y_a, lmap_a, rmap_a, feat_names, _, _ = build_source_matrix("A", dataset)
    if X_a is not None:
        clf_a, remap_a, reverse_r_a, unique_a = train_cross_farm_classifier(
            X_a, y_a)
        del X_a, y_a
        gc.collect()

        results_ac = evaluate_on_target(
            "C", clf_a, remap_a, reverse_r_a, lmap_a, rmap_a,
            dataset, feat_names, tgt_mu=mu_c, tgt_std=std_c
        )
        metrics_ac = compute_cross_farm_metrics(results_ac, "A", "C")
        all_cross_results["A_to_C"] = metrics_ac
        all_importances["A_to_C"] = get_feature_importance(
            clf_a, remap_a, rmap_a, feat_names
        )
        del clf_a
        gc.collect()

    # ========================================================================
    # Direction 2: Farm C → Farm A
    # ========================================================================
    print(f"\n{'='*80}")
    print("DIRECTION 2: Train Farm C → Predict Farm A")
    print("=" * 80)

    X_c, y_c, lmap_c, rmap_c, feat_names, _, _ = build_source_matrix("C", dataset)
    if X_c is not None:
        clf_c, remap_c, reverse_r_c, unique_c = train_cross_farm_classifier(
            X_c, y_c)
        del X_c, y_c
        gc.collect()

        results_ca = evaluate_on_target(
            "A", clf_c, remap_c, reverse_r_c, lmap_c, rmap_c,
            dataset, feat_names, tgt_mu=mu_a, tgt_std=std_a
        )
        metrics_ca = compute_cross_farm_metrics(results_ca, "C", "A")
        all_cross_results["C_to_A"] = metrics_ca
        all_importances["C_to_A"] = get_feature_importance(
            clf_c, remap_c, rmap_c, feat_names
        )

        # Also evaluate Farm C on Farm B (supplementary)
        print(f"\n  Supplementary: Farm C → Farm B")
        results_cb = evaluate_on_target(
            "B", clf_c, remap_c, reverse_r_c, lmap_c, rmap_c,
            dataset, feat_names, tgt_mu=mu_b, tgt_std=std_b
        )
        metrics_cb = compute_cross_farm_metrics(results_cb, "C", "B")
        all_cross_results["C_to_B"] = metrics_cb

        del clf_c
        gc.collect()

    # ========================================================================
    # Direction 3: Farm B → Farm A and Farm C
    # ========================================================================
    print(f"\n{'='*80}")
    print("DIRECTION 3: Train Farm B → Predict Farm A and Farm C")
    print("=" * 80)

    X_b, y_b, lmap_b, rmap_b, feat_names, _, _ = build_source_matrix("B", dataset)
    if X_b is not None:
        clf_b, remap_b, reverse_r_b, unique_b = train_cross_farm_classifier(
            X_b, y_b)
        del X_b, y_b
        gc.collect()

        print(f"\n  Farm B → Farm A:")
        results_ba = evaluate_on_target(
            "A", clf_b, remap_b, reverse_r_b, lmap_b, rmap_b,
            dataset, feat_names, tgt_mu=mu_a, tgt_std=std_a
        )
        metrics_ba = compute_cross_farm_metrics(results_ba, "B", "A")
        all_cross_results["B_to_A"] = metrics_ba

        print(f"\n  Farm B → Farm C:")
        results_bc = evaluate_on_target(
            "C", clf_b, remap_b, reverse_r_b, lmap_b, rmap_b,
            dataset, feat_names, tgt_mu=mu_c, tgt_std=std_c
        )
        metrics_bc = compute_cross_farm_metrics(results_bc, "B", "C")
        all_cross_results["B_to_C"] = metrics_bc

        del clf_b
        gc.collect()

    # ========================================================================
    # Summary
    # ========================================================================
    print(f"\n{'='*80}")
    print("CROSS-FARM VALIDATION SUMMARY")
    print("=" * 80)
    print(f"\n  {'Direction':20} {'Comparable':>12} {'All types':>10} "
          f"{'Avg Horizon':>14}")
    print(f"  {'─'*60}")

    for key, metrics in all_cross_results.items():
        src = metrics["source_farm"]
        tgt = metrics["target_farm"]
        comp = metrics.get("comparable") or {}
        all_ = metrics.get("all_faults")  or {}
        cr   = comp.get("detection_rate",   0)
        ar   = all_.get("detection_rate",   0)
        ah   = comp.get("avg_horizon_days", 0)
        print(f"  Farm {src} → Farm {tgt}:     "
              f"{cr:>10.1%}   {ar:>8.1%}   {ah:>12.1f}d")

    # ========================================================================
    # Plots
    # ========================================================================
    print("\nGenerating plots...")
    plot_cross_farm_results(all_cross_results, OUTPUT_DIR)

    # ========================================================================
    # Save
    # ========================================================================
    def json_safe(obj):
        if isinstance(obj, dict):   return {k: json_safe(v) for k,v in obj.items()}
        if isinstance(obj, list):   return [json_safe(i) for i in obj]
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.bool_):       return bool(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return obj

    out = json_safe({
        "timestamp":    datetime.now().isoformat(),
        "script":       "script_3_cross_farm_validation",
        "feature_groups": CROSS_FARM_GROUPS,
        "n_features":   len(CROSS_FARM_GROUPS) * 9,
        "consecutive_window": CONSECUTIVE_WINDOW,
        "results":      all_cross_results,
        "feature_importance": all_importances,
    })

    rj = os.path.join(OUTPUT_DIR, "cross_farm_results.json")
    with open(rj, "w") as f:
        json.dump(out, f, indent=2)
    print(f"\nSaved: {rj}")

    print(f"\nRuntime: {datetime.now() - start}")
    print("✅ DONE")


if __name__ == "__main__":
    main()

Building sensor rename dictionaries...
  Farm A: 81 mappings
  Farm B: 252 mappings
  Farm C: 952 mappings
Building subsystem group dictionaries...
  Farm A: 81 column→group mappings
  Farm B: 252 column→group mappings
  Farm C: 952 column→group mappings

Loading event metadata...
  Farm A: 22 events | fault types: {'transformer': 1, 'hydraulic': 6, 'gearbox': 3, 'generator_bearing': 2}
  Farm B: 15 events | fault types: {'transformer': 3, 'rotor_bearing': 3}
  Farm C: 58 events | fault types: {'pitch_electrical': 11, 'converter': 2, 'hydraulic': 1, 'gearbox_oil': 3, 'rotor_brake': 2, 'communication': 3, 'pitch_mechanical': 1, 'cooling_water': 1, 'transformer': 1, 'yaw': 1, 'electrical_internal': 1}
CROSS-FARM FAULT CLASSIFICATION VALIDATION
Started: 12:19:30

Strategy: Subsystem-aggregate features (farm-agnostic representation)
Groups: ['gearbox_bearing', 'gearbox_oil', 'gearbox_other', 'generator_bearing', 'generator_stator', 'generator_other', 'generator_cooling', 'hydraulic', 'roto

In [3]:
"""
================================================================================
SCRIPT 3 Option 2(Not used)— CROSS-FARM FAULT CLASSIFICATION VALIDATION
================================================================================
Tests whether fault signatures learned on one wind farm generalise to another.

Experimental design:
  Direction 1: Train on Farm A → Predict Farm C
  Direction 2: Train on Farm C → Predict Farm A
  Direction 3: Train on Farm B → Predict Farm A and Farm C (supplementary)

Feature representation:
  Raw sensor data cannot be used directly — Farm A has 86 features,
  Farm C has 957. Instead, we use SUBSYSTEM-LEVEL AGGREGATE features:
  for each physical subsystem group (gearbox_bearing, hydraulic, transformer,
  etc.) we compute mean, std, min, max, and trend of all sensors in that group.
  This produces a fixed-length representation (~50 features) that is
  physically meaningful and farm-agnostic.

Author: Peter
Dataset: CARE benchmark v6 (Gück et al., 2024)
"""

import os
import gc
import re
import json
import warnings
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import xgboost as xgb

warnings.filterwarnings("ignore")

# ============================================================================
# ① PATHS & CONFIGURATION
# ============================================================================

BASE_DIR = r"D:\files\project\CARE_To_Compare"
EVENT_INFO_PATH = os.path.join(BASE_DIR, "df_all__1_.csv")

FARMS = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "datasets"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "datasets"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "datasets"),
}

FEATURE_DESC = {
    "A": os.path.join(BASE_DIR, "Wind Farm A", "feature_description.csv"),
    "B": os.path.join(BASE_DIR, "Wind Farm B", "feature_description.csv"),
    "C": os.path.join(BASE_DIR, "Wind Farm C", "feature_description.csv"),
}

OUTPUT_DIR = os.path.join(BASE_DIR, "cross_farm_outputs(OptionB)")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSV_SEP = ";"
FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
NORMAL_STATUS = {0, 2}
CONSECUTIVE_WINDOW = 5

# ============================================================================
# ② FAULT LABELS
# ============================================================================

FAULT_LABELS = {
    "A": {
        68: "transformer",
        22: "hydraulic", 72: "gearbox",
        73: "hydraulic", 0: "generator_bearing",
        26: "hydraulic", 40: "generator_bearing",
        42: "hydraulic", 10: "gearbox",
        45: "hydraulic", 84: "hydraulic",
        51: "gearbox",
    },
    "B": {
        34: "transformer", 7: "transformer",
        53: "rotor_bearing", 27: "rotor_bearing",
        19: "transformer", 77: "rotor_bearing",
    },
    "C": {
        55: "pitch_electrical", 81: "converter",
        47: "hydraulic", 12: "gearbox_oil",
        4: "pitch_electrical", 18: "rotor_brake",
        28: "pitch_electrical", 39: "gearbox_oil",
        66: "pitch_electrical", 15: "pitch_electrical",
        78: "rotor_brake", 79: "communication",
        30: "pitch_electrical", 33: "pitch_mechanical",
        11: "pitch_electrical", 44: "cooling_water",
        49: "gearbox_oil", 31: "communication",
        67: "transformer", 9: "yaw",
        91: "pitch_electrical", 5: "electrical_internal",
        90: "communication", 70: "pitch_electrical",
        35: "converter", 16: "pitch_electrical",
        76: "pitch_electrical",
    },
}

COMPARABLE_FAULT_TYPES = {
    "transformer",
    "gearbox",
    "gearbox_oil",
    "generator_bearing",
    "hydraulic",
    "rotor_bearing",
}

# ============================================================================
# ③ SENSOR RENAMING
# ============================================================================

STAT_MAP = {"average": "avg", "std_dev": "std", "maximum": "max", "minimum": "min"}
META_COLS = {"time_stamp", "asset_id", "id", "train_test", "status_type_id"}


def parse_bool(value):
    """Robust bool parsing for CSV fields."""
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    s = str(value).strip().lower()
    return s in {"1", "true", "yes", "y", "t"}


def _clean(text, is_unit=False):
    if pd.isna(text) or str(text).strip().lower() in {"", "nan", "-", "none"}:
        return ""
    t = str(text).strip()
    for old, new in [
        ("���C", "degC"), ("°C", "degC"), ("ºC", "degC"), ("½C", "degC"),
        ("Celsius", "degC"), ("���", "deg"), ("°", "deg"), ("º", "deg"),
        ("½", ""), ("/", "_per_"), ("%", "pct"), (" ", ""), ("^", ""), (".", ""),
    ]:
        t = t.replace(old, new)
    t = re.sub(r"[^\x00-\x7F]+", "", t)
    if not is_unit:
        t = re.sub(r"[^\w\s]", " ", t)
        t = re.sub(r"\s+", "_", t.strip())
        t = re.sub(r"_+", "_", t).strip("_")
    else:
        t = re.sub(r"[^\w]", "", t)
    return t


def read_csv_try_encodings(path, sep=";"):
    last_err = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err if last_err else ValueError(f"Could not read {path}")


def build_rename_dict(farm_id):
    df = read_csv_try_encodings(FEATURE_DESC[farm_id], sep=";")

    rename, counter_new, angle_new = {}, set(), set()
    seen = {}

    for _, row in df.iterrows():
        sid = str(row["sensor_name"]).strip()
        desc = _clean(row.get("description", ""))
        unit = _clean(row.get("unit", ""), is_unit=True)
        is_counter = parse_bool(row.get("is_counter", False))
        is_angle = parse_bool(row.get("is_angle", False))

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else [
            s.strip() for s in stats_raw.split(",") if s.strip()
        ]

        entries = (
            [(f"{sid}_{STAT_MAP.get(s, s)}", STAT_MAP.get(s, s)) for s in stats]
            if stats else [(sid, "")]
        )

        for old_col, sc in entries:
            parts = [desc] + ([unit] if unit else []) + ([sc] if sc else [])
            new_col = "_".join([p for p in parts if p])

            if not new_col:
                new_col = old_col

            if new_col in seen and seen[new_col] != old_col:
                new_col = f"{new_col}_{sid}"

            seen[new_col] = old_col
            rename[old_col] = new_col

            if is_counter:
                counter_new.add(new_col)
            if is_angle:
                angle_new.add(new_col)

    return rename, counter_new, angle_new


print("Building sensor rename dictionaries...")
rename_dicts = {}
counter_sets = {}
angle_sets = {}

for farm in ["A", "B", "C"]:
    rd, co, an = build_rename_dict(farm)
    rename_dicts[farm] = rd
    counter_sets[farm] = co
    angle_sets[farm] = an
    print(f"  Farm {farm}: {len(rd)} mappings")

# ============================================================================
# ④ SUBSYSTEM GROUP RULES
# ============================================================================

SUBSYSTEM_RULES = [
    (10, "gearbox_bearing", ["planetary bearing", "gearbox bearing", "bearing on high speed shaft"]),
    (11, "gearbox_oil", ["gearbox oil", "gear oil", "oil inlet temperature", "oil pressure", "oil level", "oil tank", "oil temperature"]),
    (12, "gearbox_other", ["gearbox rotational", "gearbox speed", "gearbox main shaft", "gearbox"]),
    (20, "generator_bearing", ["generator bearing"]),
    (21, "generator_stator", ["stator winding", "stator"]),
    (22, "generator_cooling", ["generator cooling", "cooling air", "cooling water temp. generator", "generator temperature", "cooling air inlet"]),
    (23, "generator_other", ["generator acceleration", "generator angle", "generator rpm", "generator converter", "generator rms", "generator"]),
    (30, "rotor_bearing", ["rotor bearing", "axial bearing", "rotor bearing inner ring"]),
    (40, "hydraulic", ["hydraulic pressure", "hydraulic oil", "hydraulic pump", "hydraulic tank", "hydraulic group", "aggregate system pressure", "azimuth circuit", "azimuth memory", "rotor brake", "locking memory", "pollution indicator", "oil container"]),
    (50, "pitch", ["pitch angle", "pitch motor", "pitch axis", "position motor axis", "position rotor blade", "motor temperature kty", "motor temperature pt", "battery charge", "battery discharge", "rpm motor axis", "min pitch", "motor current axis", "motor temperature axis"]),
    (60, "vibration", ["vibration", "drive train vibration", "tower vibration"]),
    (70, "transformer", ["transformer", "transformator", "hv transformer", "transformer cell"]),
    (80, "electrical", ["voltage", "current", "frequency", "active power", "reactive power", "apparent power", "grid power", "grid voltage", "grid frequency", "available power", "inverter torque", "cable load", "dc link voltage", "internal consumption"]),
    (90, "wind", ["wind speed", "wind direction", "anemometer", "estimated windspeed", "absolute wind", "relative wind"]),
    (100, "environment", ["ambient temperature", "outside temperature", "nacelle temperature", "nacelle external", "hub temperature", "board temperature", "nacelle outside"]),
    (110, "cooling_water", ["water temperature", "water pressure", "water flow", "water conductivity", "water cooler", "cooling water", "flow meter stator"]),
    (120, "rotor", ["rotor speed", "rotor rpm", "nacelle direction", "yaw"]),
]


def assign_group(description):
    desc_lower = str(description).lower()
    for _, group, keywords in sorted(SUBSYSTEM_RULES, key=lambda x: x[0]):
        if any(kw in desc_lower for kw in keywords):
            return group
    return "other"


def build_group_dict(farm_id):
    df = read_csv_try_encodings(FEATURE_DESC[farm_id], sep=";")
    rdict = rename_dicts[farm_id]
    gdict = {}

    for _, row in df.iterrows():
        sid = str(row["sensor_name"]).strip()
        desc = str(row.get("description", "")).strip()
        grp = assign_group(desc)

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan", "") else [
            s.strip() for s in stats_raw.split(",") if s.strip()
        ]

        entries = [f"{sid}_{STAT_MAP.get(s, s)}" for s in stats] if stats else [sid]

        for old_col in entries:
            new_col = rdict.get(old_col, old_col)
            gdict[new_col] = grp

    return gdict


print("Building subsystem group dictionaries...")
group_dicts = {}
for farm in ["A", "B", "C"]:
    group_dicts[farm] = build_group_dict(farm)
    print(f"  Farm {farm}: {len(group_dicts[farm])} column→group mappings")

# ============================================================================
# ⑤ DATA LOADING
# ============================================================================

def load_event_raw(filepath, farm_id):
    """Load event CSV with renaming. Returns (train_df, pred_df)."""
    rdict = rename_dicts[farm_id]
    chunks = []

    for chunk in pd.read_csv(
        filepath,
        sep=CSV_SEP,
        chunksize=10000,
        low_memory=False,
        encoding="latin-1"
    ):
        chunk = chunk.rename(columns=rdict)

        num_cols = [
            c for c in chunk.columns
            if c not in META_COLS and pd.api.types.is_numeric_dtype(chunk[c])
        ]

        counter_n = counter_sets[farm_id]

        for col in num_cols:
            if col in counter_n:
                continue

            vals = chunk[col].to_numpy(copy=True, dtype=np.float32)
            n = len(vals)
            i = 0

            while i < n:
                if vals[i] == 0.0:
                    s = i
                    while i < n and vals[i] == 0.0:
                        i += 1
                    if i - s >= 6:
                        vals[s:i] = np.nan
                else:
                    i += 1

            chunk[col] = vals

        chunk[num_cols] = chunk[num_cols].astype(np.float32)
        chunks.append(chunk)
        del chunk

    if not chunks:
        return pd.DataFrame(), pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()

    if "train_test" not in df.columns:
        raise KeyError(f"'train_test' column missing in {filepath}")

    train_df = df[df["train_test"] == "train"].copy()
    pred_df = df[df["train_test"] == "prediction"].copy()

    if "status_type_id" in train_df.columns:
        train_df = train_df[train_df["status_type_id"].isin(NORMAL_STATUS)]

    del df
    gc.collect()
    return train_df, pred_df

# ============================================================================
# ⑥ SUBSYSTEM AGGREGATE FEATURES
# ============================================================================

CROSS_FARM_GROUPS = [
    "gearbox_bearing", "gearbox_oil", "gearbox_other",
    "generator_bearing", "generator_stator", "generator_other",
    "generator_cooling", "hydraulic", "rotor_bearing",
    "transformer", "electrical", "wind", "environment",
    "rotor", "pitch", "vibration",
]


def safe_group_row_stats(mat):
    """
    Compute row-wise stats safely even when mat is empty or all-NaN.
    Returns mean, std, min, max arrays.
    """
    if mat.size == 0:
        n = mat.shape[0] if mat.ndim == 2 else 0
        z = np.zeros(n, dtype=np.float32)
        return z, z, z, z

    valid = np.isfinite(mat)
    row_has_valid = valid.any(axis=1)

    mean_ = np.where(row_has_valid, np.nanmean(mat, axis=1), 0.0).astype(np.float32)
    std_ = np.where(row_has_valid, np.nanstd(mat, axis=1), 0.0).astype(np.float32)
    min_ = np.where(row_has_valid, np.nanmin(mat, axis=1), 0.0).astype(np.float32)
    max_ = np.where(row_has_valid, np.nanmax(mat, axis=1), 0.0).astype(np.float32)

    return mean_, std_, min_, max_


def compute_subsystem_features(df, farm_id, row_level=True):
    """
    Compute subsystem-level aggregate features.

    Returns:
        row_level=True  -> DataFrame (n_rows, n_features)
        row_level=False -> (1D numpy array, feature_names)
    """
    gdict = group_dicts[farm_id]
    excluded = counter_sets[farm_id] | angle_sets[farm_id]

    group_cols = defaultdict(list)
    for col in df.columns:
        if (
            col not in META_COLS
            and col not in excluded
            and "avg" in col.lower()
            and pd.api.types.is_numeric_dtype(df[col])
        ):
            grp = gdict.get(col, "other")
            if grp in CROSS_FARM_GROUPS:
                group_cols[grp].append(col)

    if row_level:
        feat_dict = {}

        for grp in CROSS_FARM_GROUPS:
            cols = group_cols.get(grp, [])
            n = len(df)

            if cols:
                mat = df[cols].to_numpy(dtype=np.float32)
                group_mean, group_std, group_min, group_max = safe_group_row_stats(mat)
                group_trend = pd.Series(group_mean).diff(6).fillna(0).to_numpy(dtype=np.float32)
            else:
                group_mean = np.zeros(n, dtype=np.float32)
                group_std = np.zeros(n, dtype=np.float32)
                group_min = np.zeros(n, dtype=np.float32)
                group_max = np.zeros(n, dtype=np.float32)
                group_trend = np.zeros(n, dtype=np.float32)

            feat_dict[f"{grp}_mean"] = group_mean
            feat_dict[f"{grp}_std"] = group_std
            feat_dict[f"{grp}_min"] = group_min
            feat_dict[f"{grp}_max"] = group_max
            feat_dict[f"{grp}_trend"] = group_trend

            for w, wn in [(6, "1hr"), (18, "3hr")]:
                s_mean = pd.Series(group_mean)
                feat_dict[f"{grp}_roll{wn}_mean"] = (
                    s_mean.rolling(w, min_periods=1).mean().to_numpy(dtype=np.float32)
                )
                feat_dict[f"{grp}_roll{wn}_std"] = (
                    s_mean.rolling(w, min_periods=1).std().fillna(0).to_numpy(dtype=np.float32)
                )

        return pd.DataFrame(feat_dict, index=df.index)

    feats = []
    feat_names = []

    for grp in CROSS_FARM_GROUPS:
        cols = group_cols.get(grp, [])
        if cols:
            mat = df[cols].to_numpy(dtype=np.float32)
            mean_, _, _, _ = safe_group_row_stats(mat)
            grp_vals = mean_
        else:
            grp_vals = np.zeros(len(df), dtype=np.float32)

        half = len(grp_vals) // 2
        half_trend = (
            float(np.mean(grp_vals[half:]) - np.mean(grp_vals[:half]))
            if len(grp_vals) > 1 and half > 0 else 0.0
        )

        feats.extend([
            float(np.mean(grp_vals)) if len(grp_vals) else 0.0,
            float(np.std(grp_vals)) if len(grp_vals) else 0.0,
            float(np.min(grp_vals)) if len(grp_vals) else 0.0,
            float(np.max(grp_vals)) if len(grp_vals) else 0.0,
            float(grp_vals[-1] - grp_vals[0]) if len(grp_vals) > 1 else 0.0,
            half_trend,
        ])

        for stat in ["mean", "std", "min", "max", "trend", "half_trend"]:
            feat_names.append(f"{grp}_{stat}")

    return np.array(feats, dtype=np.float32), feat_names


def get_feature_names():
    names = []
    for grp in CROSS_FARM_GROUPS:
        for stat in [
            "mean", "std", "min", "max", "trend",
            "roll1hr_mean", "roll1hr_std", "roll3hr_mean", "roll3hr_std"
        ]:
            names.append(f"{grp}_{stat}")
    return names

# ============================================================================
# ⑥b FARM NORMAL BASELINE
# ============================================================================

def compute_farm_baseline(farm_id, dataset):
    normal_events = [d for d in dataset if d["farm"] == farm_id and d["fault_type"] == "normal"]

    if not normal_events:
        n_feat = len(get_feature_names())
        return np.zeros(n_feat, dtype=np.float32), np.ones(n_feat, dtype=np.float32)

    print(f"  Computing normal baseline from {len(normal_events)} normal events in Farm {farm_id}...")

    all_rows = []
    for d in normal_events:
        try:
            tr, _ = load_event_raw(d["filepath"], farm_id)
            if len(tr) == 0:
                continue
            feat_df = compute_subsystem_features(tr, farm_id, row_level=True)
            all_rows.append(feat_df.to_numpy(dtype=np.float32))
            del tr, feat_df
            gc.collect()
        except Exception:
            continue

    if not all_rows:
        n_feat = len(get_feature_names())
        return np.zeros(n_feat, dtype=np.float32), np.ones(n_feat, dtype=np.float32)

    all_data = np.vstack(all_rows).astype(np.float32)
    del all_rows
    gc.collect()

    mu = np.nanmean(all_data, axis=0).astype(np.float32)
    std = np.nanstd(all_data, axis=0).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)

    print(f"    Baseline: {all_data.shape[0]:,} rows, {all_data.shape[1]} features")
    del all_data
    gc.collect()
    return mu, std


def zscore_normalise(X, mu, std):
    return ((X - mu) / std).astype(np.float32)

# ============================================================================
# ⑦ EVENT METADATA
# ============================================================================

print("\nLoading event metadata...")

event_info = pd.read_csv(EVENT_INFO_PATH)
event_info["farm"] = event_info["Farm"].map(FARM_NAME_MAP)
event_info["event_id"] = event_info["event_id"].astype(str).str.strip()
event_info["event_label"] = event_info["event_label"].fillna("").astype(str).str.strip()
event_info["event_description"] = event_info["event_description"].fillna("normal").astype(str)
event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"] = pd.to_datetime(event_info["event_end"], errors="coerce")

dataset = []
for _, row in event_info.iterrows():
    farm = row["farm"]
    if pd.isna(farm):
        continue

    eid = row["event_id"]
    eid_int = int(eid) if eid.isdigit() else -1
    label = row["event_label"].lower()
    filepath = os.path.join(FARMS[farm], f"{eid}.csv")

    if not os.path.exists(filepath):
        continue

    fault_type = FAULT_LABELS.get(farm, {}).get(eid_int, "normal")
    if label == "normal":
        fault_type = "normal"

    dataset.append({
        "event_id": eid,
        "farm": farm,
        "label": label,
        "fault_type": fault_type,
        "filepath": filepath,
        "event_start": row["event_start"],
        "event_end": row["event_end"],
    })

for farm in ["A", "B", "C"]:
    fe = [d for d in dataset if d["farm"] == farm]
    anom = [d for d in fe if d["label"] == "anomaly"]
    print(f"  Farm {farm}: {len(fe)} events | fault types: {dict(Counter(d['fault_type'] for d in anom))}")

# ============================================================================
# ⑧ BUILD TRAINING MATRIX
# ============================================================================

def build_source_matrix(source_farm, dataset):
    """
    Build training matrix from all events in source farm.

    Returns:
        X, y, label_map, reverse_map, feat_names, src_mu, src_std
    """
    farm_events = [d for d in dataset if d["farm"] == source_farm]
    all_fault_types = sorted(set(d["fault_type"] for d in farm_events))
    label_map = {ft: i for i, ft in enumerate(all_fault_types)}
    reverse_map = {i: ft for ft, i in label_map.items()}
    feat_names = get_feature_names()

    X_parts, y_parts = [], []

    print(f"\n  Building training matrix from Farm {source_farm}...")
    for d in farm_events:
        try:
            tr, pred = load_event_raw(d["filepath"], source_farm)

            if d["fault_type"] != "normal" and len(pred) > 0:
                source_df = pred
            elif len(tr) > 0:
                source_df = tr
            else:
                continue

            feat_df = compute_subsystem_features(source_df, source_farm, row_level=True)

            X = np.nan_to_num(
                feat_df.to_numpy(dtype=np.float32),
                nan=0.0, posinf=0.0, neginf=0.0
            )
            y = np.full(len(X), label_map[d["fault_type"]], dtype=np.int32)

            X_parts.append(X)
            y_parts.append(y)

            del tr, pred, source_df, feat_df
            gc.collect()

        except Exception as e:
            print(f"    ⚠️  Skip {d['event_id']}: {e}")

    if not X_parts:
        return None, None, None, None, None, None, None

    X = np.vstack(X_parts).astype(np.float32)
    y = np.concatenate(y_parts).astype(np.int32)
    del X_parts, y_parts
    gc.collect()

    print(f"  Training matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
    print(f"  Classes: {all_fault_types}")

    src_mu, src_std = compute_farm_baseline(source_farm, dataset)
    X = zscore_normalise(X, src_mu, src_std)
    print(f"  Z-score normalised using Farm {source_farm} normal baseline")

    return X, y, label_map, reverse_map, feat_names, src_mu, src_std

# ============================================================================
# ⑨ TRAIN CLASSIFIER
# ============================================================================

def train_cross_farm_classifier(X_train, y_train):
    unique_cls = sorted(np.unique(y_train).tolist())
    n_cls = len(unique_cls)

    remap = {old: new for new, old in enumerate(unique_cls)}
    reverse_r = {new: old for old, new in remap.items()}
    y_remapped = np.array([remap[y] for y in y_train], dtype=np.int32)

    ulabels, counts = np.unique(y_remapped, return_counts=True)
    raw_w = {c: len(y_remapped) / (n_cls * cnt) for c, cnt in zip(ulabels, counts)}
    min_w = min(raw_w.values())
    weights = {c: min(w / min_w, 8.0) for c, w in raw_w.items()}
    sw = np.array([weights[y] for y in y_remapped], dtype=np.float32)

    params = dict(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )

    if n_cls > 2:
        clf = xgb.XGBClassifier(
            objective="multi:softmax",
            num_class=n_cls,
            eval_metric="mlogloss",
            **params
        )
    else:
        clf = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            **params
        )

    clf.fit(X_train, y_remapped, sample_weight=sw)
    return clf, remap, reverse_r, unique_cls

# ============================================================================
# ⑩ EVALUATE
# ============================================================================

def detect_failure(predictions, true_label, timestamps):
    n = len(predictions)
    w = CONSECUTIVE_WINDOW
    for i in range(n - w + 1):
        if np.all(predictions[i:i + w] == true_label):
            if timestamps is not None and len(timestamps) > 0:
                det_ts = pd.Timestamp(timestamps[i])
                end_ts = pd.Timestamp(timestamps[-1])
                return True, (end_ts - det_ts).total_seconds() / 86400.0
            return True, None
    return False, None


def evaluate_on_target(target_farm, clf, remap, reverse_r,
                       source_label_map, source_reverse_map,
                       dataset, feat_names, tgt_mu=None, tgt_std=None):
    farm_events = [d for d in dataset if d["farm"] == target_farm]
    results = []

    print(f"\n  Evaluating on Farm {target_farm} ({len(farm_events)} events)...")

    for d in farm_events:
        is_fault = d["label"] == "anomaly"
        true_type = d["fault_type"]

        try:
            _, pr = load_event_raw(d["filepath"], target_farm)
            if len(pr) == 0:
                continue

            feat_df = compute_subsystem_features(pr, target_farm, row_level=True)
            X_test = np.nan_to_num(
                feat_df.to_numpy(dtype=np.float32),
                nan=0.0, posinf=0.0, neginf=0.0
            )

            if tgt_mu is not None and tgt_std is not None:
                X_test = zscore_normalise(X_test, tgt_mu, tgt_std)

            true_label_int = source_label_map.get(true_type, -1)
            timestamps = pr["time_stamp"].values if "time_stamp" in pr.columns else None

            preds_remapped = clf.predict(X_test)

            if np.issubdtype(np.asarray(preds_remapped).dtype, np.floating):
                preds_remapped = np.rint(preds_remapped).astype(int)

            preds_source = np.array([reverse_r[int(p)] for p in preds_remapped], dtype=np.int32)
            preds_type = np.array([source_reverse_map[int(p)] for p in preds_source], dtype=object)

            detected, horizon = False, None
            if is_fault and true_label_int >= 0:
                detected, horizon = detect_failure(preds_source, true_label_int, timestamps)

            pred_type = Counter(preds_type.tolist()).most_common(1)[0][0]
            is_comparable = (true_type in COMPARABLE_FAULT_TYPES) and (true_type in source_label_map)

            status = "✓" if detected else ("✗" if is_fault else "—")
            comparable_str = "[comparable]" if is_comparable else "[source-only]"

            if is_fault:
                print(
                    f"    E{str(d['event_id']):>4} "
                    f"[{true_type:22}] → [{pred_type:22}] {status} {comparable_str}"
                )

            results.append({
                "event_id": d["event_id"],
                "target_farm": target_farm,
                "fault_type": true_type,
                "pred_type": pred_type,
                "is_fault": is_fault,
                "is_comparable": is_comparable,
                "detected": bool(detected),
                "horizon_days": round(horizon, 2) if horizon is not None else None,
            })

            del pr, feat_df, X_test, preds_remapped, preds_source, preds_type
            gc.collect()

        except Exception as e:
            print(f"    ⚠️  E{d['event_id']}: {e}")

    return results

# ============================================================================
# ⑪ METRICS
# ============================================================================

def compute_cross_farm_metrics(results, source_farm, target_farm):
    fault_res = [r for r in results if r["is_fault"]]
    comparable_res = [r for r in fault_res if r["is_comparable"]]
    sourceonly_res = [r for r in fault_res if not r["is_comparable"]]

    def _metrics(res_list, label):
        if not res_list:
            return None

        total = len(res_list)
        detected = sum(1 for r in res_list if r["detected"])
        rate = detected / total if total > 0 else 0.0
        horizons = [r["horizon_days"] for r in res_list if r["horizon_days"] is not None]
        avg_h = float(np.mean(horizons)) if horizons else 0.0

        by_type = defaultdict(lambda: {"total": 0, "detected": 0, "horizons": []})
        for r in res_list:
            ft = r["fault_type"]
            by_type[ft]["total"] += 1
            if r["detected"]:
                by_type[ft]["detected"] += 1
            if r["horizon_days"] is not None:
                by_type[ft]["horizons"].append(r["horizon_days"])

        print(f"\n  {label}:")
        print(f"  Overall: {detected}/{total} = {rate:.1%} | avg horizon: {avg_h:.1f} days")
        print(f"  {'Fault Type':25} {'Det':>4}{'Tot':>4}  {'Rate':>6}  {'Avg Horizon':>12}")
        print(f"  {'─' * 58}")

        for ft in sorted(by_type.keys()):
            v = by_type[ft]
            r_ = v["detected"] / v["total"] if v["total"] > 0 else 0.0
            h_ = float(np.mean(v["horizons"])) if v["horizons"] else 0.0
            print(f"  {ft:25} {v['detected']:4d}{v['total']:4d}  {r_:6.1%}  {h_:10.1f}d")

        return {
            "total": total,
            "detected": detected,
            "detection_rate": round(rate, 4),
            "avg_horizon_days": round(avg_h, 2),
            "by_type": {
                ft: {
                    "total": int(v["total"]),
                    "detected": int(v["detected"]),
                    "horizons": [float(x) for x in v["horizons"]],
                }
                for ft, v in by_type.items()
            },
        }

    print(f"\n  {'=' * 60}")
    print(f"  Farm {source_farm} → Farm {target_farm} Cross-Farm Results")
    print(f"  {'=' * 60}")

    comp_m = _metrics(comparable_res, "Comparable fault types (exist in both farms)")
    solo_m = _metrics(sourceonly_res, "Source-only fault types (not in source training)")
    all_m = _metrics(fault_res, "All fault types")

    return {
        "source_farm": source_farm,
        "target_farm": target_farm,
        "comparable": comp_m,
        "source_only": solo_m,
        "all_faults": all_m,
    }

# ============================================================================
# ⑫ FEATURE IMPORTANCE
# ============================================================================

def get_feature_importance(clf, reverse_r, source_reverse_map, feat_names):
    if not hasattr(clf, "feature_importances_"):
        return {}
    importance = {name: float(imp) for name, imp in zip(feat_names, clf.feature_importances_)}
    return dict(sorted(importance.items(), key=lambda x: x[1], reverse=True)[:20])

# ============================================================================
# ⑬ PLOTS
# ============================================================================

def plot_cross_farm_results(all_cross_results, output_dir):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("  matplotlib not available — skipping plots")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    directions, comp_rates, all_rates = [], [], []

    for _, metrics in all_cross_results.items():
        src, tgt = metrics["source_farm"], metrics["target_farm"]
        directions.append(f"Farm {src}→{tgt}")
        comp = metrics.get("comparable") or {}
        all_ = metrics.get("all_faults") or {}
        comp_rates.append(comp.get("detection_rate", 0))
        all_rates.append(all_.get("detection_rate", 0))

    ax = axes[0]
    x = np.arange(len(directions))
    w = 0.35
    b1 = ax.bar(x - w/2, comp_rates, w, label="Comparable types", alpha=0.85)
    b2 = ax.bar(x + w/2, all_rates, w, label="All fault types", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(directions, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Detection Rate", fontsize=10)
    ax.set_title("Cross-Farm Detection Rate\n(Train on source → Test on target)", fontsize=11, fontweight="bold")
    ax.bar_label(b1, fmt="%.0%", padding=2, fontsize=8)
    ax.bar_label(b2, fmt="%.0%", padding=2, fontsize=8)
    ax.axhline(0.5, color="#555", ls="--", lw=1, alpha=0.6, label="50% reference")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    ax = axes[1]
    horizons = []
    for _, metrics in all_cross_results.items():
        comp = metrics.get("comparable") or {}
        horizons.append(comp.get("avg_horizon_days", 0))

    bars = ax.bar(directions, horizons, alpha=0.85)
    ax.set_ylabel("Average Detection Horizon (days)", fontsize=10)
    ax.set_title("Cross-Farm Average Early Warning Time\n(Comparable fault types only)", fontsize=11, fontweight="bold")
    ax.bar_label(bars, fmt="%.1f d", padding=3, fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right", fontsize=9)

    plt.tight_layout()
    p = os.path.join(output_dir, "cross_farm_detection_rates.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {p}")

    for key, label in [("A_to_C", "Farm A → Farm C"), ("C_to_A", "Farm C → Farm A")]:
        if key not in all_cross_results:
            continue

        metrics = all_cross_results[key]
        by_type = (metrics.get("all_faults") or {}).get("by_type", {})
        if not by_type:
            continue

        types = sorted(by_type.keys())
        rates = [
            by_type[t]["detected"] / by_type[t]["total"] if by_type[t]["total"] > 0 else 0
            for t in types
        ]
        colors_ = ["#2ca02c" if r >= 0.5 else "#d62728" for r in rates]

        fig, ax = plt.subplots(figsize=(max(10, len(types) * 1.2), 5))
        bars = ax.bar(types, rates, color=colors_, alpha=0.85)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("Detection Rate", fontsize=10)
        ax.set_title(f"Cross-Farm: {label} — Detection Rate by Fault Type", fontsize=11, fontweight="bold")
        ax.bar_label(bars, fmt="%.0%", padding=3, fontsize=9)
        ax.axhline(0.5, color="#555", ls="--", lw=1, alpha=0.6)
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right", fontsize=9)
        plt.tight_layout()

        fname = f"cross_farm_{key.lower()}_by_type.png"
        p = os.path.join(output_dir, fname)
        plt.savefig(p, dpi=130, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {p}")

# ============================================================================
# ⑭ MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("CROSS-FARM FAULT CLASSIFICATION VALIDATION")
    print("=" * 80)
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}\n")
    start = datetime.now()

    print("Strategy: Subsystem-aggregate features (farm-agnostic representation)")
    print(f"Groups: {CROSS_FARM_GROUPS}")
    print("Features per group: mean, std, min, max, trend, roll1hr_mean, roll1hr_std, roll3hr_mean, roll3hr_std")
    print(f"Total features: {len(CROSS_FARM_GROUPS) * 9}")

    all_cross_results = {}
    all_importances = {}

    print("\nPre-computing normal baselines for z-score normalisation...")
    mu_a, std_a = compute_farm_baseline("A", dataset)
    mu_b, std_b = compute_farm_baseline("B", dataset)
    mu_c, std_c = compute_farm_baseline("C", dataset)
    print("Baselines computed.\n")

    print(f"\n{'=' * 80}")
    print("DIRECTION 1: Train Farm A → Predict Farm C")
    print("=" * 80)

    X_a, y_a, lmap_a, rmap_a, feat_names, _, _ = build_source_matrix("A", dataset)
    if X_a is not None:
        clf_a, remap_a, reverse_r_a, unique_a = train_cross_farm_classifier(X_a, y_a)
        del X_a, y_a
        gc.collect()

        results_ac = evaluate_on_target(
            "C", clf_a, remap_a, reverse_r_a, lmap_a, rmap_a,
            dataset, feat_names, tgt_mu=mu_c, tgt_std=std_c
        )
        metrics_ac = compute_cross_farm_metrics(results_ac, "A", "C")
        all_cross_results["A_to_C"] = metrics_ac
        all_importances["A_to_C"] = get_feature_importance(clf_a, remap_a, rmap_a, feat_names)

        del clf_a
        gc.collect()

    print(f"\n{'=' * 80}")
    print("DIRECTION 2: Train Farm C → Predict Farm A")
    print("=" * 80)

    X_c, y_c, lmap_c, rmap_c, feat_names, _, _ = build_source_matrix("C", dataset)
    if X_c is not None:
        clf_c, remap_c, reverse_r_c, unique_c = train_cross_farm_classifier(X_c, y_c)
        del X_c, y_c
        gc.collect()

        results_ca = evaluate_on_target(
            "A", clf_c, remap_c, reverse_r_c, lmap_c, rmap_c,
            dataset, feat_names, tgt_mu=mu_a, tgt_std=std_a
        )
        metrics_ca = compute_cross_farm_metrics(results_ca, "C", "A")
        all_cross_results["C_to_A"] = metrics_ca
        all_importances["C_to_A"] = get_feature_importance(clf_c, remap_c, rmap_c, feat_names)

        print("\n  Supplementary: Farm C → Farm B")
        results_cb = evaluate_on_target(
            "B", clf_c, remap_c, reverse_r_c, lmap_c, rmap_c,
            dataset, feat_names, tgt_mu=mu_b, tgt_std=std_b
        )
        metrics_cb = compute_cross_farm_metrics(results_cb, "C", "B")
        all_cross_results["C_to_B"] = metrics_cb

        del clf_c
        gc.collect()

    print(f"\n{'=' * 80}")
    print("DIRECTION 3: Train Farm B → Predict Farm A and Farm C")
    print("=" * 80)

    X_b, y_b, lmap_b, rmap_b, feat_names, _, _ = build_source_matrix("B", dataset)
    if X_b is not None:
        clf_b, remap_b, reverse_r_b, unique_b = train_cross_farm_classifier(X_b, y_b)
        del X_b, y_b
        gc.collect()

        print("\n  Farm B → Farm A:")
        results_ba = evaluate_on_target(
            "A", clf_b, remap_b, reverse_r_b, lmap_b, rmap_b,
            dataset, feat_names, tgt_mu=mu_a, tgt_std=std_a
        )
        metrics_ba = compute_cross_farm_metrics(results_ba, "B", "A")
        all_cross_results["B_to_A"] = metrics_ba

        print("\n  Farm B → Farm C:")
        results_bc = evaluate_on_target(
            "C", clf_b, remap_b, reverse_r_b, lmap_b, rmap_b,
            dataset, feat_names, tgt_mu=mu_c, tgt_std=std_c
        )
        metrics_bc = compute_cross_farm_metrics(results_bc, "B", "C")
        all_cross_results["B_to_C"] = metrics_bc

        del clf_b
        gc.collect()

    print(f"\n{'=' * 80}")
    print("CROSS-FARM VALIDATION SUMMARY")
    print("=" * 80)
    print(f"\n  {'Direction':20} {'Comparable':>12} {'All types':>10} {'Avg Horizon':>14}")
    print(f"  {'─' * 60}")

    for _, metrics in all_cross_results.items():
        src = metrics["source_farm"]
        tgt = metrics["target_farm"]
        comp = metrics.get("comparable") or {}
        all_ = metrics.get("all_faults") or {}
        cr = comp.get("detection_rate", 0)
        ar = all_.get("detection_rate", 0)
        ah = comp.get("avg_horizon_days", 0)
        print(f"  Farm {src} → Farm {tgt}:     {cr:>10.1%}   {ar:>8.1%}   {ah:>12.1f}d")

    print("\nGenerating plots...")
    plot_cross_farm_results(all_cross_results, OUTPUT_DIR)

    def json_safe(obj):
        if isinstance(obj, dict):
            return {k: json_safe(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [json_safe(i) for i in obj]
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    out = json_safe({
        "timestamp": datetime.now().isoformat(),
        "script": "script_3_cross_farm_validation",
        "base_dir": BASE_DIR,
        "feature_groups": CROSS_FARM_GROUPS,
        "n_features": len(CROSS_FARM_GROUPS) * 9,
        "consecutive_window": CONSECUTIVE_WINDOW,
        "results": all_cross_results,
        "feature_importance": all_importances,
    })

    rj = os.path.join(OUTPUT_DIR, "cross_farm_results.json")
    with open(rj, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"\nSaved: {rj}")
    print(f"\nRuntime: {datetime.now() - start}")
    print("✅ DONE")


if __name__ == "__main__":
    main()

Building sensor rename dictionaries...
  Farm A: 81 mappings
  Farm B: 252 mappings
  Farm C: 952 mappings
Building subsystem group dictionaries...
  Farm A: 81 column→group mappings
  Farm B: 252 column→group mappings
  Farm C: 952 column→group mappings

Loading event metadata...
  Farm A: 22 events | fault types: {'transformer': 1, 'hydraulic': 6, 'gearbox': 3, 'generator_bearing': 2}
  Farm B: 15 events | fault types: {'transformer': 3, 'rotor_bearing': 3}
  Farm C: 58 events | fault types: {'pitch_electrical': 11, 'converter': 2, 'hydraulic': 1, 'gearbox_oil': 3, 'rotor_brake': 2, 'communication': 3, 'pitch_mechanical': 1, 'cooling_water': 1, 'transformer': 1, 'yaw': 1, 'electrical_internal': 1}
CROSS-FARM FAULT CLASSIFICATION VALIDATION
Started: 13:15:24

Strategy: Subsystem-aggregate features (farm-agnostic representation)
Groups: ['gearbox_bearing', 'gearbox_oil', 'gearbox_other', 'generator_bearing', 'generator_stator', 'generator_other', 'generator_cooling', 'hydraulic', 'roto

In [7]:
"""
================================================================================
POWER COLUMN INSPECTOR — CARE Benchmark Dataset
================================================================================
Loads one representative event per farm, renames columns using the feature
description file, and produces a full audit of every column that could
plausibly be an active power measurement.

For each candidate column it reports:
  - Column name (renamed)
  - Original sensor name
  - Feature description text
  - Unit
  - Min / Mean / Max / Std values
  - % of rows that are positive (generating)
  - % of rows that are zero or negative
  - % of rows that are NaN
  - Implied capacity factor (if values appear to be in kW)
  - Whether CARE appears to have normalised it (max ~ 1.0)

Outputs:
  - Console report
  - power_column_audit.csv  (full table, all candidates)
  - column_statistics.csv   (all renamed columns, sorted by mean desc)

Run this BEFORE Script 4 to confirm which power column to use.
================================================================================
"""

import os, re, gc, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIG — update BASE_DIR if needed
# ============================================================================

BASE_DIR = Path(r"D:\files\project\CARE_To_Compare")

FARMS = {
    "A": BASE_DIR / "Wind Farm A" / "datasets",
    "B": BASE_DIR / "Wind Farm B" / "datasets",
    "C": BASE_DIR / "Wind Farm C" / "datasets",
}
FEATURE_DESC = {
    "A": BASE_DIR / "Wind Farm A" / "feature_description.csv",
    "B": BASE_DIR / "Wind Farm B" / "feature_description.csv",
    "C": BASE_DIR / "Wind Farm C" / "feature_description.csv",
}
EVENT_CSV = BASE_DIR / "df_all__1_.csv"

OUTPUT_DIR = BASE_DIR / "power_column_audit"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RATED_MW = {"A": 2.0, "B": 5.0, "C": 5.0}
RATED_KW = {f: v * 1000 for f, v in RATED_MW.items()}

# Keywords that suggest a column might be power-related
POWER_KEYWORDS = [
    "power", "watt", " kw", "mw", "gridpower", "activepower",
    "active_power", "real_power", "grid_power", "netpower",
    "production", "output", "generatedpower", "generatedpower",
    "p_gen", "p_grid", "p_ac", "generator_power",
]
EXCLUDE_KEYWORDS = [
    "reactive", "apparent", "kwh", "mwh", "total", "cumulative",
    "counter", "energy", "var", "kva", "available_power",
    "setpoint", "limit", "reference", "demand", "consumed",
    "auxiliary", "aux", "losses", "loss",
]

# ============================================================================
# STEP 1: Build rename dict from feature_description.csv
# ============================================================================

STAT_MAP = {"average": "avg", "std_dev": "std",
            "maximum": "max", "minimum": "min"}

def _clean(text, is_unit=False):
    if pd.isna(text) or str(text).strip() in ("", "nan", "-", "none"):
        return ""
    t = str(text).strip()
    for old, new in [
        ("°C","degC"),("ºC","degC"),("Celsius","degC"),
        ("/","_per_"),("%","pct"),(" ",""),("^",""),(".",""),
    ]:
        t = t.replace(old, new)
    t = re.sub(r"[^\x00-\x7F]+", "", t)
    if not is_unit:
        t = re.sub(r"[^\w\s]", " ", t)
        t = re.sub(r"\s+", "_", t.strip())
        t = re.sub(r"_+", "_", t).strip("_")
    else:
        t = re.sub(r"[^\w]", "", t)
    return t

def build_full_registry(farm_id):
    """
    Returns:
        rename:    {original_col → new_col}
        registry:  {new_col → {sensor_name, description, unit, stat}}
    """
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(FEATURE_DESC[farm_id], sep=";", encoding=enc)
            break
        except UnicodeDecodeError:
            continue

    rename   = {}
    registry = {}

    for _, row in df.iterrows():
        sid      = str(row["sensor_name"]).strip()
        desc_raw = str(row.get("description", "")).strip()
        unit_raw = str(row.get("unit", "")).strip()
        desc     = _clean(desc_raw)
        unit     = _clean(unit_raw, is_unit=True)

        stats_raw = str(row.get("statistics_type", "")).strip()
        stats = [] if stats_raw.lower() in ("nan","") else \
                [s.strip() for s in stats_raw.split(",") if s.strip()]
        entries = [(f"{sid}_{STAT_MAP.get(s,s)}", STAT_MAP.get(s,s))
                   for s in stats] if stats else [(sid, "")]

        for old_col, sc in entries:
            parts   = [p for p in [desc, unit, sc] if p]
            new_col = "_".join(parts) if parts else old_col
            rename[old_col]   = new_col
            registry[new_col] = {
                "sensor_name": sid,
                "description": desc_raw,
                "unit":        unit_raw,
                "stat":        sc,
                "original_col": old_col,
            }

    return rename, registry

# ============================================================================
# STEP 2: Load a representative event for each farm
#         Use one anomaly event + one normal event for robustness
# ============================================================================

def pick_events(farm_id, event_df, n=3):
    """Pick n events from this farm — mix of anomaly and normal."""
    farm_ev = event_df[event_df["farm"] == farm_id].copy()
    anomaly = farm_ev[farm_ev["event_label"] == "anomaly"]
    normal  = farm_ev[farm_ev["event_label"] == "normal"]

    picks = []
    for pool in [anomaly, normal]:
        for _, row in pool.head(max(1, n//2)).iterrows():
            fp = FARMS[farm_id] / f"{row['event_id']}.csv"
            if fp.exists():
                picks.append({"event_id": row["event_id"],
                               "label":    row["event_label"],
                               "filepath": fp})
            if len(picks) >= n:
                break
        if len(picks) >= n:
            break
    return picks

def load_events(farm_id, event_df, n=3):
    rename, registry = build_full_registry(farm_id)
    picks = pick_events(farm_id, event_df, n)

    frames = []
    for pick in picks:
        chunks = []
        for chunk in pd.read_csv(
            pick["filepath"], sep=";", chunksize=50_000,
            low_memory=False, encoding="latin-1"
        ):
            chunk = chunk.rename(columns=rename)
            chunks.append(chunk)
        df = pd.concat(chunks, ignore_index=True)
        df["_event_id"] = pick["event_id"]
        df["_label"]    = pick["label"]
        frames.append(df)
        del chunks, df; gc.collect()

    combined = pd.concat(frames, ignore_index=True)
    return combined, registry, picks

# ============================================================================
# STEP 3: Score every column and identify power candidates
# ============================================================================

def score_columns(df, registry, farm_id, rated_kw):
    """
    For every numeric column, compute statistics and flag power candidates.
    Returns a DataFrame sorted by power-likelihood score.
    """
    rows = []
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # exclude meta columns
    numeric_cols = [c for c in numeric_cols
                    if not c.startswith("_") and c != "status_type_id"]

    total_rows = len(df)

    for col in numeric_cols:
        vals = df[col].values.astype(np.float64)

        n_nan  = int(np.sum(np.isnan(vals)))
        vals_c = vals[~np.isnan(vals)]
        n_valid = len(vals_c)

        if n_valid == 0:
            continue

        col_min  = float(np.min(vals_c))
        col_max  = float(np.max(vals_c))
        col_mean = float(np.mean(vals_c))
        col_std  = float(np.std(vals_c))
        col_p25  = float(np.percentile(vals_c, 25))
        col_p50  = float(np.percentile(vals_c, 50))
        col_p75  = float(np.percentile(vals_c, 75))

        pct_nan      = round(n_nan   / total_rows * 100, 1)
        pct_zero_neg = round(np.sum(vals_c <= 0) / n_valid * 100, 1)
        pct_positive = round(np.sum(vals_c >  0) / n_valid * 100, 1)

        # Is this likely normalised (CARE divides by rated power)?
        is_normalised = (0 <= col_max <= 1.2) and (col_mean < 1.0)
        # Is this likely in kW?
        is_kw_scale   = (col_max > 10)
        # Is this likely in MW?
        is_mw_scale   = (1 < col_max <= rated_kw / 1000 * 1.5)

        # Implied CF if interpreted as kW avg measurement
        if is_normalised:
            implied_cf = col_mean   # already fractional
            kw_equiv   = col_mean * rated_kw
        elif is_kw_scale:
            implied_cf = col_mean / rated_kw
            kw_equiv   = col_mean
        elif is_mw_scale:
            implied_cf = col_mean / (rated_kw / 1000)
            kw_equiv   = col_mean * 1000
        else:
            implied_cf = np.nan
            kw_equiv   = np.nan

        # ── Power keyword matching ────────────────────────────────────────
        reg_entry    = registry.get(col, {})
        desc_raw     = str(reg_entry.get("description","")).lower()
        unit_raw     = str(reg_entry.get("unit","")).lower()
        sensor_name  = reg_entry.get("sensor_name","")
        stat         = reg_entry.get("stat","")
        col_lower    = col.lower()

        # Score: higher = more likely to be the right power column
        score = 0

        # Positive signals
        if any(k in col_lower  for k in POWER_KEYWORDS): score += 3
        if any(k in desc_raw   for k in POWER_KEYWORDS): score += 3
        if "kw"   in unit_raw  and "kwh" not in unit_raw: score += 4
        if "mw"   in unit_raw  and "mwh" not in unit_raw: score += 3
        if stat   == "avg":  score += 2
        if pct_positive > 50: score += 2
        if 0.1 < implied_cf < 0.7 if not np.isnan(implied_cf) else False: score += 3

        # Negative signals (likely not the right column)
        if any(k in col_lower  for k in EXCLUDE_KEYWORDS): score -= 5
        if any(k in desc_raw   for k in EXCLUDE_KEYWORDS): score -= 5
        if "kwh"  in unit_raw: score -= 8
        if "mwh"  in unit_raw: score -= 8
        if "var"  in unit_raw: score -= 5
        if "kva"  in unit_raw: score -= 5
        if stat   in ("max","min"): score -= 1
        if implied_cf > 1.0 if not np.isnan(implied_cf) else False: score -= 6
        if pct_positive < 20: score -= 3

        rows.append({
            "farm":          farm_id,
            "column":        col,
            "sensor_name":   sensor_name,
            "description":   reg_entry.get("description",""),
            "unit":          reg_entry.get("unit",""),
            "stat":          stat,
            "min":           round(col_min, 4),
            "p25":           round(col_p25, 4),
            "median":        round(col_p50, 4),
            "mean":          round(col_mean, 4),
            "p75":           round(col_p75, 4),
            "max":           round(col_max, 4),
            "std":           round(col_std, 4),
            "pct_positive":  pct_positive,
            "pct_zero_neg":  pct_zero_neg,
            "pct_nan":       pct_nan,
            "is_normalised": is_normalised,
            "is_kw_scale":   is_kw_scale,
            "implied_cf":    round(implied_cf, 3) if not np.isnan(implied_cf) else None,
            "score":         score,
        })

    result = pd.DataFrame(rows).sort_values("score", ascending=False)
    return result

# ============================================================================
# STEP 4: Print a clean report
# ============================================================================

def print_farm_report(farm_id, scored, picks, rated_kw):
    print(f"\n{'='*80}")
    print(f"  FARM {farm_id} — POWER COLUMN AUDIT")
    print(f"{'='*80}")
    print(f"  Rated capacity: {rated_kw/1000:.1f} MW  ({rated_kw:,.0f} kW)")
    print(f"  Events analysed: "
          + ", ".join([f"E{p['event_id']} ({p['label']})" for p in picks]))

    # Top candidates (score > 0)
    candidates = scored[scored["score"] > 0].head(15)

    if candidates.empty:
        print("\n  No strong candidates found — check sensor descriptions.")
        return

    print(f"\n  TOP POWER COLUMN CANDIDATES (sorted by score):\n")
    print(f"  {'#':<3} {'Score':>5}  {'Column':<45}  {'Unit':<8}  "
          f"{'Mean':>8}  {'Max':>9}  {'CF%':>6}  {'Pos%':>5}  "
          f"{'Norm?':>5}")
    print(f"  {'─'*130}")

    for rank, (_, row) in enumerate(candidates.iterrows(), 1):
        cf_str   = f"{row['implied_cf']*100:.1f}%" if row["implied_cf"] else "  —  "
        norm_str = "YES" if row["is_normalised"] else "no"
        print(f"  {rank:<3} {row['score']:>5}  "
              f"{row['column'][:45]:<45}  "
              f"{str(row['unit'])[:8]:<8}  "
              f"{row['mean']:>8.3f}  "
              f"{row['max']:>9.3f}  "
              f"{cf_str:>6}  "
              f"{row['pct_positive']:>5.1f}%  "
              f"{norm_str:>5}")
        if row["description"]:
            print(f"       Description: {row['description'][:80]}")
        print()

    # Best recommendation
    best = candidates.iloc[0]
    print(f"\n  ✅ RECOMMENDED COLUMN: '{best['column']}'")
    print(f"     Description: {best['description']}")
    print(f"     Unit: {best['unit']}")
    cf_str = f"{best['implied_cf']*100:.1f}%" if best["implied_cf"] else "unknown"
    print(f"     Implied capacity factor: {cf_str}")
    norm_txt = ("NORMALISED — multiply by rated kW to get real kW"
                if best["is_normalised"] else "RAW kW values")
    print(f"     Scale: {norm_txt}")

    # Status-conditional breakdown
    print(f"\n  STATUS-CONDITIONAL STATS for '{best['column']}':")
    print(f"  (to verify power is > 0 only during status 0)\n")

# ============================================================================
# STEP 5: Status-conditional power check
# ============================================================================

def status_conditional_check(df, col, farm_id, rated_kw):
    if col not in df.columns or "status_type_id" not in df.columns:
        return

    is_norm = df[col].max() <= 1.5

    print(f"  {'Status':<35} {'N rows':>8}  {'Mean':>9}  "
          f"{'Max':>9}  {'Med':>9}  {'Pos%':>6}")
    print(f"  {'─'*80}")

    status_labels = {
        0: "Status 0 — Normal generation",
        1: "Status 1 — Curtailed",
        2: "Status 2 — Standby (healthy)",
        3: "Status 3 — Service/maintenance",
        4: "Status 4 — Fault-active",
    }

    for code in sorted(df["status_type_id"].dropna().unique()):
        try:
            code_int = int(float(code))
        except (ValueError, TypeError):
            continue
        mask  = df["status_type_id"] == code
        sub   = df.loc[mask, col].dropna().values.astype(np.float64)
        if len(sub) == 0:
            continue
        if is_norm:
            sub_kw = sub * rated_kw
        else:
            sub_kw = sub

        label     = status_labels.get(code_int, f"Status {code_int}")
        n         = len(sub_kw)
        mean_kw   = np.mean(sub_kw)
        max_kw    = np.max(sub_kw)
        med_kw    = np.median(sub_kw)
        pct_pos   = np.sum(sub_kw > 0) / n * 100

        unit_lbl = "kW" if is_norm else "kW"
        print(f"  {label:<35} {n:>8,}  "
              f"{mean_kw:>8.1f}  "
              f"{max_kw:>9.1f}  "
              f"{med_kw:>9.1f}  "
              f"{pct_pos:>5.1f}%")

    print()
    # Total implied CF if only summing status-0 rows
    status0 = df[df["status_type_id"] == 0][col].dropna().values.astype(np.float64)
    if len(status0) > 0:
        if is_norm:
            status0_kw = status0 * rated_kw
        else:
            status0_kw = status0
        mean_gen = np.mean(status0_kw)
        cf_gen   = mean_gen / rated_kw
        print(f"  → Status-0 mean output: {mean_gen:.1f} kW  "
              f"(implied CF during generation: {cf_gen:.1%})")
        print(f"  → Rated: {rated_kw:,} kW")

# ============================================================================
# MAIN
# ============================================================================

def main():
    print("="*80)
    print("POWER COLUMN INSPECTOR — CARE Benchmark Dataset")
    print("="*80)

    FARM_NAME_MAP = {"Farm A": "A", "Farm B": "B", "Farm C": "C"}
    event_df = pd.read_csv(EVENT_CSV)
    event_df["farm"]      = event_df["Farm"].map(FARM_NAME_MAP)
    event_df["event_id"]  = event_df["event_id"].astype(str)

    all_scored = []

    for farm_id in ["A", "B", "C"]:
        print(f"\nLoading Farm {farm_id} events...")
        rated_kw = RATED_KW[farm_id]

        df, registry, picks = load_events(farm_id, event_df, n=4)
        print(f"  Loaded {len(df):,} rows across {len(picks)} events")

        scored = score_columns(df, registry, farm_id, rated_kw)
        scored["farm"] = farm_id
        all_scored.append(scored)

        print_farm_report(farm_id, scored, picks, rated_kw)

        # Status-conditional check on top candidate
        if not scored[scored["score"] > 0].empty:
            best_col = scored[scored["score"] > 0].iloc[0]["column"]
            status_conditional_check(df, best_col, farm_id, rated_kw)

        # Save per-farm CSV
        out = Path(OUTPUT_DIR) / f"farm_{farm_id}_column_scores.csv"
        scored.to_csv(out, index=False)
        print(f"\n  Full column scores saved → {out}")

        del df; gc.collect()

    # Save combined audit CSV
    combined = pd.concat(all_scored, ignore_index=True)
    combined.to_csv(Path(OUTPUT_DIR) / "power_column_audit_all_farms.csv",
                    index=False)
    print(f"\n\nFull audit saved → "
          f"{Path(OUTPUT_DIR) / 'power_column_audit_all_farms.csv'}")

    print(f"\n{'='*80}")
    print("DONE — check the printed report and CSV for the right power column.")
    print("="*80)

if __name__ == "__main__":
    main()

POWER COLUMN INSPECTOR — CARE Benchmark Dataset

Loading Farm A events...
  Loaded 216,919 rows across 4 events

  FARM A — POWER COLUMN AUDIT
  Rated capacity: 2.0 MW  (2,000 kW)
  Events analysed: E68 (anomaly), E22 (anomaly), E25 (normal), E69 (normal)

  TOP POWER COLUMN CANDIDATES (sorted by score):

  #   Score  Column                                         Unit          Mean        Max     CF%   Pos%  Norm?
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1      17  Possiblegridactivepower_kW_avg                 kW           0.288      0.976   28.8%   85.6%    YES
       Description: Possible grid active power

  2      17  Gridpower_kW_avg                               kW           0.272      0.976   27.2%   69.5%    YES
       Description: Grid power

  3      14  Possiblegridactivepower_kW_max                 kW           0.392      0.976   39.2%   86.1%    YES
       Description: Possible 

In [4]:
# ============================================================================
# SCRIPT 4 — OPERATIONAL ANALYSIS 
# ============================================================================

from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================================
# CONFIG
# ============================================================================

BASE_DIR = Path(r"D:\files\project\CARE_To_Compare")

# Keep the old output structure, just update the root path as requested
STAGE1_EVENT_CSV = BASE_DIR / "pipeline_final_v5" / "event_results_final.csv"
STAGE1_RESULTS_JSON = BASE_DIR / "pipeline_final_v5" / "results_final.json"

STAGE2_RESULTS_CSV = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.csv"
STAGE2_RESULTS_JSON = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.json"

STAGE3_RESULTS_JSON = BASE_DIR / "cross_farm_outputs" / "cross_farm_results.json"

OUTPUT_DIR = BASE_DIR / "operational_analysis_outputs"
REPORTS_DIR = OUTPUT_DIR / "reports"
PLOTS_DIR = OUTPUT_DIR / "plots"

for d in [OUTPUT_DIR, REPORTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================================
# HELPERS
# ============================================================================

def safe_read_csv(path: Path) -> pd.DataFrame | None:
    if path.exists():
        return pd.read_csv(path)
    return None


def safe_read_json(path: Path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple, set, np.ndarray)) else False:
        return None
    return obj


def label_is_fault(v) -> bool:
    s = str(v).strip().lower()
    return s in {"anomaly", "fault", "1", "true", "yes"}


def ensure_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


# ============================================================================
# LOADERS
# ============================================================================

def load_stage1():
    df = safe_read_csv(STAGE1_EVENT_CSV)
    js = safe_read_json(STAGE1_RESULTS_JSON)
    return df, js


def load_stage2():
    df = safe_read_csv(STAGE2_RESULTS_CSV)
    js = safe_read_json(STAGE2_RESULTS_JSON)
    return df, js


def load_stage3():
    js = safe_read_json(STAGE3_RESULTS_JSON)
    return js


# ============================================================================
# STAGE 1 SUMMARY
# Actual columns:
# ['event_id', 'farm', 'label', 'description', 'alarm_raised', 'alarm_source',
#  'max_criticality', 'crit_global', 'crit_subsystem', 'anomaly_rate',
#  'earliness', 'top_subsystem', 'n_pred_rows']
# ============================================================================

def build_stage1_summary(stage1_df: pd.DataFrame, stage1_json=None) -> pd.DataFrame:
    if stage1_df is None or stage1_df.empty:
        return pd.DataFrame()

    df = stage1_df.copy()
    df = ensure_numeric(df, ["alarm_raised", "anomaly_rate", "earliness"])

    df["is_fault_event"] = df["label"].apply(label_is_fault)
    df["is_normal_event"] = ~df["is_fault_event"]
    df["alarm_raised"] = df["alarm_raised"].fillna(0).astype(int)

    rows = []
    for farm, g in df.groupby("farm"):
        anomaly = g[g["is_fault_event"]]
        normal = g[g["is_normal_event"]]

        anomaly_events = len(anomaly)
        normal_events = len(normal)

        detected_events = int(anomaly["alarm_raised"].sum()) if anomaly_events else 0
        false_alarm_events = int(normal["alarm_raised"].sum()) if normal_events else 0

        stage1_detection_rate = detected_events / anomaly_events if anomaly_events else np.nan
        stage1_false_alarm_rate = false_alarm_events / normal_events if normal_events else np.nan

        mean_anomaly_rate_fault = anomaly["anomaly_rate"].mean() if "anomaly_rate" in anomaly.columns else np.nan
        mean_anomaly_rate_normal = normal["anomaly_rate"].mean() if "anomaly_rate" in normal.columns else np.nan
        care_earliness = anomaly.loc[anomaly["alarm_raised"] == 1, "earliness"].mean() if anomaly_events else np.nan

        care_score = np.nan
        care_cov = np.nan
        care_acc = np.nan
        care_rel = np.nan
        care_ear = np.nan

        if isinstance(stage1_json, dict):
            care_block = None

            if "care_scores" in stage1_json and isinstance(stage1_json["care_scores"], dict):
                care_block = stage1_json["care_scores"].get(str(farm))
            elif farm in stage1_json and isinstance(stage1_json[farm], dict):
                care_block = stage1_json[farm]

            if isinstance(care_block, dict):
                care_score = care_block.get("CARE", care_block.get("care_score", np.nan))
                care_cov = care_block.get("coverage", care_block.get("cov", np.nan))
                care_acc = care_block.get("accuracy", care_block.get("acc", np.nan))
                care_rel = care_block.get("reliability", care_block.get("rel", np.nan))
                care_ear = care_block.get("earliness", care_block.get("ear", np.nan))

        rows.append({
            "farm": farm,
            "anomaly_events": anomaly_events,
            "normal_events": normal_events,
            "detected_events": detected_events,
            "false_alarm_events": false_alarm_events,
            "stage1_detection_rate": stage1_detection_rate,
            "stage1_false_alarm_rate": stage1_false_alarm_rate,
            "mean_anomaly_rate_fault": mean_anomaly_rate_fault,
            "mean_anomaly_rate_normal": mean_anomaly_rate_normal,
            "care_earliness_detected_only": care_earliness,
            "care_score": care_score,
            "care_coverage": care_cov,
            "care_accuracy": care_acc,
            "care_reliability": care_rel,
            "care_earliness": care_ear,
        })

    return pd.DataFrame(rows).sort_values("farm").reset_index(drop=True)


# ============================================================================
# STAGE 2 SUMMARY
# Actual columns:
# ['event_id', 'farm', 'fault_type', 'pred_type', 'label_int', 'is_fault',
#  'detected', 'detected_window', 'horizon_days', 'mean_confidence',
#  'uncertain_frac', 'n_train_events', 'n_classes_train',
#  'n_selected_cols', 'sensor_groups']
# ============================================================================

def build_stage2_summary(stage2_df: pd.DataFrame):
    if stage2_df is None or stage2_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = stage2_df.copy()
    df = ensure_numeric(
        df,
        [
            "is_fault", "detected", "detected_window", "horizon_days",
            "mean_confidence", "uncertain_frac", "n_train_events",
            "n_classes_train", "n_selected_cols"
        ]
    )

    df["is_fault"] = df["is_fault"].fillna(0).astype(int)
    df["detected"] = df["detected"].fillna(0).astype(int)
    df["detected_window"] = df["detected_window"].fillna(0).astype(int)

    # only anomaly events count for classification performance
    fault_df = df[df["is_fault"] == 1].copy()

    if fault_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    fault_df["correct_class"] = (fault_df["fault_type"].astype(str) == fault_df["pred_type"].astype(str)).astype(int)

    farm_rows = []
    for farm, g in fault_df.groupby("farm"):
        total_faults = len(g)
        detected_5 = int(g["detected"].sum())
        detected_window = int(g["detected_window"].sum())
        correctly_classified = int(g["correct_class"].sum())

        farm_rows.append({
            "farm": farm,
            "stage2_fault_events": total_faults,
            "stage2_detected_5consec": detected_5,
            "stage2_detected_window": detected_window,
            "stage2_detection_rate_5consec": detected_5 / total_faults if total_faults else np.nan,
            "stage2_detection_rate_window": detected_window / total_faults if total_faults else np.nan,
            "stage2_classification_accuracy": correctly_classified / total_faults if total_faults else np.nan,
            "stage2_mean_horizon_days": g.loc[g["detected"] == 1, "horizon_days"].mean(),
            "stage2_mean_confidence": g["mean_confidence"].mean(),
            "stage2_mean_uncertain_frac": g["uncertain_frac"].mean(),
            "stage2_mean_selected_cols": g["n_selected_cols"].mean(),
            "stage2_mean_train_events": g["n_train_events"].mean(),
            "stage2_mean_train_classes": g["n_classes_train"].mean(),
        })

    fault_rows = []
    for (farm, fault_type), g in fault_df.groupby(["farm", "fault_type"]):
        total = len(g)
        det = int(g["detected"].sum())
        det_win = int(g["detected_window"].sum())
        cls = int(g["correct_class"].sum())

        fault_rows.append({
            "farm": farm,
            "fault_type": fault_type,
            "total_events": total,
            "detected_5consec": det,
            "detected_window": det_win,
            "correct_classifications": cls,
            "detection_rate_5consec": det / total if total else np.nan,
            "detection_rate_window": det_win / total if total else np.nan,
            "classification_accuracy": cls / total if total else np.nan,
            "avg_horizon_days": g.loc[g["detected"] == 1, "horizon_days"].mean(),
            "avg_confidence": g["mean_confidence"].mean(),
        })

    farm_summary = pd.DataFrame(farm_rows).sort_values("farm").reset_index(drop=True)
    fault_summary = pd.DataFrame(fault_rows).sort_values(["farm", "fault_type"]).reset_index(drop=True)
    return farm_summary, fault_summary


# ============================================================================
# STAGE 3 SUMMARY
# ============================================================================

def build_stage3_summary(stage3_json) -> pd.DataFrame:
    if stage3_json is None:
        return pd.DataFrame()

    rows = []

    # flexible parsing for different JSON layouts
    if isinstance(stage3_json, dict):
        # expected keys may be "results", "directions", or direct direction keys
        candidate = None

        for k in ["results", "directions", "transfers"]:
            if k in stage3_json and isinstance(stage3_json[k], dict):
                candidate = stage3_json[k]
                break

        if candidate is None:
            candidate = stage3_json

        for direction, block in candidate.items():
            if not isinstance(block, dict):
                continue

            comparable_detected = (
                block.get("comparable_detected")
                or block.get("detected_comparable")
                or block.get("comparable_hits")
                or 0
            )
            comparable_total = (
                block.get("comparable_total")
                or block.get("n_comparable")
                or block.get("comparable_events")
                or 0
            )
            all_detected = (
                block.get("all_detected")
                or block.get("detected_all")
                or block.get("all_hits")
                or 0
            )
            all_total = (
                block.get("all_total")
                or block.get("n_all")
                or block.get("all_events")
                or 0
            )

            comparable_rate = block.get("comparable_rate", np.nan)
            all_rate = block.get("all_rate", np.nan)
            avg_horizon = block.get("avg_horizon_days", block.get("avg_horizon", np.nan))

            if pd.isna(comparable_rate) and comparable_total:
                comparable_rate = comparable_detected / comparable_total
            if pd.isna(all_rate) and all_total:
                all_rate = all_detected / all_total

            rows.append({
                "direction": direction,
                "comparable_detected": comparable_detected,
                "comparable_total": comparable_total,
                "comparable_rate": comparable_rate,
                "all_detected": all_detected,
                "all_total": all_total,
                "all_rate": all_rate,
                "avg_horizon_days": avg_horizon,
            })

    return pd.DataFrame(rows)


# ============================================================================
# FARM-LEVEL MASTER SUMMARY
# ============================================================================

def build_farm_level_summary(stage1_df: pd.DataFrame | None, stage2_df: pd.DataFrame | None) -> pd.DataFrame:
    farms = set()

    if stage1_df is not None and not stage1_df.empty and "farm" in stage1_df.columns:
        farms.update(stage1_df["farm"].dropna().astype(str).unique())

    if stage2_df is not None and not stage2_df.empty and "farm" in stage2_df.columns:
        farms.update(stage2_df["farm"].dropna().astype(str).unique())

    return pd.DataFrame({"farm": sorted(farms)})


def merge_master_summary(farm_ops, stage1_summary, stage2_summary):
    out = farm_ops.copy()

    if not stage1_summary.empty and "farm" in stage1_summary.columns:
        out = out.merge(stage1_summary, on="farm", how="left")

    if not stage2_summary.empty and "farm" in stage2_summary.columns:
        out = out.merge(stage2_summary, on="farm", how="left")

    return out.sort_values("farm").reset_index(drop=True)


# ============================================================================
# PLOTS
# ============================================================================

def plot_stage1_care(stage1_summary: pd.DataFrame):
    if stage1_summary.empty:
        return

    use = stage1_summary.dropna(subset=["care_score"]).copy()
    if use.empty:
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(use["farm"], use["care_score"])
    ax.set_title("Stage 1 CARE Score by Farm")
    ax.set_xlabel("Farm")
    ax.set_ylabel("CARE score")
    ax.set_ylim(0, max(1.0, use["care_score"].max() * 1.15))
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage1_care_score_by_farm.png", dpi=300)
    plt.close(fig)


def plot_stage1_detection_false_alarm(stage1_summary: pd.DataFrame):
    if stage1_summary.empty:
        return

    use = stage1_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, use["stage1_detection_rate"].fillna(0), width=w, label="Detection rate")
    ax.bar(x + w/2, use["stage1_false_alarm_rate"].fillna(0), width=w, label="False alarm rate")
    ax.set_xticks(x)
    ax.set_xticklabels(use["farm"])
    ax.set_ylim(0, 1)
    ax.set_title("Stage 1 Detection and False Alarm Rates")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage1_detection_false_alarm_rates.png", dpi=300)
    plt.close(fig)


def plot_stage2_detection(stage2_farm_summary: pd.DataFrame):
    if stage2_farm_summary.empty:
        return

    use = stage2_farm_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, use["stage2_detection_rate_5consec"].fillna(0), width=w, label="5-consecutive")
    ax.bar(x + w/2, use["stage2_classification_accuracy"].fillna(0), width=w, label="Classification accuracy")
    ax.set_xticks(x)
    ax.set_xticklabels(use["farm"])
    ax.set_ylim(0, 1)
    ax.set_title("Stage 2 Detection and Classification Accuracy")
    ax.set_xlabel("Farm")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage2_detection_classification_by_farm.png", dpi=300)
    plt.close(fig)


def plot_stage2_horizon(stage2_fault_summary: pd.DataFrame):
    if stage2_fault_summary.empty:
        return

    use = stage2_fault_summary.dropna(subset=["avg_horizon_days"]).copy()
    if use.empty:
        return

    use["farm_fault"] = use["farm"].astype(str) + " — " + use["fault_type"].astype(str)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(use["farm_fault"], use["avg_horizon_days"])
    ax.set_title("Average Warning Horizon by Farm and Fault Type")
    ax.set_xlabel("Farm / Fault type")
    ax.set_ylabel("Average horizon (days)")
    ax.tick_params(axis="x", rotation=75)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "stage2_horizon_by_farm_fault.png", dpi=300)
    plt.close(fig)


def plot_cross_farm(stage3_summary: pd.DataFrame):
    if stage3_summary.empty:
        return

    use = stage3_summary.copy()

    x = np.arange(len(use))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, use["comparable_rate"].fillna(0), width=w, label="Comparable faults")
    ax.bar(x + w/2, use["all_rate"].fillna(0), width=w, label="All faults")
    ax.set_xticks(x)
    ax.set_xticklabels(use["direction"], rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title("Cross-Farm Transfer Detection Rates")
    ax.set_xlabel("Transfer direction")
    ax.set_ylabel("Rate")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "cross_farm_detection_rates.png", dpi=300)
    plt.close(fig)


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("SCRIPT 4 — OPERATIONAL ANALYSIS (FINAL CORRECTED VERSION)")
    print("=" * 80)
    print(f"Base directory: {BASE_DIR}")
    print()

    print("Loading Stage 1 outputs...")
    stage1_df, stage1_json = load_stage1()
    if stage1_df is None:
        print("  Stage 1 CSV not found.")
    else:
        print(f"  Stage 1 rows: {len(stage1_df)}")
        print(f"  Stage 1 columns: {list(stage1_df.columns)}")

    print()
    print("Loading Stage 2 outputs...")
    stage2_df, stage2_json = load_stage2()
    if stage2_df is None:
        print("  Stage 2 CSV not found.")
    else:
        print(f"  Stage 2 rows: {len(stage2_df)}")
        print(f"  Stage 2 columns: {list(stage2_df.columns)}")

    print()
    print("Loading Stage 3 outputs...")
    stage3_json = load_stage3()
    if stage3_json is None:
        print("  Stage 3 JSON not found.")
    else:
        print("  Stage 3 JSON loaded.")

    print()
    print("Building summaries...")
    farm_ops_summary = build_farm_level_summary(stage1_df, stage2_df)
    stage1_summary = build_stage1_summary(stage1_df, stage1_json) if stage1_df is not None else pd.DataFrame()
    stage2_farm_summary, stage2_fault_summary = build_stage2_summary(stage2_df) if stage2_df is not None else (pd.DataFrame(), pd.DataFrame())
    stage3_summary = build_stage3_summary(stage3_json)

    master_summary = merge_master_summary(farm_ops_summary, stage1_summary, stage2_farm_summary)

    print(f"  farm_ops_summary columns: {list(farm_ops_summary.columns)}")
    print(f"  stage1_summary columns: {list(stage1_summary.columns)}")
    print(f"  stage2_farm_summary columns: {list(stage2_farm_summary.columns)}")
    print(f"  stage3_summary columns: {list(stage3_summary.columns)}")
    print()

    print("Saving reports...")
    if not master_summary.empty:
        master_summary.to_csv(REPORTS_DIR / "master_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'master_farm_summary.csv'}")

    if not stage1_summary.empty:
        stage1_summary.to_csv(REPORTS_DIR / "stage1_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage1_summary.csv'}")

    if not stage2_farm_summary.empty:
        stage2_farm_summary.to_csv(REPORTS_DIR / "stage2_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage2_farm_summary.csv'}")

    if not stage2_fault_summary.empty:
        stage2_fault_summary.to_csv(REPORTS_DIR / "stage2_fault_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'stage2_fault_summary.csv'}")

    if not stage3_summary.empty:
        stage3_summary.to_csv(REPORTS_DIR / "cross_farm_summary.csv", index=False)
        print(f"  Saved: {REPORTS_DIR / 'cross_farm_summary.csv'}")

    manifest = {
        "timestamp": datetime.now().isoformat(),
        "script": "script_4_operational_analysis_final_corrected",
        "base_dir": str(BASE_DIR),
        "inputs": {
            "stage1_event_csv": str(STAGE1_EVENT_CSV),
            "stage1_results_json": str(STAGE1_RESULTS_JSON),
            "stage2_results_csv": str(STAGE2_RESULTS_CSV),
            "stage2_results_json": str(STAGE2_RESULTS_JSON),
            "stage3_results_json": str(STAGE3_RESULTS_JSON),
        },
        "outputs": {
            "output_dir": str(OUTPUT_DIR),
            "reports_dir": str(REPORTS_DIR),
            "plots_dir": str(PLOTS_DIR),
        },
        "row_counts": {
            "stage1_rows": 0 if stage1_df is None else int(len(stage1_df)),
            "stage2_rows": 0 if stage2_df is None else int(len(stage2_df)),
            "stage1_summary_rows": int(len(stage1_summary)),
            "stage2_farm_summary_rows": int(len(stage2_farm_summary)),
            "stage2_fault_summary_rows": int(len(stage2_fault_summary)),
            "stage3_summary_rows": int(len(stage3_summary)),
        },
    }

    with open(OUTPUT_DIR / "operational_analysis_manifest.json", "w", encoding="utf-8") as f:
        json.dump(json_safe(manifest), f, indent=2)
    print(f"  Saved: {OUTPUT_DIR / 'operational_analysis_manifest.json'}")

    print()
    print("Generating plots...")
    plot_stage1_care(stage1_summary)
    plot_stage1_detection_false_alarm(stage1_summary)
    plot_stage2_detection(stage2_farm_summary)
    plot_stage2_horizon(stage2_fault_summary)
    plot_cross_farm(stage3_summary)
    print(f"  Plots saved to: {PLOTS_DIR}")

    print()
    print("=" * 80)
    print("SCRIPT 4 COMPLETE")
    print("=" * 80)
    print(f"Outputs -> {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

SCRIPT 4 — OPERATIONAL ANALYSIS (FINAL CORRECTED VERSION)
Base directory: D:\files\project\CARE_To_Compare

Loading Stage 1 outputs...
  Stage 1 rows: 95
  Stage 1 columns: ['event_id', 'farm', 'label', 'description', 'alarm_raised', 'alarm_source', 'max_criticality', 'crit_global', 'crit_subsystem', 'anomaly_rate', 'earliness', 'top_subsystem', 'n_pred_rows']

Loading Stage 2 outputs...
  Stage 2 rows: 95
  Stage 2 columns: ['event_id', 'farm', 'fault_type', 'pred_type', 'label_int', 'is_fault', 'detected', 'detected_window', 'horizon_days', 'mean_confidence', 'uncertain_frac', 'n_train_events', 'n_classes_train', 'n_selected_cols', 'sensor_groups']

Loading Stage 3 outputs...
  Stage 3 JSON loaded.

Building summaries...
  farm_ops_summary columns: ['farm']
  stage1_summary columns: ['farm', 'anomaly_events', 'normal_events', 'detected_events', 'false_alarm_events', 'stage1_detection_rate', 'stage1_false_alarm_rate', 'mean_anomaly_rate_fault', 'mean_anomaly_rate_normal', 'care_earlin

In [2]:
from __future__ import annotations

import json
import re
import time
from datetime import timedelta
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# =============================================================================
# UNIFIED SCRIPT 5 — FIGURES
# =============================================================================
# Key rule:
#   ALL DAE / Stage 1 results must come from:
#       - results_stage1_clean.json
#       - event_results_stage1_clean.csv
#       - care_score_summary_clean.csv
#
# No hardcoded Stage 1 DAE results are allowed.
#
# Hardcoded values are kept ONLY for:
#   - Isolation Forest benchmark comparison
#   - published CARE baseline
#   - conceptual / illustrative dissertation figures
# =============================================================================

# =============================================================================
# CONFIG
# =============================================================================

BASE_DIR = Path(r"D:\files\project\CARE_To_Compare")

# Stage 1 outputs (AUTHORITATIVE SOURCE FOR DAE)
STAGE1_EVENT_CSV = BASE_DIR / "stage1_clean_outputs" / "event_results_stage1_clean.csv"
STAGE1_JSON = BASE_DIR / "stage1_clean_outputs" / "results_stage1_clean.json"
STAGE1_SUMMARY_CSV = BASE_DIR / "stage1_clean_outputs" / "reports" / "care_score_summary_clean.csv"

# Stage 2 outputs
STAGE2_CSV = BASE_DIR / "fault_classification_outputs" / "fault_classification_results.csv"

# Stage 3 outputs
STAGE3_JSON = BASE_DIR / "cross_farm_outputs" / "cross_farm_results.json"
STAGE3_ALT_JSON = BASE_DIR / "cross_farm_outputs" / "results_cross_farm.json"

# Operational analysis outputs
OPS_DIR = BASE_DIR / "operational_analysis_outputs"
OPS_JSON = OPS_DIR / "operational_analysis.json"
OPS_SUMMARY_CSV = OPS_DIR / "reports" / "operational_summary.csv"
OPS_STATUS_CSV = OPS_DIR / "reports" / "status_distribution.csv"

# Metadata + raw event files
FARM_METADATA_PATHS = {
    "A": BASE_DIR / "Wind Farm A" / "feature_description.csv",
    "B": BASE_DIR / "Wind Farm B" / "feature_description.csv",
    "C": BASE_DIR / "Wind Farm C" / "feature_description.csv",
}
FARM_DATASET_PATHS = {
    "A": BASE_DIR / "Wind Farm A" / "datasets",
    "B": BASE_DIR / "Wind Farm B" / "datasets",
    "C": BASE_DIR / "Wind Farm C" / "datasets",
}

OUTPUT_DIR = BASE_DIR / "dissertation_figures_outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"
MANIFEST_PATH = REPORTS_DIR / "figure_manifest.json"

FARMS = ["A", "B", "C"]
CARE_BASELINE = 0.660

# IF comparison values can remain benchmark/hardcoded unless you have a file for them
IF_RESULTS = {
    "A": {"care": 0.518, "coverage": 0.221, "accuracy": 0.988, "reliability": 0.312, "earliness": 0.081},
    "B": {"care": 0.611, "coverage": 0.528, "accuracy": 0.854, "reliability": 0.600, "earliness": 0.218},
    "C": {"care": 0.733, "coverage": 0.402, "accuracy": 0.987, "reliability": 0.545, "earliness": 0.741},
}

STATUS_COLORS = {
    0: "#2E7D32",
    1: "#F9A825",
    2: "#81C784",
    3: "#1565C0",
    4: "#C62828",
}
STATUS_LABELS = {
    0: "Normal generation (status 0)",
    1: "Curtailed operation (status 1)",
    2: "Standby — healthy (status 2)",
    3: "Service / maintenance (status 3)",
    4: "Fault-active (status 4+5)",
}

PREFERRED_EXAMPLES = {
    "A": {"fault": ["E72", "E51", "E45", "E84", "E68", "E40"], "healthy": ["E69", "E25", "E13", "E24"]},
    "B": {"fault": ["E34", "E7", "E19", "E27", "E53", "E77"], "healthy": ["E52", "E83", "E21", "E23"]},
    "C": {"fault": ["E11", "E16", "E55", "E66", "E70", "E81", "E47"], "healthy": ["E80", "E8", "E85", "E6"]},
}

FAULT_GROUP_RULES = {
    "gearbox": ["gearbox", "oil", "bearing", "planetary", "lubric", "main shaft"],
    "gearbox_oil": ["gearbox", "oil", "lubric", "pressure", "temperature"],
    "generator_bearing": ["generator", "bearing", "drive-end", "non-drive", "nde", "de"],
    "generator_stator": ["stator", "generator", "winding", "phase", "temperature"],
    "transformer": ["transformer", "winding", "insulation", "phase", "electrical", "voltage", "current"],
    "hydraulic": ["hydraulic", "pressure", "pump", "oil", "brake", "azimuth"],
    "rotor_bearing": ["rotor bearing", "main bearing", "bearing", "shaft"],
    "pitch_electrical": ["pitch", "battery", "capacitor", "dc-link", "motor", "voltage", "electrical"],
    "pitch_mechanical": ["pitch", "bearing", "gearbox", "mechanical", "angle", "position"],
    "converter": ["converter", "igbt", "dc-link", "torque", "current", "voltage"],
    "communication": ["communication", "beckhoff", "network", "controller", "fieldbus", "ethercat"],
    "cooling_water": ["cooling", "water", "stator cooler", "flow meter", "pump"],
    "yaw": ["yaw", "drive", "motor", "angle", "position"],
    "electrical_internal": ["electrical", "voltage", "current", "phase", "internal"],
    "rotor_brake": ["brake", "rotor", "hydraulic", "pressure", "temperature"],
    "normal": [],
}
GENERIC_POWER_WORDS = [
    "total active power", "total reactive power", "active power", "reactive power", "power ", "energy ", "kwh", "kvarh"
]

FIGURE_LOG: List[dict] = []

# =============================================================================
# GENERIC HELPERS
# =============================================================================


def ensure_dirs() -> None:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def log_figure(name: str, status: str, reason: str = "", path: Optional[Path] = None) -> None:
    FIGURE_LOG.append({
        "figure": name,
        "status": status,
        "reason": reason,
        "path": str(path) if path else "",
    })


def safe_plot(name: str):
    def decorator(func):
        def wrapper(*args, **kwargs):
            try:
                result = func(*args, **kwargs)
                if result is False:
                    log_figure(name, "skipped", "Function returned False")
                return result
            except Exception as e:
                print(f"  Skipping {name} — {e}")
                log_figure(name, "error", str(e))
                return False
        return wrapper
    return decorator


def safe_read_csv(path: Path, **kwargs) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    for sep, enc in [(";", "utf-8"), (";", "latin1"), (",", "utf-8"), (",", "latin1")]:
        try:
            df = pd.read_csv(path, sep=sep, encoding=enc, **kwargs)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except Exception:
            continue
    return None


def load_json(path: Path) -> Optional[dict]:
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def save_fig(fig, filename: str, label: Optional[str] = None) -> None:
    out = PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {out}")
    log_figure(label or filename, "saved", path=out)


def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    return obj


def norm_event_id(x) -> str:
    m = re.search(r"(\d+)", str(x).strip().upper())
    return f"E{int(m.group(1))}" if m else str(x)


def norm_farm(x) -> str:
    m = re.search(r"([ABC])", str(x).strip().upper())
    return m.group(1) if m else str(x)


def clean_unit_text(unit) -> str:
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return ""
    u = str(unit).strip()
    for bad, good in [("Â°C", "°C"), ("degC", "°C"), ("DegC", "°C"), ("Celsius", "°C")]:
        u = u.replace(bad, good)
    return u


def get_ops(farm: str, col: str, ops_summary: Optional[pd.DataFrame], default: float = 0.0) -> float:
    if ops_summary is None or ops_summary.empty or col not in ops_summary.columns:
        return default
    row = ops_summary[ops_summary["farm"] == farm]
    if row.empty:
        return default
    return float(row[col].iloc[0])


def has_cols(df: Optional[pd.DataFrame], cols: set) -> bool:
    return df is not None and not df.empty and cols.issubset(df.columns)


# =============================================================================
# DAE RESULT HELPERS
# =============================================================================


def build_dae_result_map(summary_df: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    required = {
        "farm",
        "care_score",
        "care_coverage",
        "care_accuracy",
        "care_reliability",
        "care_earliness",
    }
    if not has_cols(summary_df, required):
        raise ValueError(
            "Stage 1 summary dataframe missing required columns for DAE result extraction."
        )

    out: Dict[str, Dict[str, float]] = {}
    for _, row in summary_df.iterrows():
        farm = norm_farm(row["farm"])
        out[farm] = {
            "care": float(row["care_score"]),
            "coverage": float(row["care_coverage"]),
            "accuracy": float(row["care_accuracy"]),
            "reliability": float(row["care_reliability"]),
            "earliness": float(row["care_earliness"]),
        }

    missing = [f for f in FARMS if f not in out]
    if missing:
        raise ValueError(f"Missing DAE Stage 1 results for farms: {missing}")

    return out


def build_stage1_event_stats(stage1_df: pd.DataFrame) -> Dict[str, Dict[str, int]]:
    """
    Derive Stage 1 event counts directly from Stage 1 event_results_stage1_clean.csv.

    det:
        number of fault events with alarm_raised == 1
        / total fault events

    FA:
        number of normal events with alarm_raised == 1
        / total normal events
    """
    required = {"farm", "is_fault", "alarm_raised"}
    if not has_cols(stage1_df, required):
        raise ValueError("Stage 1 event dataframe missing required columns for derived event stats.")

    out: Dict[str, Dict[str, int]] = {}
    for farm in FARMS:
        sub = stage1_df[stage1_df["farm"] == farm].copy()
        if sub.empty:
            raise ValueError(f"No Stage 1 rows found for farm {farm}")

        fault = sub[sub["is_fault"] == 1]
        normal = sub[sub["is_fault"] == 0]

        det_num = int(pd.to_numeric(fault["alarm_raised"], errors="coerce").fillna(0).astype(int).sum()) if not fault.empty else 0
        det_den = int(len(fault))

        fa_num = int(pd.to_numeric(normal["alarm_raised"], errors="coerce").fillna(0).astype(int).sum()) if not normal.empty else 0
        fa_den = int(len(normal))

        out[farm] = {
            "det_num": det_num,
            "det_den": det_den,
            "fa_num": fa_num,
            "fa_den": fa_den,
        }

    return out


# =============================================================================
# LOADING HELPERS
# =============================================================================


def load_stage1_event_df() -> pd.DataFrame:
    if not STAGE1_EVENT_CSV.exists():
        raise FileNotFoundError(f"Required Stage 1 event CSV not found: {STAGE1_EVENT_CSV}")

    print(f"  Stage 1 CSV : {STAGE1_EVENT_CSV}")
    df = pd.read_csv(STAGE1_EVENT_CSV)
    df.columns = [str(c).strip() for c in df.columns]

    if "event_id" in df.columns:
        df["event_id"] = df["event_id"].map(norm_event_id)
    if "farm" in df.columns:
        df["farm"] = df["farm"].map(norm_farm)
    if "label" in df.columns:
        df["label"] = df["label"].astype(str).str.lower()
        df["is_fault"] = (df["label"] != "normal").astype(int)
    elif "is_fault" not in df.columns:
        df["is_fault"] = 0

    for c in ["alarm_raised", "anomaly_rate", "earliness", "max_criticality", "crit_global", "crit_subsystem", "n_pred_rows"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def load_stage1_json() -> dict:
    j = load_json(STAGE1_JSON)
    if j is None:
        raise FileNotFoundError(f"Required Stage 1 JSON not found or unreadable: {STAGE1_JSON}")
    print(f"  Stage 1 JSON : {STAGE1_JSON}")
    return j


def load_stage1_summary() -> pd.DataFrame:
    if not STAGE1_SUMMARY_CSV.exists():
        raise FileNotFoundError(f"Required Stage 1 summary CSV not found: {STAGE1_SUMMARY_CSV}")

    df = pd.read_csv(STAGE1_SUMMARY_CSV)
    df.columns = [str(c).strip() for c in df.columns]

    if "farm" in df.columns:
        df["farm"] = df["farm"].map(norm_farm)

    rename = {}
    for c in df.columns:
        cl = c.lower().strip()
        if cl == "care":
            rename[c] = "care_score"
        elif cl == "coverage":
            rename[c] = "care_coverage"
        elif cl == "accuracy":
            rename[c] = "care_accuracy"
        elif cl == "reliability":
            rename[c] = "care_reliability"
        elif cl == "earliness":
            rename[c] = "care_earliness"

    df = df.rename(columns=rename)

    required = {"farm", "care_score", "care_coverage", "care_accuracy", "care_reliability", "care_earliness"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"Stage 1 summary CSV is missing required columns. Found: {list(df.columns)}"
        )

    print(f"  Stage 1 summary CSV : {STAGE1_SUMMARY_CSV}")
    return df


def load_stage2_df() -> pd.DataFrame:
    if STAGE2_CSV.exists():
        print(f"  Stage 2 CSV : {STAGE2_CSV}")
        df = pd.read_csv(STAGE2_CSV)
        df.columns = [str(c).strip() for c in df.columns]
        if "event_id" in df.columns:
            df["event_id"] = df["event_id"].map(norm_event_id)
        if "farm" in df.columns:
            df["farm"] = df["farm"].map(norm_farm)
        for c in ["detected", "detected_window", "is_fault", "horizon_days", "mean_confidence", "uncertain_frac", "n_selected_cols"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        for c in ["fault_type", "pred_type"]:
            if c in df.columns:
                df[c] = df[c].astype(str).str.strip().str.lower()
        return df
    print(f"  Stage 2 CSV not found: {STAGE2_CSV}")
    return pd.DataFrame()


def load_stage3_json() -> Optional[dict]:
    for p in [STAGE3_JSON, STAGE3_ALT_JSON]:
        j = load_json(p)
        if j:
            print(f"  Stage 3 JSON: {p}")
            return j
    print("  Stage 3 JSON not found")
    return None


def load_operational_data() -> Tuple[Optional[dict], Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    ops_json = load_json(OPS_JSON)
    ops_summary = None
    ops_status = None

    if OPS_SUMMARY_CSV.exists():
        ops_summary = pd.read_csv(OPS_SUMMARY_CSV)
        ops_summary.columns = [str(c).strip() for c in ops_summary.columns]
        if "farm" in ops_summary.columns:
            ops_summary["farm"] = ops_summary["farm"].astype(str).str.strip().str.upper()

    if OPS_STATUS_CSV.exists():
        ops_status = pd.read_csv(OPS_STATUS_CSV)
        ops_status.columns = [str(c).strip() for c in ops_status.columns]
        if "farm" in ops_status.columns:
            ops_status["farm"] = ops_status["farm"].astype(str).str.strip().str.upper()

    if ops_summary is None and ops_json:
        metrics = ops_json.get("metrics", {})
        rows = []
        for farm, m in metrics.items():
            if isinstance(m, dict):
                rows.append({"farm": str(farm).strip().upper(), **m})
        if rows:
            ops_summary = pd.DataFrame(rows)

    return ops_json, ops_summary, ops_status


# =============================================================================
# SENSOR / RAW EVENT HELPERS
# =============================================================================


def build_sensor_metadata_map(farm: str) -> Dict[str, dict]:
    path = FARM_METADATA_PATHS.get(farm)
    if path is None or not path.exists():
        return {}
    df = safe_read_csv(path)
    if df is None or df.empty:
        return {}
    lower_cols = {str(c).lower().strip(): c for c in df.columns}
    sensor_col = lower_cols.get("sensor_name")
    desc_col = lower_cols.get("description")
    unit_col = lower_cols.get("unit")
    angle_col = lower_cols.get("is_angle")
    counter_col = lower_cols.get("is_counter")
    if sensor_col is None:
        return {}
    out = {}
    for _, row in df.iterrows():
        raw = str(row.get(sensor_col, "")).strip()
        if not raw:
            continue
        desc = str(row.get(desc_col, "")).strip() if desc_col else ""
        unit = clean_unit_text(row.get(unit_col, "")) if unit_col else ""
        out[raw] = {
            "label": f"{desc} [{unit}]" if unit else (desc if desc else raw),
            "description": desc,
            "unit": unit,
            "is_angle": bool(row.get(angle_col, False)) if angle_col else False,
            "is_counter": bool(row.get(counter_col, False)) if counter_col else False,
        }
    return out


def split_sensor_stat(col_name: str) -> Tuple[str, Optional[str]]:
    m = re.match(r"^(.*)_(avg|max|min|std)$", str(col_name).strip(), re.IGNORECASE)
    if m:
        return m.group(1), m.group(2).lower()
    return str(col_name).strip(), None


def pretty_sensor_name(col: str, farm: str, meta: Dict) -> str:
    base, stat = split_sensor_stat(col)
    md = meta.get(base, {})
    lbl = md.get("label", base.replace("_", " "))
    sl = {"avg": "mean", "max": "max", "min": "min", "std": "std"}.get(stat or "", stat or "")
    return f"{lbl} ({sl})" if sl else lbl


def index_raw_event_files() -> Dict[Tuple[str, str], Path]:
    index = {}
    for farm, ds_path in FARM_DATASET_PATHS.items():
        if not ds_path.exists():
            continue
        for p in ds_path.glob("*.csv"):
            if p.stem.isdigit():
                index[(farm, f"E{int(p.stem)}")] = p
    print(f"  Indexed {len(index)} raw event files")
    return index


def split_train_pred(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if "train_test" not in df.columns:
        return df.copy(), df.copy()
    s = df["train_test"].astype(str).str.lower().str.strip()
    train_df = df[s == "train"].copy()
    pred_df = df[s.isin(["pred", "prediction", "predict", "test"])].copy()
    if pred_df.empty:
        pred_df = df[~s.eq("train")].copy()
    if train_df.empty:
        train_df = df.copy()
    return train_df, pred_df


def coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    meta_cols = {"train_test", "status_type_id", "status", "event_id", "id", "timestamp", "time_stamp", "time", "datetime", "date", "asset_id"}
    data = {}
    for c in df.columns:
        if str(c).lower().strip() in meta_cols:
            continue
        num = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
        if num.notna().mean() >= 0.3:
            data[c] = num
    return pd.DataFrame(data, index=df.index)


def normalise_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    mu, sd = s.mean(), s.std()
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def score_shift(t: pd.Series, p: pd.Series) -> float:
    t = pd.to_numeric(t, errors="coerce")
    p = pd.to_numeric(p, errors="coerce")
    sd = t.std()
    if pd.isna(sd) or sd == 0:
        return 0.0
    return float(abs((p.mean() - t.mean()) / sd))


def kw_score(col: str, fault: str, meta: Dict) -> int:
    base, _ = split_sensor_stat(col)
    txt = f"{base} {meta.get(base, {}).get('description', '')}".lower()
    s = sum(3 for kw in FAULT_GROUP_RULES.get(fault, []) if kw.lower() in txt)
    if any(w in txt for w in GENERIC_POWER_WORDS):
        s -= 2
    if meta.get(base, {}).get("is_counter", False):
        s -= 5
    if meta.get(base, {}).get("is_angle", False):
        s -= 2
    return s


def pick_sensors(farm: str, fault: str, train_df: pd.DataFrame, pred_df: pd.DataFrame, meta: Dict, n: int = 4) -> List[str]:
    tn = coerce_numeric(train_df)
    pn = coerce_numeric(pred_df)
    shared = [c for c in tn.columns if c in pn.columns]
    if not shared:
        return []
    scored = sorted(
        [{"col": c, "kw": kw_score(c, fault, meta), "total": score_shift(tn[c], pn[c]) * 2.5 + kw_score(c, fault, meta)} for c in shared],
        key=lambda x: x["total"],
        reverse=True,
    )
    if any(r["kw"] > 0 for r in scored):
        scored = [r for r in scored if r["kw"] > 0]
    sel, bases = [], set()
    for r in scored:
        base, _ = split_sensor_stat(r["col"])
        if base not in bases:
            sel.append(r["col"])
            bases.add(base)
        if len(sel) >= n:
            break
    return sel[:n]


def best_fault_type(stage2_df: pd.DataFrame, farm: str, event_id: str) -> str:
    if stage2_df.empty or not {"farm", "event_id", "fault_type"}.issubset(stage2_df.columns):
        return "normal"
    m = stage2_df[(stage2_df["farm"] == farm) & (stage2_df["event_id"] == event_id)]
    vals = m["fault_type"].dropna().astype(str).str.lower().tolist() if not m.empty else []
    return vals[0] if vals else "normal"


# =============================================================================
# FIGURES
# =============================================================================


@safe_plot("01_dataset_balance")
def plot_01_dataset_balance(stage1_df: pd.DataFrame):
    if not has_cols(stage1_df, {"farm", "is_fault"}):
        return False
    rows = []
    for farm, g in stage1_df.groupby("farm"):
        rows.append({"farm": farm, "anomaly": int((g["is_fault"] == 1).sum()), "normal": int((g["is_fault"] == 0).sum())})
    df = pd.DataFrame(rows).sort_values("farm")
    x = np.arange(len(df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w / 2, df["anomaly"], w, label="Anomaly events", alpha=0.88)
    b2 = ax.bar(x + w / 2, df["normal"], w, label="Normal events", alpha=0.88)
    ax.bar_label(b1, padding=3, fontsize=10)
    ax.bar_label(b2, padding=3, fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in df["farm"]])
    ax.set_ylabel("Count")
    ax.set_title("Dataset Balance by Farm")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "01_dataset_balance.png")


@safe_plot("02_stage1_care_by_farm")
def plot_02_stage1_care_by_farm(summary_df: pd.DataFrame):
    if not has_cols(summary_df, {"farm", "care_score"}):
        return False
    sdf = summary_df.sort_values("farm")
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f"Farm {f}" for f in sdf["farm"]], sdf["care_score"], alpha=0.88)
    ax.axhline(CARE_BASELINE, linestyle="--", linewidth=1.6, label=f"Published baseline = {CARE_BASELINE:.3f}")
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("CARE score")
    ax.set_title("CARE Score by Farm")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "02_stage1_care_by_farm.png")


@safe_plot("03_stage1_care_components")
def plot_03_stage1_care_components(summary_df: pd.DataFrame):
    if not has_cols(summary_df, {"farm", "care_coverage", "care_accuracy", "care_reliability", "care_earliness"}):
        return False
    sdf = summary_df.sort_values("farm")
    farms = sdf["farm"].tolist()
    comps = ["care_coverage", "care_accuracy", "care_reliability", "care_earliness"]
    labs = ["Coverage", "Accuracy", "Reliability", "Earliness"]
    x = np.arange(len(farms))
    w = 0.18
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (comp, lab) in enumerate(zip(comps, labs)):
        vals = sdf[comp].astype(float).tolist()
        bars = ax.bar(x + (i - 1.5) * w, vals, w, label=lab, alpha=0.88)
        ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("CARE Component Breakdown")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "03_stage1_care_components.png")


@safe_plot("04_stage1_benchmark_comparison")
def plot_04_stage1_benchmark_comparison(summary_df: pd.DataFrame):
    dae_map = build_dae_result_map(summary_df)

    farms = ["A", "B", "C"]
    dae_scores = [dae_map[f]["care"] for f in farms]
    if_scores = [IF_RESULTS[f]["care"] for f in farms]

    x = np.arange(len(farms))
    w = 0.32
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    b1 = ax.bar(x - w / 2, dae_scores, w, label="DAE", alpha=0.9)
    b2 = ax.bar(x + w / 2, if_scores, w, label="Isolation Forest", alpha=0.9)
    ax.axhline(CARE_BASELINE, linestyle="--", linewidth=1.5, label=f"Published baseline = {CARE_BASELINE:.3f}")
    ax.bar_label(b1, fmt="%.3f", padding=2, fontsize=9)
    ax.bar_label(b2, fmt="%.3f", padding=2, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("CARE Score")
    ax.set_title("DAE vs Isolation Forest - CARE Score")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    ax2 = axes[1]
    comps = ["coverage", "accuracy", "reliability", "earliness"]
    comp_labels = ["Coverage", "Accuracy", "Reliability", "Earliness"]
    dae_sub = [dae_map["B"][c] for c in comps]
    if_sub = [IF_RESULTS["B"][c] for c in comps]
    xi = np.arange(len(comps))
    b3 = ax2.bar(xi - w / 2, dae_sub, w, label="DAE", alpha=0.9)
    b4 = ax2.bar(xi + w / 2, if_sub, w, label="IF", alpha=0.9)
    ax2.bar_label(b3, fmt="%.3f", padding=2, fontsize=9)
    ax2.bar_label(b4, fmt="%.3f", padding=2, fontsize=9)
    ax2.set_xticks(xi)
    ax2.set_xticklabels(comp_labels)
    ax2.set_ylim(0, 1.15)
    ax2.set_ylabel("Sub-score")
    ax2.set_title("Farm B Sub-score Comparison")
    ax2.legend()
    ax2.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "04_stage1_benchmark_comparison.png")


@safe_plot("05_stage2_performance_by_farm")
def plot_05_stage2_performance_by_farm(stage2_df: pd.DataFrame):
    needed = {"farm", "is_fault", "detected"}
    if not has_cols(stage2_df, needed):
        return False
    rows = []
    for farm, g in stage2_df.groupby("farm"):
        fault = g[g["is_fault"] == 1]
        if fault.empty:
            continue
        cls_acc = np.nan
        if {"fault_type", "pred_type"}.issubset(fault.columns):
            cls_acc = (fault["fault_type"].astype(str) == fault["pred_type"].astype(str)).mean()
        rows.append({"farm": farm, "det": fault["detected"].mean(), "cls_acc": cls_acc, "n_det": int(fault["detected"].sum()), "n_tot": len(fault)})
    df = pd.DataFrame(rows).sort_values("farm")
    if df.empty:
        return False
    x = np.arange(len(df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w / 2, df["det"], w, label="Detection rate", alpha=0.88)
    b2 = ax.bar(x + w / 2, df["cls_acc"].fillna(0), w, label="Classification accuracy", alpha=0.88)
    for bar, v in zip(b1, df["det"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{v:.1%}", ha="center", fontsize=10)
    for bar, v in zip(b2, df["cls_acc"].fillna(0)):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{v:.1%}", ha="center", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in df["farm"]])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Rate")
    ax.set_title("Performance by Farm")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "05_stage2_performance_by_farm.png")


@safe_plot("06_stage2_horizon_by_farm")
def plot_06_stage2_horizon_by_farm(stage2_df: pd.DataFrame):
    needed = {"farm", "is_fault", "detected", "horizon_days"}
    if not has_cols(stage2_df, needed):
        return False
    rows = []
    for farm, g in stage2_df.groupby("farm"):
        det = g[(g["is_fault"] == 1) & (g["detected"] == 1)]
        h = det["horizon_days"].dropna()
        if h.empty:
            continue
        rows.append({"farm": farm, "avg": h.mean(), "std": h.std()})
    df = pd.DataFrame(rows).sort_values("farm")
    if df.empty:
        return False
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f"Farm {f}" for f in df["farm"]], df["avg"], alpha=0.88, yerr=df["std"].fillna(0), capsize=5)
    ax.axhline(30, linestyle="--", linewidth=1.4, alpha=0.75, label="30-day threshold")
    ax.axhline(8, linestyle=":", linewidth=1.2, alpha=0.75, label="8-day threshold")
    ax.bar_label(bars, fmt="%.1f d", padding=5, fontsize=10)
    ax.set_ylabel("Average warning horizon (days)")
    ax.set_title("Warning Horizon by Farm")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "06_stage2_horizon_by_farm.png")


@safe_plot("07_stage2_farmC_by_type")
def plot_07_stage2_farmC_by_type(stage2_df: pd.DataFrame):
    if not has_cols(stage2_df, {"farm", "is_fault", "fault_type", "detected"}):
        return False
    c_df = stage2_df[(stage2_df["farm"] == "C") & (stage2_df["is_fault"] == 1)].copy()
    if c_df.empty:
        return False
    by_type = c_df.groupby("fault_type").agg(total=("detected", "size"), detected=("detected", "sum")).reset_index()
    by_type["rate"] = by_type["detected"] / by_type["total"]
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(by_type["fault_type"], by_type["rate"], alpha=0.88)
    for bar, rate, total in zip(bars, by_type["rate"], by_type["total"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{rate:.0%}\n(n={int(total)})", ha="center", fontsize=8)
    ax.set_ylim(0, 1.25)
    ax.set_ylabel("Detection rate")
    ax.set_title("Farm C Detection Rate by Fault Type")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "07_stage2_farmC_by_type.png")


@safe_plot("08_stage2_selected_columns")
def plot_08_stage2_selected_columns(stage2_df: pd.DataFrame):
    if not has_cols(stage2_df, {"farm", "n_selected_cols"}):
        return False
    g = stage2_df.groupby("farm")["n_selected_cols"].mean().reset_index().sort_values("farm")
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f"Farm {f}" for f in g["farm"]], g["n_selected_cols"], alpha=0.88)
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=10)
    ax.set_ylabel("Average selected columns")
    ax.set_title("Selected Stage 2 Feature Columns by Farm")
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "08_stage2_selected_columns.png")


@safe_plot("09_cross_farm_transfer")
def plot_09_cross_farm_transfer(stage3_json: Optional[dict]):
    if not stage3_json:
        return False
    results = stage3_json.get("results", stage3_json)
    rows = []
    for key, block in results.items():
        if not isinstance(block, dict):
            continue
        comp = block.get("comparable") or {}
        all_ = block.get("all_faults") or {}
        rows.append({
            "direction": key.replace("_to_", " → "),
            "comp_rate": float(comp.get("detection_rate", 0) or 0),
            "all_rate": float(all_.get("detection_rate", 0) or 0),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return False
    x = np.arange(len(df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    b1 = ax.bar(x - w / 2, df["comp_rate"], w, label="Comparable faults", alpha=0.88)
    b2 = ax.bar(x + w / 2, df["all_rate"], w, label="All fault types", alpha=0.88)
    ax.bar_label(b1, fmt="%.0f%%", labels=[f"{v:.0%}" for v in df["comp_rate"]], padding=2, fontsize=8)
    ax.bar_label(b2, fmt="%.0f%%", labels=[f"{v:.0%}" for v in df["all_rate"]], padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(df["direction"], rotation=20, ha="right")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Detection rate")
    ax.set_title("Cross-farm Transfer Performance")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "09_cross_farm_transfer.png")


@safe_plot("10_operational_status_distribution")
def plot_10_operational_status_distribution(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    order_keys = ["generating_pct", "standby_pct", "curtailed_pct", "service_pct", "fault_pct"]
    order_codes = [0, 2, 1, 3, 4]
    fig, ax = plt.subplots(figsize=(10, 6))
    bottoms = np.zeros(len(farms))
    for key, code in zip(order_keys, order_codes):
        if key not in ops_summary.columns:
            continue
        vals = [get_ops(f, key, ops_summary) for f in farms]
        bars = ax.bar([f"Farm {f}" for f in farms], vals, bottom=bottoms, color=STATUS_COLORS[code], label=STATUS_LABELS[code], alpha=0.9)
        for bar, val in zip(bars, vals):
            if val >= 2.0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        bottoms += np.array(vals)
    ax.set_ylabel("% of SCADA dataset")
    ax.set_title("Operational Status Distribution by Farm")
    ax.legend(loc="lower right", fontsize=9)
    ax.set_ylim(0, 108)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "10_operational_status_distribution.png")


@safe_plot("11_operational_availability")
def plot_11_operational_availability(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    healthy = [get_ops(f, "healthy_pct", ops_summary) for f in farms]
    gen = [get_ops(f, "generating_pct", ops_summary) for f in farms]
    service = [get_ops(f, "service_pct", ops_summary) for f in farms]
    fault = [get_ops(f, "fault_pct", ops_summary) for f in farms]
    x = np.arange(len(farms))
    w = 0.20
    fig, ax = plt.subplots(figsize=(11, 6))
    b1 = ax.bar(x - 1.5 * w, healthy, w, label="Technical availability", alpha=0.9)
    b2 = ax.bar(x - 0.5 * w, gen, w, label="Normal generation", alpha=0.9)
    b3 = ax.bar(x + 0.5 * w, service, w, label="Service", alpha=0.9)
    b4 = ax.bar(x + 1.5 * w, fault, w, label="Fault-active", alpha=0.9)
    for bars in [b1, b2, b3, b4]:
        ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
    ax.axhline(90, linestyle="--", linewidth=1.2, alpha=0.55, label="90% reference")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("% of total dataset hours")
    ax.set_title("Turbine Availability, Generation, and Downtime by Farm")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 115)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "11_operational_availability.png")


@safe_plot("12_operational_power_economics")
def plot_12_operational_power_economics(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    act_gwh = [get_ops(f, "annual_mwh_actual", ops_summary) / 1000 for f in farms]
    exp_gwh = [get_ops(f, "annual_mwh_expected", ops_summary) / 1000 for f in farms]
    fault_rev = [get_ops(f, "fault_revenue_lost_eur", ops_summary) / 1e6 for f in farms]
    service_rev = [get_ops(f, "service_revenue_lost_eur", ops_summary) / 1e6 for f in farms]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    x = np.arange(len(farms))
    w = 0.35
    ax = axes[0]
    b1 = ax.bar(x - w / 2, exp_gwh, w, label="Expected", alpha=0.8)
    b2 = ax.bar(x + w / 2, act_gwh, w, label="Actual", alpha=0.9)
    ax.bar_label(b1, fmt="%.1f GWh", padding=3, fontsize=8)
    ax.bar_label(b2, fmt="%.1f GWh", padding=3, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("Annual energy per turbine (GWh/yr)")
    ax.set_title("Expected vs Actual Annual Generation")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    ax2 = axes[1]
    b3 = ax2.bar(x - w / 2, fault_rev, w, label="Fault revenue loss", alpha=0.9)
    b4 = ax2.bar(x + w / 2, service_rev, w, label="Service revenue loss", alpha=0.9)
    ax2.bar_label(b3, fmt="€%.2fM", padding=3, fontsize=8)
    ax2.bar_label(b4, fmt="€%.2fM", padding=3, fontsize=8)
    ax2.set_xticks(x)
    ax2.set_xticklabels([f"Farm {f}" for f in farms])
    ax2.set_ylabel("Estimated revenue loss (€ millions)")
    ax2.set_title("Revenue Loss: Fault vs Service")
    ax2.legend(fontsize=9)
    ax2.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "12_operational_power_economics.png")


@safe_plot("13_maintenance_cost_comparison")
def plot_13_maintenance_cost_comparison():
    farms = ["A", "B", "C"]
    farm_labels = ["Farm A\n(onshore)", "Farm B\n(offshore)", "Farm C\n(offshore)"]
    plan_mid = [(8000 + 15000) / 2 / 1000] * 3
    emrg_mid = [(25000 + 45000) / 2 / 1000, (34000 + 68000) / 2 / 1000, (34000 + 68000) / 2 / 1000]
    saving = [e - p for e, p in zip(emrg_mid, plan_mid)]
    x = np.arange(len(farms))
    w = 0.26
    fig, ax = plt.subplots(figsize=(11, 6))
    for xpos, vals, lab in [(x - w, plan_mid, "Planned maintenance"), (x, emrg_mid, "Emergency mobilisation"), (x + w, saving, "Saving per event")]:
        bars = ax.bar(xpos, vals, w, label=lab, alpha=0.9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4, f"€{v:.0f}k", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(farm_labels)
    ax.set_ylabel("Vessel mobilisation cost (€ thousands, per event)")
    ax.set_title("Planned vs Emergency Maintenance Cost Comparison")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "13_maintenance_cost_comparison.png")


@safe_plot("14_stage2_horizon_by_fault_type")
def plot_14_stage2_horizon_by_fault_type(stage2_df: pd.DataFrame):
    needed = {"farm", "is_fault", "detected", "horizon_days", "fault_type"}
    if not has_cols(stage2_df, needed):
        return False
    det = stage2_df[(stage2_df["is_fault"] == 1) & (stage2_df["detected"] == 1) & (stage2_df["horizon_days"].notna())].copy()
    if det.empty:
        return False
    det["label"] = det["farm"] + " — " + det["fault_type"].str.replace("_", " ")
    summary = det.groupby(["farm", "fault_type", "label"])["horizon_days"].mean().reset_index().sort_values("horizon_days", ascending=False)
    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.bar(summary["label"], summary["horizon_days"], alpha=0.88)
    ax.axhline(30, linestyle="--", linewidth=1.3, alpha=0.7, label="30-day threshold")
    ax.axhline(8, linestyle=":", linewidth=1.2, alpha=0.7, label="8-day threshold")
    ax.bar_label(bars, fmt="%.1f d", padding=2, fontsize=8)
    ax.set_ylabel("Average warning horizon (days)")
    ax.set_title("Warning Horizon by Farm and Fault Type")
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=40)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "14_stage2_horizon_by_fault_type.png")


@safe_plot("15_cross_farm_transfer_with_horizon")
def plot_15_cross_farm_transfer_with_horizon(stage3_json: Optional[dict]):
    if not stage3_json:
        return False
    results = stage3_json.get("results", stage3_json)
    dir_order = ["A_to_C", "C_to_A", "C_to_B", "B_to_A", "B_to_C"]
    dir_labels = {"A_to_C": "Farm A → Farm C", "C_to_A": "Farm C → Farm A", "C_to_B": "Farm C → Farm B", "B_to_A": "Farm B → Farm A", "B_to_C": "Farm B → Farm C"}
    rows = []
    for key in dir_order:
        block = results.get(key, {})
        if not isinstance(block, dict):
            continue
        comp = block.get("comparable") or {}
        all_ = block.get("all_faults") or {}
        rows.append({
            "label": dir_labels[key],
            "comp_rate": float(comp.get("detection_rate", 0) or 0),
            "all_rate": float(all_.get("detection_rate", 0) or 0),
            "horizon": float(comp.get("avg_horizon_days", 0) or 0),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return False
    x = np.arange(len(df))
    w = 0.32
    fig, ax1 = plt.subplots(figsize=(13, 5))
    b1 = ax1.bar(x - w / 2, df["comp_rate"], w, label="Comparable faults", alpha=0.88)
    b2 = ax1.bar(x + w / 2, df["all_rate"], w, label="All fault types", alpha=0.88)
    ax1.set_xticks(x)
    ax1.set_xticklabels(df["label"], rotation=15, ha="right", fontsize=9)
    ax1.set_ylim(0, 1.1)
    ax1.set_ylabel("Detection rate")
    ax2 = ax1.twinx()
    ax2.plot(x, df["horizon"], marker="o", linewidth=2, markersize=7, label="Avg horizon (days)")
    ax2.set_ylabel("Avg warning horizon (days)")
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=9, loc="upper right")
    ax1.set_title("Cross-farm Transfer - Detection Rates with Warning Horizon")
    ax1.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "15_cross_farm_transfer_with_horizon.png")


@safe_plot("16_actual_vs_expected_energy")
def plot_16_actual_vs_expected_energy(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    actual = [get_ops(f, "actual_mwh", ops_summary, get_ops(f, "annual_mwh_actual", ops_summary)) for f in farms]
    expected = [get_ops(f, "expected_mwh", ops_summary, get_ops(f, "annual_mwh_expected", ops_summary)) for f in farms]
    x = np.arange(len(farms))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w / 2, expected, w, label="Expected", alpha=0.8)
    b2 = ax.bar(x + w / 2, actual, w, label="Actual", alpha=0.9)
    ax.bar_label(b1, fmt="%.1f", padding=2, fontsize=8)
    ax.bar_label(b2, fmt="%.1f", padding=2, fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("Energy (MWh)")
    ax.set_title("Actual vs Expected Energy")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "16_actual_vs_expected_energy.png")


@safe_plot("17_recoverable_energy_breakdown")
def plot_17_recoverable_energy_breakdown(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    fault_sfl = [get_ops(f, "fault_shortfall_mwh", ops_summary) / 1000 for f in farms]
    service_sfl = [get_ops(f, "service_shortfall_mwh", ops_summary) / 1000 for f in farms]
    fig, ax = plt.subplots(figsize=(9, 6))
    b1 = ax.bar([f"Farm {f}" for f in farms], fault_sfl, label="Fault-active shortfall", alpha=0.9)
    b2 = ax.bar([f"Farm {f}" for f in farms], service_sfl, bottom=fault_sfl, label="Service-mode shortfall", alpha=0.9)
    totals = [f + s for f, s in zip(fault_sfl, service_sfl)]
    for xi, tot in enumerate(totals):
        ax.text(xi, tot + 1.5, f"Total: {tot:.0f} GWh", ha="center", fontsize=9, fontweight="bold")
    ax.set_ylabel("Generation shortfall (GWh)")
    ax.set_title("Generation Shortfall by Downtime Category")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "17_recoverable_energy_breakdown.png")


@safe_plot("A_care_score_vs_baseline")
def plot_A_care_score_vs_baseline(summary_df: pd.DataFrame, stage1_df: pd.DataFrame):
    if not has_cols(summary_df, {"farm", "care_score"}):
        return False

    sdf = summary_df.sort_values("farm").copy()
    farms = ["A", "B", "C"]
    dae_map = {row["farm"]: float(row["care_score"]) for _, row in sdf.iterrows()}
    scores = [dae_map[f] for f in farms]

    event_stats = build_stage1_event_stats(stage1_df)

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar([f"Farm {f}" for f in farms], scores, alpha=0.88, width=0.5)
    ax.axhline(CARE_BASELINE, linestyle="--", linewidth=1.8, label=f"Published baseline = {CARE_BASELINE:.3f}")
    ax.axhline(0.500, linestyle=":", linewidth=1.2, label="Random baseline = 0.500", alpha=0.7)

    for bar, score, farm in zip(bars, scores, farms):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012, f"{score:.3f}", ha="center", fontsize=11, fontweight="bold")

        stat = event_stats[farm]
        det = f"{stat['det_num']}/{stat['det_den']}"
        fa = f"{stat['fa_num']}/{stat['fa_den']}"

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            0.03,
            f"det={det}\nFA={fa}",
            ha="center",
            fontsize=8,
            color="white",
            fontweight="bold",
        )

    ax.set_ylim(0, 0.95)
    ax.set_ylabel("CARE Score")
    ax.set_title("CARE Score by Farm - DAE NBM vs Published Baseline")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "A_care_score_vs_baseline.png")


@safe_plot("B_farmB_subsystem_contribution")
def plot_B_farmB_subsystem_contribution():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, title, detected, total in [
        (axes[0], "Global NBM\n(344 features, no subsystem grouping)", 0, 6),
        (axes[1], "Subsystem NBMs\n(5 dedicated models, transformer isolated)", 6, 6),
    ]:
        missed = total - detected
        ax.bar(["Detected", "Missed"], [detected, missed], alpha=0.88)
        ax.set_ylim(0, 7)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_ylabel("Events")
        for i, v in enumerate([detected, missed]):
            ax.text(i, v + 0.1, str(v), ha="center", va="bottom", fontsize=14, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Farm B - Subsystem NBMs Resolve High-Dimensional Dilution", fontsize=11, fontweight="bold")
    save_fig(fig, "B_farmB_subsystem_contribution.png")


@safe_plot("C_sensor_architecture_comparison")
def plot_C_sensor_architecture_comparison():
    groups = ["gearbox_bearing", "gearbox_oil", "hydraulic", "transformer", "electrical", "generator_bearing", "generator_stator", "pitch", "vibration", "rotor_bearing", "wind", "environment"]
    labels = [g.replace("_", "\n").title() for g in groups]
    counts = {
        "A": [1, 0, 1, 3, 14, 2, 3, 0, 0, 0, 1, 2],
        "B": [4, 2, 0, 9, 16, 2, 0, 6, 3, 2, 4, 2],
        "C": [16, 16, 15, 1, 53, 0, 7, 24, 4, 8, 3, 13],
    }
    x = np.arange(len(groups))
    w = 0.25
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, farm in enumerate(["A", "B", "C"]):
        ax.bar(x + (i - 1) * w, counts[farm], w, label=f"Farm {farm}", alpha=0.82)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8.5)
    ax.set_ylabel("Number of sensors")
    ax.set_title("Sensor Architecture Comparison")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    save_fig(fig, "C_sensor_architecture_comparison.png")


@safe_plot("D_operational_hours_economics")
def plot_D_operational_hours_economics(ops_summary: Optional[pd.DataFrame]):
    if ops_summary is None or ops_summary.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in ops_summary["farm"].values]
    if not farms:
        return False
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax = axes[0, 0]
    total = [get_ops(f, "total_hrs", ops_summary) for f in farms]
    ax.bar(farms, total, alpha=0.88)
    ax.set_title("Total SCADA hours")
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    ax = axes[0, 1]
    generating = [get_ops(f, "generating_hrs", ops_summary) for f in farms]
    fault = [get_ops(f, "fault_hrs", ops_summary) for f in farms]
    ax.bar(farms, generating, label="Generating", alpha=0.88)
    ax.bar(farms, fault, bottom=generating, label="Fault", alpha=0.88)
    ax.set_title("Generating vs fault hours")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    ax = axes[1, 0]
    actual = [get_ops(f, "annual_mwh_actual", ops_summary) for f in farms]
    expected = [get_ops(f, "annual_mwh_expected", ops_summary) for f in farms]
    ax.plot(farms, actual, marker="o", label="Actual")
    ax.plot(farms, expected, marker="o", label="Expected")
    ax.set_title("Annual MWh")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25, linestyle="--")

    ax = axes[1, 1]
    revenue = [get_ops(f, "total_revenue_lost_eur", ops_summary) / 1e6 for f in farms]
    bars = ax.bar(farms, revenue, alpha=0.88)
    ax.bar_label(bars, fmt="€%.2fM", padding=3, fontsize=8)
    ax.set_title("Total revenue lost")
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    fig.suptitle("Operational Hours and Economics Overview", fontsize=12, fontweight="bold")
    save_fig(fig, "D_operational_hours_economics.png")


@safe_plot("E_detection_timeline")
def plot_E_detection_timeline(stage1_df: pd.DataFrame):
    needed = {"farm", "event_id", "is_fault", "alarm_raised", "n_pred_rows"}
    if not has_cols(stage1_df, needed):
        return False

    df = stage1_df[stage1_df["is_fault"] == 1].copy()
    df["n_pred_rows"] = pd.to_numeric(df["n_pred_rows"], errors="coerce")
    df = df[df["n_pred_rows"].notna() & (df["n_pred_rows"] > 0)].copy()
    if df.empty:
        return False

    df["alarm_raised"] = pd.to_numeric(df["alarm_raised"], errors="coerce").fillna(0).astype(int)

    visible_farms = [f for f in ["A", "B", "C"] if (df["farm"] == f).any()]
    if not visible_farms:
        return False

    farm_dfs = {}
    max_rows = 0
    for farm in visible_farms:
        fdf = df[df["farm"] == farm].copy()
        fdf = fdf.sort_values(
            ["alarm_raised", "n_pred_rows", "event_id"],
            ascending=[False, False, True]
        ).reset_index(drop=True)

        farm_dfs[farm] = fdf
        max_rows = max(max_rows, len(fdf))

    bar_color = "#9AA4A6"
    marker_color = "red"

    width_ratios = [1.0 if f != "C" else 1.35 for f in visible_farms]

    fig, axes = plt.subplots(
        1,
        len(visible_farms),
        figsize=(20, max(9, max_rows * 0.42)),
        gridspec_kw={"width_ratios": width_ratios},
        sharex=False,
        sharey=False
    )

    if len(visible_farms) == 1:
        axes = [axes]

    for ax, farm in zip(axes, visible_farms):
        fdf = farm_dfs[farm].copy().iloc[::-1].reset_index(drop=True)
        y = np.arange(len(fdf))

        ax.barh(
            y,
            fdf["n_pred_rows"],
            color=bar_color,
            edgecolor="white",
            linewidth=0.8,
            height=0.72
        )

        for i, row in fdf.iterrows():
            if int(row["alarm_raised"]) == 1:
                x_det = max(1, int(row["n_pred_rows"] * 0.70))
                ax.vlines(
                    x_det,
                    i - 0.16,
                    i + 0.16,
                    color=marker_color,
                    linewidth=2.0,
                    zorder=3
                )

        n_det = int(fdf["alarm_raised"].sum())
        n_tot = len(fdf)

        ax.set_yticks(y)
        ax.set_yticklabels(fdf["event_id"], fontsize=9)

        ax.set_title(
            f"Farm {farm}\n{n_det}/{n_tot} events detected",
            fontsize=12,
            fontweight="bold",
            pad=6
        )

        ax.set_xlabel("Prediction-window length (10-minute intervals)", fontsize=10)
        ax.grid(axis="x", linestyle="--", alpha=0.25)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        xmax = int(fdf["n_pred_rows"].max() * 1.08)
        ax.set_xlim(0, xmax)

    legend_handles = [
        mpatches.Patch(color=bar_color, label="Fault event window"),
        plt.Line2D([0], [0], color=marker_color, lw=2, label="Alarm raised")
    ]

    fig.suptitle(
        "Anomaly Detection Timeline by Farm",
        fontsize=18,
        fontweight="bold",
        y=0.98
    )

    # Move legend to the right side
    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(0.91, 0.5),
        frameon=False,
        fontsize=11,
        borderaxespad=0.0
    )

    # Leave room on the right for the legend
    fig.subplots_adjust(
        top=0.88,
        right=0.88,
        left=0.06,
        bottom=0.09,
        wspace=0.10
    )

    save_fig(fig, "E_detection_timeline.png")
    return True

@safe_plot("F_anomaly_score_distribution")
def plot_F_anomaly_score_distribution(stage1_df: pd.DataFrame):
    needed = {"farm", "is_fault", "anomaly_rate"}
    if not has_cols(stage1_df, needed):
        return False
    df = stage1_df.copy()
    df["anomaly_rate"] = pd.to_numeric(df["anomaly_rate"], errors="coerce")
    df = df.dropna(subset=["anomaly_rate"])
    if df.empty:
        return False
    farms = [f for f in ["A", "B", "C"] if f in df["farm"].unique()]
    fig, axes = plt.subplots(1, len(farms), figsize=(5 * len(farms), 5), sharey=True)
    if len(farms) == 1:
        axes = [axes]
    for ax, farm in zip(axes, farms):
        fdf = df[df["farm"] == farm]
        normal = fdf[fdf["is_fault"] == 0]["anomaly_rate"].dropna()
        fault = fdf[fdf["is_fault"] == 1]["anomaly_rate"].dropna()
        data, labels = [], []
        if not normal.empty:
            data.append(normal.values)
            labels.append(f"Normal\n(n={len(normal)})")
        if not fault.empty:
            data.append(fault.values)
            labels.append(f"Fault\n(n={len(fault)})")
        if not data:
            continue
        ax.boxplot(data, patch_artist=True, notch=False, medianprops=dict(color="black", linewidth=2))
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=10)
        ax.set_title(f"Farm {farm}", fontsize=11, fontweight="bold")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("Anomaly Rate")
    fig.suptitle("Anomaly Score Distribution - Normal vs Fault Events", fontsize=11, fontweight="bold")
    save_fig(fig, "F_anomaly_score_distribution.png")


@safe_plot("G_pipeline_results_table")
def plot_G_pipeline_results_table(stage1_summary: pd.DataFrame, stage2_df: pd.DataFrame, stage3_json: Optional[dict]):
    dae_map = build_dae_result_map(stage1_summary)
    rows = []
    for farm in ["A", "B", "C"]:
        row = {"Farm": farm}
        row["Stage 1 CARE"] = dae_map[farm]["care"]
        if has_cols(stage2_df, {"farm", "is_fault", "detected"}):
            sub = stage2_df[(stage2_df["farm"] == farm) & (stage2_df["is_fault"] == 1)]
            row["Stage 2 Det. Rate"] = float(sub["detected"].mean()) if not sub.empty else np.nan
        else:
            row["Stage 2 Det. Rate"] = np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis("off")
    tbl = ax.table(
        cellText=np.round(df[["Stage 1 CARE", "Stage 2 Det. Rate"]].fillna(0).values, 3),
        rowLabels=df["Farm"].tolist(),
        colLabels=["Stage 1 CARE", "Stage 2 Det. Rate"],
        cellLoc="center",
        rowLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.2, 1.8)
    ax.set_title("Pipeline Results Summary", fontsize=12, fontweight="bold", pad=10)
    save_fig(fig, "G_pipeline_results_table.png")


@safe_plot("H_degradation_trace_farmC_E47")
def plot_H_degradation_trace_farmC_E47(raw_index: Dict, stage2_df: pd.DataFrame):
    farm = "C"
    event_id = "E47"
    fault = best_fault_type(stage2_df, farm, event_id)
    raw_path = raw_index.get((farm, event_id))
    if raw_path is None:
        return False
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False
    meta = build_sensor_metadata_map(farm)
    train_df, pred_df = split_train_pred(df)
    if pred_df.empty:
        pred_df = df
    sensors = pick_sensors(farm, fault, train_df if not train_df.empty else df, pred_df, meta, n=4)
    if not sensors:
        return False
    fig, ax = plt.subplots(figsize=(11, 5))
    for s in sensors:
        if s not in pred_df.columns:
            continue
        ax.plot(normalise_series(pred_df[s]).reset_index(drop=True), label=pretty_sensor_name(s, farm, meta)[:70], linewidth=1.2)
    ax.set_xlabel("Prediction-window timestep (10-min intervals)")
    ax.set_ylabel("Normalised sensor value (z-score)")
    ax.set_title(f"Farm {farm} Event {event_id} — Prediction Window Sensor Behaviour")
    ax.axhline(0, linewidth=0.8, linestyle="--", alpha=0.4)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
    save_fig(fig, "H_degradation_trace_farmC_E47.png")


@safe_plot("I_cross_farm_transfer_matrix")
def plot_I_cross_farm_transfer_matrix(stage3_json: Optional[dict]):
    if not stage3_json:
        return False
    results = stage3_json.get("results", stage3_json)
    dir_map = {"A_to_C": "A → C", "C_to_A": "C → A", "C_to_B": "C → B", "B_to_A": "B → A", "B_to_C": "B → C"}
    matrix_data = {}
    for key, block in results.items():
        if not isinstance(block, dict):
            continue
        label = dir_map.get(key, key.replace("_to_", " → "))
        for sub_key in ["all_faults", "comparable"]:
            sub = block.get(sub_key, {})
            if not isinstance(sub, dict):
                continue
            by_type = sub.get("by_type", {})
            if not by_type:
                continue
            matrix_data[label] = {}
            for ft, v in by_type.items():
                if isinstance(v, dict) and "total" in v:
                    tot = v["total"]
                    det = v.get("detected", 0)
                    matrix_data[label][ft] = det / tot if tot > 0 else 0.0
            break
    if not matrix_data:
        return False
    directions = [d for d in dir_map.values() if d in matrix_data]
    fault_types = sorted(set(ft for m in matrix_data.values() for ft in m.keys()))
    Z = np.full((len(directions), len(fault_types)), np.nan)
    for i, d in enumerate(directions):
        for j, ft in enumerate(fault_types):
            if ft in matrix_data[d]:
                Z[i, j] = matrix_data[d][ft]
    fig, ax = plt.subplots(figsize=(max(9, len(fault_types) * 1.4), max(5, len(directions) * 0.9)))
    im = ax.imshow(np.nan_to_num(Z, nan=-1), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(len(directions)):
        for j in range(len(fault_types)):
            val = Z[i, j]
            if np.isnan(val):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=9, color="#aaaaaa")
            else:
                c = "white" if val > 0.5 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=11, fontweight="bold", color=c)
    ax.set_xticks(range(len(fault_types)))
    ax.set_xticklabels([ft.replace("_", "\n") for ft in fault_types], fontsize=9)
    ax.set_yticks(range(len(directions)))
    ax.set_yticklabels(directions, fontsize=10)
    ax.set_xlabel("Fault type in target farm")
    ax.set_ylabel("Transfer direction")
    ax.set_title("Cross-farm Transfer — Detection Rate Matrix")
    plt.colorbar(im, ax=ax, label="Detection rate")
    save_fig(fig, "I_cross_farm_transfer_matrix.png")


@safe_plot("J_research_contributions")
def plot_J_research_contributions():
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axis("off")
    data = [
        ["C1 — Reproducible benchmark evaluation", "All 3 farms, all 4 CARE sub-scores; Farm C CARE=0.727 (+0.067 above baseline 0.660)"],
        ["C2 — Sensor architecture ceiling quantified", "Farm A coverage ceiling reflects missing sensors rather than pure model failure"],
        ["C3 — Subsystem NBMs resolve dilution", "Farm B improves when subsystem NBMs isolate rare sensors"],
        ["C4 — Cross-farm transfer under schema mismatch", "5 directions tested; normalisation helps but does not fully solve transfer shift"],
        ["C5 — Service mode dominance quantified", "Service-related loss can exceed fault-only loss in offshore farms"],
    ]
    tbl = ax.table(cellText=[[r[1]] for r in data], rowLabels=[r[0] for r in data], colLabels=["Key finding and evidence"], cellLoc="left", rowLoc="center", loc="center", colWidths=[0.78])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 2.8)
    ax.set_title("Five Research Contributions — CARE Benchmark Evaluation", fontsize=12, fontweight="bold", pad=20)
    save_fig(fig, "J_research_contributions.png")


@safe_plot("K_sensor_correlation_matrices")
def plot_K_sensor_correlation_matrices(raw_index: Dict):
    produced = False
    for farm in ["A", "B", "C"]:
        farm_events = [p for (f, _), p in raw_index.items() if f == farm]
        if not farm_events:
            continue
        df = safe_read_csv(farm_events[0])
        if df is None or df.empty:
            continue
        num = coerce_numeric(df)
        if num.shape[1] < 4:
            continue
        corr = num.iloc[:, : min(12, num.shape[1])].corr()
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(corr.values, aspect="auto")
        ax.set_xticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
        ax.set_yticks(range(len(corr.columns)))
        ax.set_yticklabels(corr.columns, fontsize=7)
        ax.set_title(f"Sensor Correlation Matrix — Farm {farm}")
        plt.colorbar(im, ax=ax)
        save_fig(fig, f"K_sensor_correlation_farm{farm}.png")
        produced = True
    return produced or False


@safe_plot("L_sensor_importance_and_gaps")
def plot_L_sensor_importance_and_gaps(stage2_df: pd.DataFrame):
    if not has_cols(stage2_df, {"farm", "n_selected_cols"}):
        return False
    g = stage2_df.groupby("farm")["n_selected_cols"].agg(["mean", "median", "max"]).reset_index().sort_values("farm")
    x = np.arange(len(g))
    w = 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w, g["mean"], w, label="Mean", alpha=0.88)
    ax.bar(x, g["median"], w, label="Median", alpha=0.88)
    ax.bar(x + w, g["max"], w, label="Max", alpha=0.88)
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in g["farm"]])
    ax.set_ylabel("Selected columns")
    ax.set_title("Sensor Importance Proxy and Feature Gaps")
    ax.legend()
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "L_sensor_importance_and_gaps.png")


@safe_plot("M_missing_sensor_gap_analysis")
def plot_M_missing_sensor_gap_analysis():
    subsystems = ["Gearbox\nbearing", "Gearbox\noil", "Hydraulic", "Rotor\nbearing", "Pitch\nmotors", "Vibration"]
    farm_A = [1, 0, 1, 0, 0, 0]
    farm_C = [16, 16, 15, 8, 24, 4]
    x = np.arange(len(subsystems))
    w = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    b1 = ax.bar(x - w / 2, farm_A, w, label="Farm A (onshore)", alpha=0.85)
    b2 = ax.bar(x + w / 2, farm_C, w, label="Farm C (offshore)", alpha=0.85)
    ax.bar_label(b1, padding=2, fontsize=9)
    ax.bar_label(b2, padding=2, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(subsystems, fontsize=10)
    ax.set_ylabel("Number of sensors in subsystem group")
    ax.set_title("Sensor Coverage Gap — Farm A vs Farm C")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    save_fig(fig, "M_missing_sensor_gap_analysis.png")


@safe_plot("N_failure_mode_framework")
def plot_N_failure_mode_framework():
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.axis("off")
    text = (
        "Failure Mode Framework\n\n"
        "Stage 1: anomaly detection identifies abnormal behaviour windows\n"
        "↓\n"
        "Stage 2: classifier maps pre-fault patterns to likely subsystem/fault type\n"
        "↓\n"
        "Stage 3: transfer analysis tests whether patterns generalise across farms\n\n"
        "Used in dissertation to connect sensor evidence, maintenance value, and transferability."
    )
    ax.text(0.5, 0.5, text, ha="center", va="center", fontsize=12, bbox=dict(boxstyle="round,pad=0.8", fc="#f5f5f5", ec="#888"))
    ax.set_title("Failure Mode Framework", fontsize=13, fontweight="bold")
    save_fig(fig, "N_failure_mode_framework.png")


@safe_plot("O_stage2_classification_results")
def plot_O_stage2_classification_results(stage2_df: pd.DataFrame):
    if not has_cols(stage2_df, {"farm", "is_fault", "detected", "fault_type", "pred_type"}):
        return False
    rows = []
    for farm, g in stage2_df.groupby("farm"):
        fault = g[g["is_fault"] == 1].copy()
        if fault.empty:
            continue
        rows.append({
            "farm": farm,
            "n_fault": len(fault),
            "det_rate": fault["detected"].mean(),
            "cls_acc": (fault["fault_type"] == fault["pred_type"]).mean(),
            "mean_horizon": pd.to_numeric(fault.get("horizon_days"), errors="coerce").mean() if "horizon_days" in fault.columns else np.nan,
        })
    df = pd.DataFrame(rows).sort_values("farm")
    if df.empty:
        return False
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.axis("off")
    display = df.copy()
    for c in ["det_rate", "cls_acc", "mean_horizon"]:
        if c in display.columns:
            display[c] = display[c].round(3)
    tbl = ax.table(cellText=display.values, colLabels=display.columns, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.1, 1.8)
    ax.set_title("Stage 2 Classification Summary", fontsize=12, fontweight="bold")
    save_fig(fig, "O_stage2_classification_results.png")


# =============================================================================
# EXTRA FIGURES
# =============================================================================


@safe_plot("N1_dataset_overview")
def plot_N1_dataset_overview(stage1_df: pd.DataFrame):
    if not has_cols(stage1_df, {"farm", "is_fault"}):
        return False

    farms = ["A", "B", "C"]
    anomaly = []
    normal = []
    for f in farms:
        sub = stage1_df[stage1_df["farm"] == f]
        anomaly.append(int((sub["is_fault"] == 1).sum()))
        normal.append(int((sub["is_fault"] == 0).sum()))

    x = np.arange(len(farms))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w / 2, anomaly, w, label="Anomaly events", alpha=0.88)
    b2 = ax.bar(x + w / 2, normal, w, label="Normal events", alpha=0.88)
    ax.bar_label(b1, padding=3, fontsize=11)
    ax.bar_label(b2, padding=3, fontsize=11)
    sensor_counts = {"A": 86, "B": 257, "C": 957}
    for i, farm in enumerate(farms):
        tot = anomaly[i] + normal[i]
        ax.text(i, max(anomaly[i], normal[i]) + 2.2, f"Total: {tot} events\n{sensor_counts[farm]} sensors", ha="center", fontsize=8.5, color="#333", style="italic")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Farm {f}" for f in farms])
    ax.set_ylabel("Number of Events")
    ax.set_ylim(0, 45)
    ax.set_title("CARE Benchmark Dataset — Event Distribution by Farm")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "N1_dataset_overview.png")


@safe_plot("N2_pca_explained_variance")
def plot_N2_pca_explained_variance():
    farm_data = {
        "A": {"n_keep": 50, "var": 95.1, "total_features": 194, "curve": [(1, 30), (10, 60), (20, 74), (30, 83), (40, 90), (50, 95.1), (60, 97), (80, 99)]},
        "B": {"n_keep": 67, "var": 95.1, "total_features": 344, "curve": [(1, 20), (10, 50), (20, 65), (30, 75), (40, 83), (50, 89), (67, 95.1), (80, 97), (100, 99)]},
        "C": {"n_keep": 150, "var": 93.8, "total_features": 1024, "curve": [(1, 12), (20, 40), (40, 58), (60, 70), (80, 79), (100, 86), (120, 90), (150, 93.8), (180, 96)]},
    }
    fig, ax = plt.subplots(figsize=(10, 5))
    for farm, d in farm_data.items():
        xs = [p[0] for p in d["curve"]]
        ys = [p[1] for p in d["curve"]]
        ax.plot(xs, ys, linewidth=2, label=f"Farm {farm} ({d['total_features']} features)")
        ax.axvline(d["n_keep"], linestyle="--", linewidth=1.2, alpha=0.7)
        ax.scatter([d["n_keep"]], [d["var"]], s=80, zorder=5)
    ax.axhline(95, linestyle=":", linewidth=1.2, label="95% variance target")
    ax.set_xlabel("Number of PCA components retained")
    ax.set_ylabel("Cumulative explained variance (%)")
    ax.set_title("PCA Dimensionality Reduction — Cumulative Explained Variance")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.25, linestyle="--")
    save_fig(fig, "N2_pca_explained_variance.png")


@safe_plot("N3_stage2_farmAB_by_type")
def plot_N3_stage2_farmAB_by_type(stage2_df: pd.DataFrame):
    farm_A_data = {"gearbox": {"total": 3, "detected": 2}, "generator_bearing": {"total": 2, "detected": 1}, "hydraulic": {"total": 6, "detected": 3}, "transformer": {"total": 1, "detected": 0}}
    farm_B_data = {"rotor_bearing": {"total": 3, "detected": 3}, "transformer": {"total": 3, "detected": 2}}
    if has_cols(stage2_df, {"farm", "fault_type", "is_fault", "detected"}):
        for farm_id, farm_dict in [("A", farm_A_data), ("B", farm_B_data)]:
            sub = stage2_df[(stage2_df["farm"] == farm_id) & (stage2_df["is_fault"] == 1)]
            if not sub.empty:
                for ft, g in sub.groupby("fault_type"):
                    farm_dict[ft] = {"total": len(g), "detected": int(g["detected"].sum())}
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, title, data in [
        (axes[0], "Farm A — Detection Rate by Fault Type", farm_A_data),
        (axes[1], "Farm B — Detection Rate by Fault Type", farm_B_data),
    ]:
        types = sorted(data.keys())
        rates = [data[t]["detected"] / data[t]["total"] for t in types]
        totals = [data[t]["total"] for t in types]
        bars = ax.bar(types, rates, alpha=0.88)
        ax.axhline(0.5, linestyle="--", linewidth=1.1, alpha=0.6)
        for bar, rate, total, det in zip(bars, rates, totals, [data[t]["detected"] for t in types]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{rate:.0%}\n({det}/{total})", ha="center", va="bottom", fontsize=9)
        ax.set_ylim(0, 1.3)
        ax.set_ylabel("Detection Rate")
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.25, linestyle="--")
    save_fig(fig, "N3_stage2_farmAB_by_type.png")


@safe_plot("N4_pipeline_architecture")
def plot_N4_pipeline_architecture():
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 5)
    ax.axis("off")

    def box(x, y, w, h, text, fontsize=9):
        rect = plt.Rectangle((x, y), w, h, linewidth=1.5, zorder=3, alpha=0.92)
        ax.add_patch(rect)
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, fontweight="bold", color="white", zorder=4, wrap=True)

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=2), zorder=5)

    box(0.2, 1.5, 2.0, 2.0, "SCADA DATA\nFarm A/B/C", 8)
    arrow(2.2, 2.5, 2.8, 2.5)
    box(2.8, 0.5, 3.2, 4.0, "STAGE 1\nAnomaly Detection\nNBM + alarm", 8)
    arrow(6.0, 2.5, 6.6, 2.5)
    box(6.6, 0.5, 3.2, 4.0, "STAGE 2\nFault Classification\nXGBoost + 5-consecutive rule", 8)
    arrow(9.8, 2.5, 10.4, 2.5)
    box(10.4, 0.5, 3.2, 4.0, "STAGE 3\nCross-Farm Transfer\nTransfer evaluation", 8)
    ax.set_title("Two-Stage Predictive Maintenance Pipeline — CARE Benchmark", fontsize=12, fontweight="bold", pad=10)
    save_fig(fig, "N4_pipeline_architecture.png")


@safe_plot("N5_loeo_diagram")
def plot_N5_loeo_diagram():
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 4)
    ax.axis("off")
    n_events = 6
    event_w = 1.4
    event_h = 0.7
    ax.text(0.2, 3.6, "Leave-One-Event-Out (LOEO) Cross-Validation", fontsize=13, fontweight="bold")
    for fold in range(min(4, n_events)):
        y = 2.3 - fold * 0.65
        ax.text(0.1, y + 0.35, f"Fold {fold + 1}:", fontsize=9, fontweight="bold")
        for ev in range(n_events):
            x = 1.5 + ev * (event_w + 0.15)
            rect = plt.Rectangle((x, y), event_w, event_h, linewidth=1.5, alpha=0.85, zorder=3)
            ax.add_patch(rect)
            lbl = "TEST" if ev == fold else "TRAIN"
            ax.text(x + event_w / 2, y + event_h / 2, f"E{ev + 1}\n{lbl}", ha="center", va="center", fontsize=7.5, color="white", fontweight="bold", zorder=4)
    ax.text(0.1, 0.1, "Total folds = number of anomaly events per farm", fontsize=8.5, color="#555", style="italic")
    save_fig(fig, "N5_loeo_diagram.png")


# =============================================================================
# EVENT-PAIR FIGURES
# =============================================================================


@safe_plot("event_pair_behaviour")
def plot_event_sensor_behaviour(farm: str, event_id: str, kind: str, fault_type: str, raw_path: Path, sensors: List[str]):
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False
    meta = build_sensor_metadata_map(farm)
    _, pred_df = split_train_pred(df)
    plot_df = pred_df if not pred_df.empty else df
    sels = [s for s in sensors if s in plot_df.columns]
    if not sels:
        return False
    fig, ax = plt.subplots(figsize=(11, 5))
    for s in sels:
        ax.plot(normalise_series(plot_df[s]).reset_index(drop=True), label=pretty_sensor_name(s, farm, meta)[:80])
    ax.set_title(f"Farm {farm} {event_id}: {fault_type} — {kind} state")
    ax.set_xlabel("Prediction-window timestep")
    ax.set_ylabel("Normalised sensor value")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    save_fig(fig, f"10_event_behaviour_farm{farm}_{kind}_{event_id}.png", label=f"10_event_behaviour_farm{farm}_{kind}_{event_id}")
    return True


@safe_plot("event_pair_train_vs_pred")
def plot_train_vs_pred(farm: str, event_id: str, fault_type: str, raw_path: Path, sensors: List[str]):
    df = safe_read_csv(raw_path)
    if df is None or df.empty:
        return False
    meta = build_sensor_metadata_map(farm)
    train_df, pred_df = split_train_pred(df)
    if train_df.empty or pred_df.empty:
        return False
    sels = [s for s in sensors if s in train_df.columns and s in pred_df.columns]
    if not sels:
        return False
    rows = [{"sensor": s, "train_mean": pd.to_numeric(train_df[s], errors="coerce").mean(), "pred_mean": pd.to_numeric(pred_df[s], errors="coerce").mean()} for s in sels]
    df_p = pd.DataFrame(rows)
    if df_p.empty:
        return False
    x = np.arange(len(df_p))
    w = 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w / 2, df_p["train_mean"], w, label="Healthy/train mean")
    ax.bar(x + w / 2, df_p["pred_mean"], w, label="Prediction-window mean")
    ax.set_xticks(x)
    ax.set_xticklabels([pretty_sensor_name(s, farm, meta)[:55] for s in df_p["sensor"]], rotation=25, ha="right")
    ax.set_title(f"Farm {farm} {event_id}: healthy vs pre-fault sensor means")
    ax.legend()
    save_fig(fig, f"12_train_vs_pred_farm{farm}_{event_id}.png", label=f"12_train_vs_pred_farm{farm}_{event_id}")
    return True


# =============================================================================
# MAIN
# =============================================================================


def write_manifest(extra: Optional[dict] = None) -> None:
    saved = [x for x in FIGURE_LOG if x["status"] == "saved"]
    skipped = [x for x in FIGURE_LOG if x["status"] != "saved"]
    manifest = {
        "base_dir": str(BASE_DIR),
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "n_saved": len(saved),
        "n_skipped_or_error": len(skipped),
        "saved": saved,
        "skipped_or_error": skipped,
    }
    if extra:
        manifest.update(extra)
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(json_safe(manifest), f, indent=2)
    print(f"\n  Manifest saved: {MANIFEST_PATH}")


def main():
    print("=" * 76)
    print("UNIFIED SCRIPT 5 — DISSERTATION FIGURES (FINAL CLEAN VERSION)")
    print(f"Base directory: {BASE_DIR}")
    print("=" * 76)

    ensure_dirs()
    start = time.time()

    print("\nLoading Stage 1 outputs...")
    stage1_json = load_stage1_json()
    stage1_df = load_stage1_event_df()
    stage1_summary = load_stage1_summary()
    dae_results = build_dae_result_map(stage1_summary)
    stage1_event_stats = build_stage1_event_stats(stage1_df)

    print(f"  Stage 1 events : {len(stage1_df)}")
    print(f"  DAE source     : {STAGE1_SUMMARY_CSV}")

    print("\nLoading Stage 2 outputs...")
    stage2_df = load_stage2_df()
    print(f"  Stage 2 events : {len(stage2_df)}")

    print("\nLoading Stage 3 outputs...")
    stage3_json = load_stage3_json()

    print("\nLoading operational analysis outputs...")
    ops_json, ops_summary, ops_status = load_operational_data()
    if ops_summary is not None:
        print(f"  Operational summary: {len(ops_summary)} farms loaded")
    else:
        print("  No operational summary found")

    print("\nIndexing raw event files...")
    raw_index = index_raw_event_files()

    print("\nGenerating broad dissertation figures...")
    plot_01_dataset_balance(stage1_df)
    plot_02_stage1_care_by_farm(stage1_summary)
    plot_03_stage1_care_components(stage1_summary)
    plot_04_stage1_benchmark_comparison(stage1_summary)
    plot_05_stage2_performance_by_farm(stage2_df)
    plot_06_stage2_horizon_by_farm(stage2_df)
    plot_07_stage2_farmC_by_type(stage2_df)
    plot_08_stage2_selected_columns(stage2_df)
    plot_09_cross_farm_transfer(stage3_json)
    plot_10_operational_status_distribution(ops_summary)
    plot_11_operational_availability(ops_summary)
    plot_12_operational_power_economics(ops_summary)
    plot_13_maintenance_cost_comparison()
    plot_14_stage2_horizon_by_fault_type(stage2_df)
    plot_15_cross_farm_transfer_with_horizon(stage3_json)
    plot_16_actual_vs_expected_energy(ops_summary)
    plot_17_recoverable_energy_breakdown(ops_summary)

    print("\nGenerating named supplementary figures...")
    plot_A_care_score_vs_baseline(stage1_summary, stage1_df)
    plot_B_farmB_subsystem_contribution()
    plot_C_sensor_architecture_comparison()
    plot_D_operational_hours_economics(ops_summary)
    plot_E_detection_timeline(stage1_df)
    plot_F_anomaly_score_distribution(stage1_df)
    plot_G_pipeline_results_table(stage1_summary, stage2_df, stage3_json)
    plot_H_degradation_trace_farmC_E47(raw_index, stage2_df)
    plot_I_cross_farm_transfer_matrix(stage3_json)
    plot_J_research_contributions()
    plot_K_sensor_correlation_matrices(raw_index)
    plot_L_sensor_importance_and_gaps(stage2_df)
    plot_M_missing_sensor_gap_analysis()
    plot_N_failure_mode_framework()
    plot_O_stage2_classification_results(stage2_df)

    print("\nGenerating previously missed N-series figures...")
    plot_N1_dataset_overview(stage1_df)
    plot_N2_pca_explained_variance()
    plot_N3_stage2_farmAB_by_type(stage2_df)
    plot_N4_pipeline_architecture()
    plot_N5_loeo_diagram()

    print("\nGenerating event-pair figures...")
    event_pairs = []
    for farm in ["A", "B", "C"]:
        healthy_ev = None
        fault_ev = None

        for eid in PREFERRED_EXAMPLES.get(farm, {}).get("healthy", []):
            if (farm, eid) in raw_index:
                healthy_ev = eid
                break

        for eid in PREFERRED_EXAMPLES.get(farm, {}).get("fault", []):
            if (farm, eid) in raw_index:
                fault_ev = eid
                break

        if healthy_ev is None or fault_ev is None:
            continue

        ft = best_fault_type(stage2_df, farm, fault_ev)
        raw_path = raw_index.get((farm, fault_ev))
        if raw_path is None:
            continue

        df_raw = safe_read_csv(raw_path)
        if df_raw is None or df_raw.empty:
            continue

        meta = build_sensor_metadata_map(farm)
        tr, pr = split_train_pred(df_raw)
        sensors = pick_sensors(farm, ft, tr if not tr.empty else df_raw, pr if not pr.empty else df_raw, meta, n=4)
        if not sensors:
            continue

        for event_id, kind in [(healthy_ev, "healthy"), (fault_ev, "fault")]:
            rp = raw_index.get((farm, event_id))
            if rp is None:
                continue
            ok1 = plot_event_sensor_behaviour(farm, event_id, kind, ft, rp, sensors)
            ok2 = plot_train_vs_pred(farm, event_id, ft, rp, sensors)
            if ok1 or ok2:
                event_pairs.append({"farm": farm, "event_id": event_id, "kind": kind, "fault_type": ft, "sensors": sensors})

    runtime = timedelta(seconds=time.time() - start)
    write_manifest({
        "runtime": str(runtime),
        "stage1_json_source": str(STAGE1_JSON),
        "stage1_event_csv_source": str(STAGE1_EVENT_CSV),
        "stage1_summary_csv_source": str(STAGE1_SUMMARY_CSV),
        "dae_results_from_stage1_summary": dae_results,
        "stage1_event_stats_from_stage1_csv": stage1_event_stats,
        "if_results_benchmark": IF_RESULTS,
        "event_pairs": event_pairs,
        "ops_summary_csv": str(OPS_SUMMARY_CSV),
    })

    print("\n" + "=" * 76)
    print(f"DONE — Runtime: {runtime}")
    print(f"Plots   → {PLOTS_DIR}")
    print(f"Reports → {REPORTS_DIR}")
    print("=" * 76)

    pngs = sorted(PLOTS_DIR.glob("*.png"))
    print(f"\nFigures present in plots folder ({len(pngs)} files):")
    for p in pngs:
        try:
            size_kb = p.stat().st_size // 1024
            print(f"  {p.name:<55} ({size_kb} KB)")
        except Exception:
            print(f"  {p.name}")


if __name__ == "__main__":
    main()

UNIFIED SCRIPT 5 — DISSERTATION FIGURES (FINAL CLEAN VERSION)
Base directory: D:\files\project\CARE_To_Compare

Loading Stage 1 outputs...
  Stage 1 JSON : D:\files\project\CARE_To_Compare\stage1_clean_outputs\results_stage1_clean.json
  Stage 1 CSV : D:\files\project\CARE_To_Compare\stage1_clean_outputs\event_results_stage1_clean.csv
  Stage 1 summary CSV : D:\files\project\CARE_To_Compare\stage1_clean_outputs\reports\care_score_summary_clean.csv
  Stage 1 events : 95
  DAE source     : D:\files\project\CARE_To_Compare\stage1_clean_outputs\reports\care_score_summary_clean.csv

Loading Stage 2 outputs...
  Stage 2 CSV : D:\files\project\CARE_To_Compare\fault_classification_outputs\fault_classification_results.csv
  Stage 2 events : 95

Loading Stage 3 outputs...
  Stage 3 JSON: D:\files\project\CARE_To_Compare\cross_farm_outputs\cross_farm_results.json

Loading operational analysis outputs...
  Operational summary: 3 farms loaded

Indexing raw event files...
  Indexed 95 raw event file

In [3]:
# ============================================================================
# SCRIPT 6 
# ============================================================================


from __future__ import annotations

import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

warnings.filterwarnings("ignore")


# ============================================================================
# CONFIG
# ============================================================================

BASE_DIR = Path(r"D:\files\project\CARE_To_Compare")

FARM_DATASET_PATHS = {
    "A": BASE_DIR / "Wind Farm A" / "datasets",
    "B": BASE_DIR / "Wind Farm B" / "datasets",
    "C": BASE_DIR / "Wind Farm C" / "datasets",
}

FARM_METADATA_PATHS = {
    "A": BASE_DIR / "Wind Farm A" / "feature_description.csv",
    "B": BASE_DIR / "Wind Farm B" / "feature_description.csv",
    "C": BASE_DIR / "Wind Farm C" / "feature_description.csv",
}

STAGE1_EVENT_CSV = BASE_DIR / "stage1_clean_outputs" / "event_results_stage1_clean.csv"

OUTPUT_DIR = BASE_DIR / "dissertation_figures_outputs" / "3d_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Light publication-style palette
FIG_BG = "#FFFFFF"
TEXT = "#111111"
GRID = "#D0D0D0"
EDGE = "#BEBEBE"

BLUE_ACC = "#5DA5FF"
RED_ACC = "#E45756"
GREEN_ACC = "#4C9F70"
ORANGE_ACC = "#F58518"
GREY_ACC = "#A8A8A8"


# ============================================================================
# HELPERS
# ============================================================================

def safe_read_csv(path: Path) -> Optional[pd.DataFrame]:
    for sep, enc in [(";", "utf-8"), (";", "latin1"), (",", "utf-8"), (",", "latin1")]:
        try:
            df = pd.read_csv(path, sep=sep, encoding=enc)
            df.columns = [str(c).strip() for c in df.columns]
            return df
        except Exception:
            continue
    return None


def split_train_pred(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if "train_test" not in df.columns:
        return df.copy(), df.copy()

    s = df["train_test"].astype(str).str.lower().str.strip()
    train = df[s == "train"].copy()
    pred = df[s.isin(["pred", "prediction", "test"])].copy()

    if pred.empty:
        pred = df[~s.eq("train")].copy()
    if train.empty:
        train = df.copy()

    return train, pred


def get_numeric_cols(df: pd.DataFrame) -> List[str]:
    meta = {"train_test", "status_type_id", "asset_id", "id", "time_stamp", "timestamp"}
    return [c for c in df.columns if c not in meta and pd.api.types.is_numeric_dtype(df[c])]


def save_fig(fig: plt.Figure, name: str, dpi: int = 300) -> None:
    out = OUTPUT_DIR / name
    fig.savefig(out, dpi=dpi, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved: {out}")


def short_sensor_label(desc: str, fallback: str, max_len: int = 18) -> str:
    """
    Convert long metadata descriptions to short dissertation-safe labels.
    """
    d = str(desc).lower().strip()

    rules = [
        ("temperature internal consumption", "Temp int. cons."),
        ("transformer core temperature", "Core temp."),
        ("transformer cell temperature", "Cell temp."),
        ("transformer l1 (mid-voltage)", "L1 mid-V"),
        ("transformer l1 (undervoltage)", "L1 under-V"),
        ("transformer l2 (mid-voltage)", "L2 mid-V"),
        ("transformer l2 (undervoltage)", "L2 under-V"),
        ("transformer l3 (mid-voltage)", "L3 mid-V"),
        ("transformer l3 (undervoltage)", "L3 under-V"),
    ]
    for pattern, label in rules:
        if pattern in d:
            return label

    base = re.sub(r'[_\-]+', ' ', fallback)
    base = re.sub(r'_(avg|max|min|std)$', '', base, flags=re.IGNORECASE)
    base = base.strip().title()
    return base[:max_len]


def apply_3d_style(ax, grid_alpha: float = 0.55) -> None:
    ax.set_facecolor(FIG_BG)

    ax.tick_params(colors=TEXT, labelsize=8)

    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.xaxis.pane.set_edgecolor(EDGE)
    ax.yaxis.pane.set_edgecolor(EDGE)
    ax.zaxis.pane.set_edgecolor(EDGE)

    ax.grid(True, color=GRID, alpha=grid_alpha, linewidth=0.5)


# ============================================================================
# FIGURE 3D-1 — SENSOR DEGRADATION TRAJECTORY
# ============================================================================

def plot_3d_degradation_trajectory() -> None:
    """
    Dissertation-safe version of the Farm B Event E34 degradation surface.
    """
    print("\n[3D-1] Refined degradation trajectory...")

    event_path = FARM_DATASET_PATHS["B"] / "34.csv"
    if not event_path.exists():
        print(f"  Event file not found: {event_path}")
        return

    df = safe_read_csv(event_path)
    if df is None or df.empty:
        print("  Could not read event file.")
        return

    train_df, pred_df = split_train_pred(df)

    transformer_cols = []
    feat_path = FARM_METADATA_PATHS["B"]

    if feat_path.exists():
        feat_df = safe_read_csv(feat_path)
        if feat_df is not None and not feat_df.empty:
            lower_cols = {str(c).lower(): c for c in feat_df.columns}
            s_col = lower_cols.get("sensor_name")
            d_col = lower_cols.get("description")

            if s_col and d_col:
                keywords = [
                    "transformer",
                    "transformator",
                    "winding",
                    "isolation resistance",
                    "hv transformer",
                    "lv transformer",
                    "transformer cell",
                ]

                for _, row in feat_df.iterrows():
                    desc = str(row.get(d_col, "")).strip()
                    sid = str(row.get(s_col, "")).strip()

                    if any(k in desc.lower() for k in keywords):
                        for stat in ["_avg", "_max", "_min", "_std", ""]:
                            col_name = f"{sid}{stat}"
                            if col_name in pred_df.columns and col_name in train_df.columns:
                                transformer_cols.append((col_name, desc))
                                break

    if len(transformer_cols) < 3:
        print("  Transformer-specific sensors insufficient; using variance fallback.")
        exclude_kw = ["wind", "power", "angle", "direction", "speed", "frequency", "reactive", "grid"]
        numeric = get_numeric_cols(pred_df)
        filtered = [
            c for c in numeric
            if c in train_df.columns and not any(k in c.lower() for k in exclude_kw)
        ]
        if len(filtered) < 4:
            filtered = [c for c in numeric if c in train_df.columns]

        var_scores = {c: pred_df[c].var() for c in filtered}
        cols = sorted(var_scores, key=var_scores.get, reverse=True)[:8]
        sensor_names = cols
        sensor_labels = [short_sensor_label(c, c) for c in cols]
    else:
        chosen = transformer_cols[:8]
        sensor_names = [c for c, _ in chosen]
        sensor_labels = [short_sensor_label(desc, c) for c, desc in chosen]

    deviations = {}
    for c in sensor_names:
        tr = pd.to_numeric(train_df[c], errors="coerce").dropna()
        pr = pd.to_numeric(pred_df[c], errors="coerce")
        mu, sd = tr.mean(), tr.std()
        if pd.isna(sd) or sd < 1e-9:
            continue
        dev = ((pr - mu) / sd).fillna(0).values
        deviations[c] = dev

    if len(deviations) < 3:
        print("  Insufficient sensor data for refined 3D plot.")
        return

    min_len = min(len(v) for v in deviations.values())
    min_len = min(min_len, 320)

    sensor_names = list(deviations.keys())
    Z_raw = np.array([deviations[s][:min_len] for s in sensor_names])

    def smooth(arr: np.ndarray, w: int = 10) -> np.ndarray:
        return pd.Series(arr).rolling(w, min_periods=1).mean().values

    Z = np.array([smooth(row) for row in Z_raw])
    Z = np.clip(Z, -2.5, 4.0)

    n_sensors, n_time = Z.shape
    T = np.linspace(0, min_len * 10 / 60 / 24, n_time)
    S = np.arange(n_sensors)
    TT, SS = np.meshgrid(T, S)

    # Main figure
    fig = plt.figure(figsize=(12.2, 8.4), facecolor=FIG_BG)
    ax = fig.add_subplot(111, projection="3d")

    surf = ax.plot_surface(
        TT, SS, Z,
        cmap="plasma",
        alpha=0.92,
        linewidth=0,
        antialiased=True,
        vmin=-2,
        vmax=4
    )

    ax.plot_wireframe(
        TT, SS, Z,
        color="#2F2F2F",
        alpha=0.10,
        linewidth=0.3,
        rstride=5,
        cstride=3
    )

    # Healthy baseline plane
    ax.plot_surface(
        TT,
        SS,
        np.zeros_like(Z),
        color=BLUE_ACC,
        alpha=0.09,
        linewidth=0
    )

    cbar = fig.colorbar(surf, ax=ax, shrink=0.48, aspect=14, pad=0.03)
    cbar.set_label("Deviation from healthy baseline (σ)", color=TEXT, fontsize=10)
    cbar.ax.yaxis.set_tick_params(color=TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT, fontsize=8)

    ax.set_xlabel("Time in prediction window (days)", color=TEXT, fontsize=10, labelpad=10)
    ax.set_ylabel("Sensor channel", color=TEXT, fontsize=10, labelpad=10)
    ax.set_zlabel("Deviation (σ)", color=TEXT, fontsize=10, labelpad=10)

    ax.set_title(
        "Farm B Event E34 — Transformer sensor degradation trajectory",
        color=TEXT,
        fontsize=13,
        fontweight="bold",
        pad=16
    )

    # Show fewer y labels for readability
    yticks = np.arange(n_sensors)
    yticklabels = []
    for i, label in enumerate(sensor_labels[:n_sensors]):
        yticklabels.append(label if i % 2 == 0 else "")

    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels, fontsize=8, color=TEXT)

    apply_3d_style(ax)
    ax.view_init(elev=26, azim=-58)

    plt.tight_layout()
    save_fig(fig, "3D_fig1_degradation_trajectory_refined.png")

    # Alternative view
    fig2 = plt.figure(figsize=(12.0, 8.0), facecolor=FIG_BG)
    ax2 = fig2.add_subplot(111, projection="3d")

    surf2 = ax2.plot_surface(
        TT, SS, Z,
        cmap="plasma",
        alpha=0.92,
        linewidth=0,
        antialiased=True,
        vmin=-2,
        vmax=4
    )

    ax2.plot_wireframe(
        TT, SS, Z,
        color="#2F2F2F",
        alpha=0.10,
        linewidth=0.3,
        rstride=5,
        cstride=3
    )

    ax2.plot_surface(
        TT,
        SS,
        np.zeros_like(Z),
        color=BLUE_ACC,
        alpha=0.09,
        linewidth=0
    )

    ax2.set_xlabel("Time (days)", color=TEXT, fontsize=10, labelpad=10)
    ax2.set_ylabel("Sensor channel", color=TEXT, fontsize=10, labelpad=10)
    ax2.set_zlabel("Deviation (σ)", color=TEXT, fontsize=10, labelpad=10)
    ax2.set_title(
        "Farm B Event E34 — Degradation trajectory",
        color=TEXT,
        fontsize=12,
        fontweight="bold",
        pad=16
    )

    ax2.set_yticks(yticks)
    ax2.set_yticklabels(yticklabels, fontsize=8, color=TEXT)

    apply_3d_style(ax2)
    ax2.view_init(elev=18, azim=35)

    cbar2 = fig2.colorbar(surf2, ax=ax2, shrink=0.46, aspect=14, pad=0.03)
    cbar2.set_label("Deviation from healthy baseline (σ)", color=TEXT, fontsize=10)
    cbar2.ax.yaxis.set_tick_params(color=TEXT)
    plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=TEXT, fontsize=8)

    plt.tight_layout()
    save_fig(fig2, "3D_fig1b_degradation_trajectory_alt_refined.png")

    print("  [3D-1] Complete.")


# ============================================================================
# FIGURE 3D-2 — HONESTLY LABELLED PCA OF STAGE 1 OUTPUTS
# ============================================================================

def plot_3d_stage1_outputs_pca() -> None:
    """
    Appendix-only figure.
    Honest wording: this is not the AE latent space unless extracted from encoder outputs.
    """
    print("\n[3D-2] PCA of Stage 1 event-level outputs...")

    if not STAGE1_EVENT_CSV.exists():
        print(f"  Stage 1 CSV not found: {STAGE1_EVENT_CSV}")
        return

    df = pd.read_csv(STAGE1_EVENT_CSV)
    df.columns = [str(c).strip() for c in df.columns]

    feature_cols = [
        c for c in df.columns
        if c not in {
            "event_id", "farm", "label", "description", "alarm_raised",
            "alarm_source", "top_subsystem", "event_start", "event_end"
        }
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    if len(feature_cols) < 3:
        print(f"  Only {len(feature_cols)} numeric columns found.")
        return

    X = df[feature_cols].fillna(0).values.astype(float)

    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd < 1e-9] = 1.0
    Xs = (X - mu) / sd

    from numpy.linalg import svd
    U, S_vals, Vt = svd(Xs, full_matrices=False)
    X3 = Xs @ Vt[:3].T
    var_explained = (S_vals[:3] ** 2 / (S_vals ** 2).sum()) * 100

    is_fault = (df["label"].astype(str).str.lower() != "normal").values
    farm_col = df["farm"].astype(str).str.upper().values if "farm" in df.columns else np.array(["?"] * len(df))

    farm_colours = {"A": "#2E6FBB", "B": "#3A8E47", "C": "#E66B11"}

    fig = plt.figure(figsize=(11.8, 8.2), facecolor=FIG_BG)
    ax = fig.add_subplot(111, projection="3d")

    norm_mask = ~is_fault
    ax.scatter(
        X3[norm_mask, 0], X3[norm_mask, 1], X3[norm_mask, 2],
        c="#B8B8B8",
        s=36,
        alpha=0.75,
        edgecolors="#7E7E7E",
        linewidths=0.5,
        label=f"Normal events (n={norm_mask.sum()})"
    )

    for farm, colour in farm_colours.items():
        mask = is_fault & (farm_col == farm)
        if mask.sum() == 0:
            continue
        ax.scatter(
            X3[mask, 0], X3[mask, 1], X3[mask, 2],
            c=colour,
            s=75,
            alpha=0.92,
            edgecolors="white",
            linewidths=0.8,
            label=f"Farm {farm} fault events (n={mask.sum()})"
        )

    ax.scatter(
        [0], [0], [0],
        c="black",
        s=120,
        marker="*",
        label="Origin"
    )

    ax.set_xlabel(f"PC1 ({var_explained[0]:.1f}% var)", color=TEXT, fontsize=10, labelpad=10)
    ax.set_ylabel(f"PC2 ({var_explained[1]:.1f}% var)", color=TEXT, fontsize=10, labelpad=10)
    ax.set_zlabel(f"PC3 ({var_explained[2]:.1f}% var)", color=TEXT, fontsize=10, labelpad=10)

    ax.set_title(
        "Stage 1 event-level outputs — 3D PCA projection",
        color=TEXT,
        fontsize=13,
        fontweight="bold",
        pad=16
    )

    apply_3d_style(ax)
    ax.view_init(elev=21, azim=-48)

    legend = ax.legend(
        loc="upper left",
        fontsize=9,
        facecolor="white",
        edgecolor="#8C8C8C",
        framealpha=0.95
    )
    for text in legend.get_texts():
        text.set_color(TEXT)

    plt.tight_layout()
    save_fig(fig, "3D_fig2_stage1_outputs_pca_3d.png")
    print("  [3D-2] Complete.")


# ============================================================================
# FIGURE 3D-3 — REFINED CROSS-FARM TRANSFER MATRIX
# ============================================================================

def plot_3d_transfer_matrix() -> None:
    """
    Main 3D figure worth keeping in the dissertation body.
    """

    print("\n[3D-3] Refined cross-farm transfer matrix...")

    transfer_results = [
        ("A→C", 0, 2, 0.200, 2.2),
        ("B→A", 1, 0, 0.083, 8.1),
        ("C→A", 2, 0, 0.000, 0.0),
        ("C→B", 2, 1, 0.000, 0.0),
        ("B→C", 1, 2, 0.000, 0.0),
    ]

    loeo_baselines = [
        ("A", 0, 0, 0.500, 12.5),
        ("B", 1, 1, 0.833, 41.1),
        ("C", 2, 2, 0.444, 19.3),
    ]

    farm_positions = [0, 1, 2]
    farm_labels = ["Farm A", "Farm B", "Farm C"]

    fig = plt.figure(figsize=(12.0, 8.6), facecolor=FIG_BG)
    ax = fig.add_subplot(111, projection="3d")

    bar_w = 0.34
    bar_d = 0.34
    max_horizon = max([h for _, _, _, _, h in transfer_results + loeo_baselines])

    horizon_cmap = cm.get_cmap("YlOrRd")

    def draw_bar(ax_, x: float, y: float, z_height: float, face, alpha=0.90, edge="#FFFFFF"):
        verts = [
            [x, y, 0],
            [x + bar_w, y, 0],
            [x + bar_w, y + bar_d, 0],
            [x, y + bar_d, 0],
            [x, y, z_height],
            [x + bar_w, y, z_height],
            [x + bar_w, y + bar_d, z_height],
            [x, y + bar_d, z_height],
        ]
        faces = [
            [verts[0], verts[1], verts[5], verts[4]],
            [verts[2], verts[3], verts[7], verts[6]],
            [verts[0], verts[3], verts[7], verts[4]],
            [verts[1], verts[2], verts[6], verts[5]],
            [verts[4], verts[5], verts[6], verts[7]],
        ]
        poly = Poly3DCollection(faces, alpha=alpha, linewidths=0.7, edgecolors=edge)
        poly.set_facecolor(face)
        ax_.add_collection3d(poly)

    # Draw LOEO diagonal first, so they stay visually in the background
    for farm, sx, ty, rate, horizon in loeo_baselines:
        draw_bar(
            ax,
            sx - bar_w / 2,
            ty - bar_d / 2,
            rate,
            face=mcolors.to_rgba(BLUE_ACC, 0.28),
            alpha=0.28,
            edge=BLUE_ACC
        )
        ax.text(
            sx,
            ty,
            rate + 0.03,
            f"LOEO\n{rate:.0%}",
            ha="center",
            va="bottom",
            fontsize=8,
            color="#2A5D9F",
            style="italic"
        )

    # Draw transfer bars
    for label, sx, ty, rate, horizon in transfer_results:
        if rate > 0:
            colour = horizon_cmap(horizon / max_horizon if max_horizon > 0 else 0)
            draw_bar(
                ax,
                sx - bar_w / 2,
                ty - bar_d / 2,
                rate,
                face=colour,
                alpha=0.92,
                edge="#FFFFFF"
            )
            ax.text(
                sx,
                ty,
                rate + 0.04,
                f"{label}\n{rate:.0%}",
                ha="center",
                va="bottom",
                fontsize=9,
                color=TEXT,
                fontweight="bold"
            )
        else:
            draw_bar(
                ax,
                sx - bar_w / 2,
                ty - bar_d / 2,
                0.015,
                face="#ECECEC",
                alpha=0.55,
                edge="#B3B3B3"
            )
            ax.text(
                sx,
                ty,
                0.04,
                f"{label}",
                ha="center",
                va="bottom",
                fontsize=8,
                color="#666666"
            )

    ax.set_xlim(-0.45, 2.65)
    ax.set_ylim(-0.45, 2.65)
    ax.set_zlim(0, 1.0)

    ax.set_xticks(farm_positions)
    ax.set_xticklabels(farm_labels, color=TEXT, fontsize=9)
    ax.set_yticks(farm_positions)
    ax.set_yticklabels(farm_labels, color=TEXT, fontsize=9)
    ax.set_zticks(np.linspace(0, 1.0, 6))
    ax.set_zticklabels([f"{int(v * 100)}%" for v in np.linspace(0, 1.0, 6)], color=TEXT, fontsize=8)

    ax.set_xlabel("Source farm (trained on)", color=TEXT, fontsize=11, labelpad=12)
    ax.set_ylabel("Target farm (tested on)", color=TEXT, fontsize=11, labelpad=12)
    ax.set_zlabel("Detection rate", color=TEXT, fontsize=11, labelpad=10)

    ax.set_title(
        "Cross-farm transfer learning",
        color=TEXT,
        fontsize=13,
        fontweight="bold",
        pad=16
    )

    apply_3d_style(ax)
    ax.view_init(elev=27, azim=-52)

    sm = cm.ScalarMappable(
        cmap="YlOrRd",
        norm=mcolors.Normalize(vmin=0, vmax=max_horizon)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.48, aspect=14, pad=0.04)
    cbar.set_label("Warning horizon (days)", color=TEXT, fontsize=10)
    cbar.ax.yaxis.set_tick_params(color=TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT, fontsize=8)

    plt.tight_layout()
    save_fig(fig, "3D_fig3_cross_farm_transfer_refined.png")

    # Optional alternative angle
    fig2 = plt.figure(figsize=(11.6, 8.0), facecolor=FIG_BG)
    ax2 = fig2.add_subplot(111, projection="3d")

    for farm, sx, ty, rate, horizon in loeo_baselines:
        draw_bar(
            ax2,
            sx - bar_w / 2,
            ty - bar_d / 2,
            rate,
            face=mcolors.to_rgba(BLUE_ACC, 0.24),
            alpha=0.24,
            edge=BLUE_ACC
        )

    for label, sx, ty, rate, horizon in transfer_results:
        if rate > 0:
            colour = horizon_cmap(horizon / max_horizon if max_horizon > 0 else 0)
            draw_bar(ax2, sx - bar_w / 2, ty - bar_d / 2, rate, face=colour, alpha=0.92, edge="#FFFFFF")
            ax2.text(
                sx, ty, rate + 0.04,
                f"{rate:.0%}",
                ha="center", va="bottom",
                fontsize=10, color=TEXT, fontweight="bold"
            )
        else:
            draw_bar(ax2, sx - bar_w / 2, ty - bar_d / 2, 0.015, face="#ECECEC", alpha=0.55, edge="#B3B3B3")

    ax2.set_xlim(-0.45, 2.65)
    ax2.set_ylim(-0.45, 2.65)
    ax2.set_zlim(0, 1.0)

    ax2.set_xticks(farm_positions)
    ax2.set_xticklabels(farm_labels, color=TEXT, fontsize=9)
    ax2.set_yticks(farm_positions)
    ax2.set_yticklabels(farm_labels, color=TEXT, fontsize=9)
    ax2.set_zticks(np.linspace(0, 1.0, 6))
    ax2.set_zticklabels([f"{int(v * 100)}%" for v in np.linspace(0, 1.0, 6)], color=TEXT, fontsize=8)

    ax2.set_xlabel("Source farm", color=TEXT, fontsize=10, labelpad=10)
    ax2.set_ylabel("Target farm", color=TEXT, fontsize=10, labelpad=10)
    ax2.set_zlabel("Detection rate", color=TEXT, fontsize=10, labelpad=8)

    ax2.set_title(
        "Cross-farm transfer learning",
        color=TEXT,
        fontsize=12,
        fontweight="bold",
        pad=16
    )

    apply_3d_style(ax2)
    ax2.view_init(elev=17, azim=30)

    plt.tight_layout()
    save_fig(fig2, "3D_fig3b_cross_farm_transfer_alt_refined.png")

    print("  [3D-3] Complete.")


# ============================================================================
# MAIN
# ============================================================================

def main() -> None:
    print("=" * 72)
    print("SCRIPT 6 — REFINED 3D DISSERTATION FIGURES")
    print(f"Output directory: {OUTPUT_DIR}")
    print("=" * 72)

    plot_3d_degradation_trajectory()
    plot_3d_stage1_outputs_pca()
    plot_3d_transfer_matrix()

    print("\n" + "=" * 72)
    print("SCRIPT COMPLETE")
    print(f"Outputs saved to: {OUTPUT_DIR}")
    print("=" * 72)

    print("\nGenerated files:")
    for p in sorted(OUTPUT_DIR.glob("*.png")):
        print(f"  {p.name}")


if __name__ == "__main__":
    main()

SCRIPT 6 — REFINED 3D DISSERTATION FIGURES
Output directory: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots

[3D-1] Refined degradation trajectory...
  Saved: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots\3D_fig1_degradation_trajectory_refined.png
  Saved: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots\3D_fig1b_degradation_trajectory_alt_refined.png
  [3D-1] Complete.

[3D-2] PCA of Stage 1 event-level outputs...
  Saved: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots\3D_fig2_stage1_outputs_pca_3d.png
  [3D-2] Complete.

[3D-3] Refined cross-farm transfer matrix...
  Saved: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots\3D_fig3_cross_farm_transfer_refined.png
  Saved: D:\files\project\CARE_To_Compare\dissertation_figures_outputs\3d_plots\3D_fig3b_cross_farm_transfer_alt_refined.png
  [3D-3] Complete.

SCRIPT COMPLETE
Outputs saved to: D:\files\project\CARE_To_Compare